# Notebook 16 — Race Classification and Eligibility

## Bounded question

> What do the source race-classification and eligibility fields represent, how complete and internally consistent are they, how do their meanings vary by jurisdiction and race type, and which values can be safely preserved, parsed or derived?

## Initial governed scope

The source-field governance register places the following race-grain fields in `race_classification_and_conditions`:

- `race_name`
- `type`
- `class`
- `pattern`
- `rating_band`
- `age_band`
- `sex_rest`

`going` is deliberately excluded from this initial boundary because the governance register assigns it to the separate `race_conditions` family.

This notebook begins with source profiling only. It does not yet define parsers, canonical categories, regulatory equivalences, eligibility rules, or reusable implementation.

## Stage 1 — Establish storage and availability at runner and provisional-race grain

This first stage establishes the raw evidence boundary before interpreting any labels. It checks the SQLite storage classes, null/blank states, distinct raw values and within-race consistency for the seven governed fields.

The provisional race identity remains `date + course + off`. No assumption is made yet that any field has the same meaning across jurisdictions, racing codes or periods.

In [1]:
# Input grain:
#   Governed source runner rows from SQLite table `data`, restricted by
#   DATA_ROW_PREDICATE = "rowid <> 1". Each row is a runner record.
#   The seven investigated fields are expected to describe the containing race.
#
# Output grain:
#   1. One population-summary row.
#   2. One availability/distinctness row per investigated source field.
#   3. One row per field and observed SQLite storage class.
#   4. One within-race consistency row per investigated source field.
#
# Purpose:
#   Establish the raw storage, availability, distinctness and provisional-race
#   consistency of the classification and eligibility fields before attempting
#   to interpret their meaning or define any transformation rules.
#
# Raw versus derived values:
#   Raw source values are read unchanged.
#   TRIM(CAST(field AS TEXT)) is used only to distinguish populated values from
#   empty or whitespace-only text. It is not treated as a canonical value.
#   The provisional race identity is derived from raw date + course + off.
#
# Assumptions deliberately not made:
#   - populated values are not assumed to be complete or correct;
#   - field names are not treated as definitions;
#   - values are not assumed to mean the same thing across jurisdictions;
#   - values are not assumed to mean the same thing across Flat, Hurdle, Chase
#     or other race types;
#   - blanks are not converted to nulls or named categories;
#   - race_name is not parsed for hidden conditions at this stage;
#   - repeated runner-row values are not counted as independent race facts;
#   - internally mixed races are preserved as evidence rather than corrected.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - the source database cannot be found;
#   - the source table is missing any required field;
#   - the governed runner population differs from 1,851,285 rows;
#   - the provisional race population differs from 189,043 races.
#
#   No source values are written, updated or deleted. The SQLite database is
#   opened in read-only mode.

from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display


# Resolve the repository root whether the notebook server was started from the
# repository root or from inside the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# Immutable source configuration.
SOURCE_DATABASE_PATH = (
    PROJECT_ROOT
    / "data/raw/form_2015-present/form_2015-present/raceform.db"
)
SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"


# Established source-wide control totals from the completed earlier studies.
EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043


# Exact governed scope for this notebook.
CLASSIFICATION_FIELDS = [
    "race_name",
    "type",
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
]

# Existing provisional race identity established by Notebook 03.
RACE_KEY_FIELDS = ["date", "course", "off"]


# Fail immediately rather than allowing SQLite to create or open an unintended
# empty database at a mistyped path.
if not SOURCE_DATABASE_PATH.is_file():
    raise FileNotFoundError(
        f"Source database not found: {SOURCE_DATABASE_PATH}"
    )


# Open the source database explicitly in read-only mode.
connection_uri = f"file:{SOURCE_DATABASE_PATH}?mode=ro"

with sqlite3.connect(connection_uri, uri=True) as connection:
    # Confirm that the source schema still contains every race-key and
    # classification field required by this investigation.
    table_info = pd.read_sql_query(
        f"PRAGMA table_info({SOURCE_TABLE})",
        connection,
    )

    observed_columns = set(table_info["name"])
    required_columns = set(RACE_KEY_FIELDS + CLASSIFICATION_FIELDS)
    missing_columns = sorted(required_columns - observed_columns)

    if missing_columns:
        raise AssertionError(
            f"Required source columns are missing: {missing_columns}"
        )

    # Reconfirm the governed runner and provisional-race populations before
    # trusting any subsequent profile counts.
    population = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS runner_rows,
            COUNT(
                DISTINCT date || char(31) || course || char(31) || off
            ) AS provisional_races
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        """,
        connection,
    )

    runner_rows = int(population.loc[0, "runner_rows"])
    provisional_races = int(population.loc[0, "provisional_races"])

    assert runner_rows == EXPECTED_RUNNER_ROWS, (
        f"Governed runner population changed: "
        f"{runner_rows:,} != {EXPECTED_RUNNER_ROWS:,}"
    )

    assert provisional_races == EXPECTED_PROVISIONAL_RACES, (
        f"Provisional race population changed: "
        f"{provisional_races:,} != {EXPECTED_PROVISIONAL_RACES:,}"
    )

    # Each list will receive one small dataframe per investigated field.
    # They are concatenated only after all source queries have completed.
    field_summary_parts = []
    storage_summary_parts = []
    race_consistency_parts = []

    for field in CLASSIFICATION_FIELDS:
        # Profile null, blank and populated states at runner-row grain.
        #
        # Distinct raw values are counted without trimming or canonicalisation.
        # TRIM is used only to decide whether a value is whitespace-only.
        field_summary_parts.append(
            pd.read_sql_query(
                f"""
                SELECT
                    '{field}' AS source_field,
                    COUNT(*) AS runner_rows,

                    SUM(
                        CASE
                            WHEN [{field}] IS NULL THEN 1
                            ELSE 0
                        END
                    ) AS null_rows,

                    SUM(
                        CASE
                            WHEN [{field}] IS NOT NULL
                             AND TRIM(CAST([{field}] AS TEXT)) = ''
                            THEN 1
                            ELSE 0
                        END
                    ) AS blank_rows,

                    SUM(
                        CASE
                            WHEN [{field}] IS NOT NULL
                             AND TRIM(CAST([{field}] AS TEXT)) <> ''
                            THEN 1
                            ELSE 0
                        END
                    ) AS populated_rows,

                    COUNT(
                        DISTINCT CASE
                            WHEN [{field}] IS NOT NULL
                             AND TRIM(CAST([{field}] AS TEXT)) <> ''
                            THEN CAST([{field}] AS TEXT)
                        END
                    ) AS distinct_populated_raw_values

                FROM {SOURCE_TABLE}
                WHERE {DATA_ROW_PREDICATE}
                """,
                connection,
            )
        )

        # Record SQLite's actual runtime storage classes rather than relying
        # only on the table's declared column type.
        storage_summary_parts.append(
            pd.read_sql_query(
                f"""
                SELECT
                    '{field}' AS source_field,
                    typeof([{field}]) AS sqlite_storage_class,
                    COUNT(*) AS runner_rows

                FROM {SOURCE_TABLE}
                WHERE {DATA_ROW_PREDICATE}

                GROUP BY typeof([{field}])
                ORDER BY runner_rows DESC, sqlite_storage_class
                """,
                connection,
            )
        )

        # Test whether all runner rows within each provisional race carry the
        # same raw state for the field.
        #
        # The state includes SQLite storage class and raw textual rendering so
        # that, for example, integer 1 and text '1' remain distinguishable.
        race_consistency_parts.append(
            pd.read_sql_query(
                f"""
                WITH race_values AS (
                    SELECT
                        date,
                        course,
                        off,

                        COUNT(
                            DISTINCT CASE
                                WHEN [{field}] IS NULL
                                THEN '<NULL>'
                                ELSE (
                                    typeof([{field}])
                                    || ':'
                                    || CAST([{field}] AS TEXT)
                                )
                            END
                        ) AS distinct_raw_states

                    FROM {SOURCE_TABLE}
                    WHERE {DATA_ROW_PREDICATE}

                    GROUP BY
                        date,
                        course,
                        off
                )

                SELECT
                    '{field}' AS source_field,
                    COUNT(*) AS provisional_races,

                    SUM(
                        CASE
                            WHEN distinct_raw_states = 1 THEN 1
                            ELSE 0
                        END
                    ) AS internally_consistent_races,

                    SUM(
                        CASE
                            WHEN distinct_raw_states > 1 THEN 1
                            ELSE 0
                        END
                    ) AS internally_mixed_races,

                    MAX(distinct_raw_states)
                        AS maximum_states_within_race

                FROM race_values
                """,
                connection,
            )
        )


# Combine the per-field query results into three inspection tables.
field_summary = pd.concat(
    field_summary_parts,
    ignore_index=True,
)

storage_summary = pd.concat(
    storage_summary_parts,
    ignore_index=True,
)

race_consistency = pd.concat(
    race_consistency_parts,
    ignore_index=True,
)


# Add derived percentages for readability only. Counts remain the primary
# evidence and the percentages do not alter or classify any source value.
field_summary["populated_runner_pct"] = (
    field_summary["populated_rows"]
    / field_summary["runner_rows"]
    * 100
).round(3)

race_consistency["internally_mixed_race_pct"] = (
    race_consistency["internally_mixed_races"]
    / race_consistency["provisional_races"]
    * 100
).round(4)


print("Governed source population")
display(population)

print("Field availability and raw distinctness")
display(field_summary)

print("Observed SQLite storage classes")
display(storage_summary)

print("Within-provisional-race raw-state consistency")
display(race_consistency)

Governed source population


,runner_rows,provisional_races
0,1851285,189043


Field availability and raw distinctness


,source_field,runner_rows,null_rows,blank_rows,populated_rows,distinct_populated_raw_values,populated_runner_pct
0,race_name,1851285,0,0,1851285,108632,100.000
1,type,1851285,0,0,1851285,4,100.000
2,class,1851285,0,768544,1082741,7,58.486
3,pattern,1851285,0,1587927,263358,10,14.226
4,rating_band,1851285,0,1081546,769739,383,41.579
5,age_band,1851285,0,159,1851126,27,99.991
6,sex_rest,1851285,0,1612425,238860,5,12.902


Observed SQLite storage classes


,source_field,sqlite_storage_class,runner_rows
0,race_name,text,1851285
1,type,text,1851285
2,class,text,1851285
3,pattern,text,1851285
4,rating_band,text,1851285
5,age_band,text,1851285
6,sex_rest,text,1851285


Within-provisional-race raw-state consistency


,source_field,provisional_races,internally_consistent_races,internally_mixed_races,maximum_states_within_race,internally_mixed_race_pct
0,race_name,189043,189043,0,1,0.0
1,type,189043,189043,0,1,0.0
2,class,189043,189043,0,1,0.0
3,pattern,189043,189043,0,1,0.0
4,rating_band,189043,189043,0,1,0.0
5,age_band,189043,189043,0,1,0.0
6,sex_rest,189043,189043,0,1,0.0


In [2]:
# Input grain:
#   Governed source runner rows from SQLite table `data`, restricted by
#   DATA_ROW_PREDICATE = "rowid <> 1".
#
#   Because Stage 1 confirmed that all seven investigated fields are internally
#   constant within provisional races, this cell first collapses the source to
#   one row per provisional race using date + course + off.
#
# Output grain:
#   1. One cardinality and text-shape summary row per investigated field.
#   2. One row per raw value for the low-cardinality fields:
#        type, class, pattern, age_band and sex_rest.
#   3. The 60 most frequent raw rating_band values.
#   4. The 30 most frequent raw race_name values.
#
# Purpose:
#   Inspect the actual raw vocabularies before assigning any semantic meaning.
#   This establishes whether fields are small categorical vocabularies,
#   structured text, composite conditions or effectively free text.
#
# Raw versus derived values:
#   Raw values are preserved exactly as stored.
#   A derived `value_state` distinguishes blank from populated values.
#   Derived length measures use the raw text length and trimmed text length only
#   to detect surrounding whitespace or unusual formatting.
#
# Assumptions deliberately not made:
#   - no raw value is treated as canonical;
#   - similarly spelled values are not merged;
#   - case differences, punctuation and whitespace are not normalised;
#   - class labels are not assumed to be British regulatory classes;
#   - pattern labels are not assumed to map globally to Group or Grade status;
#   - rating bands are not parsed as numeric ranges;
#   - age bands are not yet treated as eligibility rules;
#   - sex restrictions are not yet interpreted;
#   - race names are not parsed for embedded race conditions.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - Stage 1 variables are unavailable;
#   - the race-level collapse does not produce exactly 189,043 rows;
#   - any investigated field is absent;
#   - any field previously found to be race-consistent unexpectedly acquires
#     multiple states during the collapse.
#
#   No source values are written or modified.

required_stage_1_names = [
    "SOURCE_DATABASE_PATH",
    "SOURCE_TABLE",
    "DATA_ROW_PREDICATE",
    "EXPECTED_PROVISIONAL_RACES",
    "CLASSIFICATION_FIELDS",
    "RACE_KEY_FIELDS",
]

missing_stage_1_names = [
    name
    for name in required_stage_1_names
    if name not in globals()
]

if missing_stage_1_names:
    raise RuntimeError(
        "Run the Stage 1 profiling cell first. "
        f"Missing variables: {missing_stage_1_names}"
    )


# Low-cardinality fields can be displayed in full without hiding rare values.
FULL_VOCABULARY_FIELDS = [
    "type",
    "class",
    "pattern",
    "age_band",
    "sex_rest",
]

# These fields have larger vocabularies, so this stage displays frequency-ranked
# samples while retaining full cardinality and text-shape statistics.
LIMITED_VOCABULARY_FIELDS = {
    "rating_band": 60,
    "race_name": 30,
}


connection_uri = f"file:{SOURCE_DATABASE_PATH}?mode=ro"

with sqlite3.connect(connection_uri, uri=True) as connection:
    # Collapse runner rows to one row per provisional race.
    #
    # MIN is used only as a deterministic selector after Stage 1 proved that
    # each investigated field has exactly one raw state within every race.
    race_level_select = ",\n".join(
        f"MIN([{field}]) AS [{field}]"
        for field in CLASSIFICATION_FIELDS
    )

    race_level = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            {race_level_select}

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}

        GROUP BY
            date,
            course,
            off
        """,
        connection,
    )


# Reconfirm the established provisional-race population after collapsing.
observed_race_rows = len(race_level)

assert observed_race_rows == EXPECTED_PROVISIONAL_RACES, (
    f"Race-level collapse produced {observed_race_rows:,} rows; "
    f"expected {EXPECTED_PROVISIONAL_RACES:,}."
)


# Confirm that all expected fields survived the race-level projection.
missing_race_level_fields = sorted(
    set(CLASSIFICATION_FIELDS) - set(race_level.columns)
)

if missing_race_level_fields:
    raise AssertionError(
        "Race-level dataframe is missing investigated fields: "
        f"{missing_race_level_fields}"
    )


field_shape_rows = []
vocabulary_parts = []

for field in CLASSIFICATION_FIELDS:
    raw_series = race_level[field]

    # All fields were text and non-null at runner grain in Stage 1, but retain
    # explicit null handling here so this cell fails transparently if the
    # source or prior conclusion changes.
    text_series = raw_series.astype("string")
    trimmed_series = text_series.str.strip()

    blank_mask = text_series.notna() & trimmed_series.eq("")
    populated_mask = text_series.notna() & trimmed_series.ne("")

    populated_raw = text_series.loc[populated_mask]

    raw_lengths = populated_raw.str.len()
    trimmed_lengths = populated_raw.str.strip().str.len()

    surrounding_whitespace_mask = raw_lengths.ne(trimmed_lengths)

    field_shape_rows.append(
        {
            "source_field": field,
            "provisional_races": len(text_series),
            "null_races": int(text_series.isna().sum()),
            "blank_races": int(blank_mask.sum()),
            "populated_races": int(populated_mask.sum()),
            "distinct_populated_raw_values": int(
                populated_raw.nunique(dropna=True)
            ),
            "minimum_raw_length": (
                int(raw_lengths.min())
                if not raw_lengths.empty
                else pd.NA
            ),
            "maximum_raw_length": (
                int(raw_lengths.max())
                if not raw_lengths.empty
                else pd.NA
            ),
            "surrounding_whitespace_races": int(
                surrounding_whitespace_mask.sum()
            ),
            "surrounding_whitespace_distinct_values": int(
                populated_raw.loc[
                    surrounding_whitespace_mask
                ].nunique(dropna=True)
            ),
        }
    )

    # Count exact raw states at provisional-race grain.
    #
    # Blank values are shown explicitly as <BLANK>. Nulls would be shown as
    # <NULL>, although Stage 1 found none.
    display_values = text_series.copy()
    display_values = display_values.fillna("<NULL>")
    display_values = display_values.mask(
        display_values.str.strip().eq(""),
        "<BLANK>",
    )

    value_counts = (
        display_values
        .value_counts(dropna=False)
        .rename_axis("raw_value")
        .reset_index(name="provisional_races")
    )

    value_counts.insert(0, "source_field", field)

    value_counts["race_pct"] = (
        value_counts["provisional_races"]
        / EXPECTED_PROVISIONAL_RACES
        * 100
    ).round(4)

    value_counts["raw_value_length"] = (
        value_counts["raw_value"]
        .astype("string")
        .str.len()
    )

    vocabulary_parts.append(value_counts)


field_shape_summary = pd.DataFrame(field_shape_rows)

raw_vocabulary = pd.concat(
    vocabulary_parts,
    ignore_index=True,
)


# Full vocabularies for fields whose Stage 1 cardinalities are small enough to
# inspect without frequency truncation.
full_vocabulary = (
    raw_vocabulary.loc[
        raw_vocabulary["source_field"].isin(FULL_VOCABULARY_FIELDS)
    ]
    .sort_values(
        ["source_field", "provisional_races", "raw_value"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


# Frequency-ranked samples for the larger structured/free-text fields.
limited_vocabulary_parts = []

for field, row_limit in LIMITED_VOCABULARY_FIELDS.items():
    field_values = (
        raw_vocabulary.loc[
            raw_vocabulary["source_field"].eq(field)
        ]
        .sort_values(
            ["provisional_races", "raw_value"],
            ascending=[False, True],
        )
        .head(row_limit)
        .copy()
    )

    limited_vocabulary_parts.append(field_values)


limited_vocabulary = pd.concat(
    limited_vocabulary_parts,
    ignore_index=True,
)


print("Race-level vocabulary and text-shape summary")
display(field_shape_summary)

print("Complete raw vocabularies for low-cardinality fields")
display(full_vocabulary)

print("Most frequent raw values for rating_band and race_name")
display(limited_vocabulary)

Race-level vocabulary and text-shape summary


,source_field,provisional_races,null_races,blank_races,populated_races,distinct_populated_raw_values,minimum_raw_length,maximum_raw_length,surrounding_whitespace_races,surrounding_whitespace_distinct_values
0,race_name,189043,0,0,189043,108632,8,100,0,0
1,type,189043,0,0,189043,4,4,7,0,0
2,class,189043,0,69207,119836,7,7,7,0,0
3,pattern,189043,0,161852,27191,10,6,7,0,0
4,rating_band,189043,0,107640,81403,383,2,8,0,0
5,age_band,189043,0,13,189030,27,3,5,0,0
6,sex_rest,189043,0,163458,25585,5,1,5,0,0


Complete raw vocabularies for low-cardinality fields


,source_field,raw_value,provisional_races,race_pct,raw_value_length
0,age_band,4yo+,62474,33.0475,4
1,age_band,3yo+,53295,28.192,4
2,age_band,3yo,25175,13.3171,3
3,age_band,2yo,19300,10.2093,3
4,age_band,5yo+,16223,8.5816,4
5,age_band,4yo,4029,2.1313,3
6,age_band,4-6yo,2634,1.3933,5
7,age_band,3-5yo,1078,0.5702,5
8,age_band,4-7yo,1021,0.5401,5
9,age_band,2yo+,884,0.4676,4


Most frequent raw values for rating_band and race_name


,source_field,raw_value,provisional_races,race_pct,raw_value_length
0,rating_band,<BLANK>,107640,56.9394,7
1,rating_band,0-100,6832,3.614,5
2,rating_band,0-75,6439,3.4061,4
3,rating_band,0-70,6337,3.3521,4
4,rating_band,0-65,5891,3.1162,4
...,...,...,...,...,...
85,race_name,Racing TV Handicap,107,0.0566,18
86,race_name,British Stallion Studs EBF Novice Stakes (GBB ...,106,0.0561,51
87,race_name,Play 4 To Win At Betway Handicap,104,0.055,32
88,race_name,Irish Stallion Farms EBF Median Auction Maiden,99,0.0524,46


In [3]:
# Input grain:
#   One row per provisional race from the `race_level` dataframe created in
#   Stage 2. The provisional race identity remains date + course + off.
#
# Output grain:
#   1. One row per exact raw rating_band value, with a provisional syntactic
#      classification and race count.
#   2. One summary row per provisional rating-band syntax family.
#   3. One row per observed combination of type, class and pattern.
#   4. One row per observed combination of type and sex_rest.
#   5. The complete set of races with blank age_band.
#
# Purpose:
#   Determine whether rating_band follows a safely recognisable syntax and
#   establish how classification fields coexist before assigning domain meaning.
#
# Raw versus derived values:
#   Raw values are retained unchanged.
#
#   Derived rating-band fields are provisional text-shape observations only:
#   - blank;
#   - closed integer range such as `0-100`;
#   - open-ended upper limit such as `0-`;
#   - single integer;
#   - any other unrecognised format.
#
#   Numeric bounds are extracted only where the entire raw string matches the
#   relevant syntax. They are not yet interpreted as official ratings,
#   eligibility limits or jurisdiction-independent scales.
#
# Assumptions deliberately not made:
#   - a rating range is not assumed to mean the same thing in every jurisdiction;
#   - zero is not assumed to be a literal lower eligibility rating;
#   - class is not derived from rating_band;
#   - pattern is not treated as a subtype of class;
#   - Group and Grade labels are not merged;
#   - blank pattern does not mean a race is definitively non-pattern;
#   - blank sex_rest does not yet mean unrestricted;
#   - the rare `C & F` value is not corrected or reclassified;
#   - blank age_band values are preserved as exceptions.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - Stage 2 has not produced `race_level`;
#   - the race-level dataframe does not contain the required fields;
#   - any nonblank rating_band value is lost from the syntax partition;
#   - parsed numeric bounds fail reconstruction of their matched raw value.
#
#   This cell performs no source writes and defines no permanent parser.

import re


required_columns = {
    "date",
    "course",
    "off",
    "race_name",
    "type",
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
}

if "race_level" not in globals():
    raise RuntimeError(
        "Run the Stage 2 race-level vocabulary cell before this cell."
    )

missing_columns = sorted(required_columns - set(race_level.columns))

if missing_columns:
    raise AssertionError(
        f"race_level is missing required columns: {missing_columns}"
    )


# Work from an explicit copy so exploratory derived columns do not alter the
# Stage 2 race-level evidence dataframe.
relationship_frame = race_level[
    [
        "date",
        "course",
        "off",
        "race_name",
        "type",
        "class",
        "pattern",
        "rating_band",
        "age_band",
        "sex_rest",
    ]
].copy()


def display_raw_state(value):
    """
    Preserve populated source text exactly, while making blank and null states
    visible in output tables.

    This is a display helper only. It does not canonicalise source values.
    """
    if pd.isna(value):
        return "<NULL>"

    if str(value).strip() == "":
        return "<BLANK>"

    return str(value)


def classify_rating_band(raw_value):
    """
    Provisionally classify the complete raw rating_band string by syntax.

    Returned values are descriptive observations, not governed semantics.
    """
    if pd.isna(raw_value):
        return {
            "rating_band_state": "null",
            "provisional_syntax": "null",
            "lower_bound": pd.NA,
            "upper_bound": pd.NA,
        }

    raw_text = str(raw_value)

    if raw_text.strip() == "":
        return {
            "rating_band_state": "blank",
            "provisional_syntax": "blank",
            "lower_bound": pd.NA,
            "upper_bound": pd.NA,
        }

    # Match a complete integer range such as 0-100 or 45-60.
    closed_range_match = re.fullmatch(r"(\d+)-(\d+)", raw_text)

    if closed_range_match:
        return {
            "rating_band_state": "populated",
            "provisional_syntax": "closed_integer_range",
            "lower_bound": int(closed_range_match.group(1)),
            "upper_bound": int(closed_range_match.group(2)),
        }

    # Match an open-ended text shape such as 0-.
    open_upper_match = re.fullmatch(r"(\d+)-", raw_text)

    if open_upper_match:
        return {
            "rating_band_state": "populated",
            "provisional_syntax": "open_upper_range",
            "lower_bound": int(open_upper_match.group(1)),
            "upper_bound": pd.NA,
        }

    # Match a single complete integer token.
    single_integer_match = re.fullmatch(r"\d+", raw_text)

    if single_integer_match:
        return {
            "rating_band_state": "populated",
            "provisional_syntax": "single_integer",
            "lower_bound": int(raw_text),
            "upper_bound": int(raw_text),
        }

    return {
        "rating_band_state": "populated",
        "provisional_syntax": "unrecognised_text",
        "lower_bound": pd.NA,
        "upper_bound": pd.NA,
    }


# Count exact raw rating-band values at provisional-race grain.
rating_value_counts = (
    relationship_frame["rating_band"]
    .map(display_raw_state)
    .value_counts(dropna=False)
    .rename_axis("raw_rating_band")
    .reset_index(name="provisional_races")
)


# Apply the provisional syntax classifier once per distinct raw value.
rating_classifications = pd.DataFrame(
    [
        {
            "raw_rating_band": raw_value,
            **classify_rating_band(
                "" if raw_value == "<BLANK>"
                else None if raw_value == "<NULL>"
                else raw_value
            ),
        }
        for raw_value in rating_value_counts["raw_rating_band"]
    ]
)


rating_band_profile = (
    rating_value_counts
    .merge(
        rating_classifications,
        on="raw_rating_band",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["provisional_syntax", "lower_bound", "upper_bound", "raw_rating_band"],
        na_position="last",
    )
    .reset_index(drop=True)
)


# Validate that every exact raw state is assigned to exactly one syntax family.
if rating_band_profile["provisional_syntax"].isna().any():
    raise AssertionError(
        "At least one rating_band value was not assigned a syntax family."
    )

observed_rating_races = int(
    rating_band_profile["provisional_races"].sum()
)

if observed_rating_races != EXPECTED_PROVISIONAL_RACES:
    raise AssertionError(
        "Rating-band syntax partition does not reconstruct the race population: "
        f"{observed_rating_races:,} != {EXPECTED_PROVISIONAL_RACES:,}"
    )


# Validate exact reconstruction for syntaxes where numeric components were
# extracted. This prevents a permissive regex from silently accepting only part
# of a raw value.
closed_range_rows = rating_band_profile.loc[
    rating_band_profile["provisional_syntax"].eq("closed_integer_range")
].copy()

closed_range_rows["reconstructed"] = (
    closed_range_rows["lower_bound"].astype("Int64").astype("string")
    + "-"
    + closed_range_rows["upper_bound"].astype("Int64").astype("string")
)

if not closed_range_rows["reconstructed"].eq(
    closed_range_rows["raw_rating_band"]
).all():
    raise AssertionError(
        "At least one closed rating range failed exact reconstruction."
    )


single_integer_rows = rating_band_profile.loc[
    rating_band_profile["provisional_syntax"].eq("single_integer")
].copy()

single_integer_rows["reconstructed"] = (
    single_integer_rows["lower_bound"]
    .astype("Int64")
    .astype("string")
)

if not single_integer_rows["reconstructed"].eq(
    single_integer_rows["raw_rating_band"]
).all():
    raise AssertionError(
        "At least one single-integer rating value failed exact reconstruction."
    )


# Summarise syntax families while preserving counts of distinct raw formats.
rating_syntax_summary = (
    rating_band_profile
    .groupby(
        ["rating_band_state", "provisional_syntax"],
        dropna=False,
        as_index=False,
    )
    .agg(
        distinct_raw_values=("raw_rating_band", "nunique"),
        provisional_races=("provisional_races", "sum"),
        minimum_lower_bound=("lower_bound", "min"),
        maximum_lower_bound=("lower_bound", "max"),
        minimum_upper_bound=("upper_bound", "min"),
        maximum_upper_bound=("upper_bound", "max"),
    )
)

rating_syntax_summary["race_pct"] = (
    rating_syntax_summary["provisional_races"]
    / EXPECTED_PROVISIONAL_RACES
    * 100
).round(4)


# Make blank states explicit for relationship tables without modifying raw data.
for field in ["class", "pattern", "sex_rest"]:
    relationship_frame[f"{field}_display"] = (
        relationship_frame[field].map(display_raw_state)
    )


# Inspect whether class and pattern coexist differently across race types.
type_class_pattern = (
    relationship_frame
    .groupby(
        ["type", "class_display", "pattern_display"],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "provisional_races"})
    .sort_values(
        ["type", "provisional_races", "class_display", "pattern_display"],
        ascending=[True, False, True, True],
    )
    .reset_index(drop=True)
)

type_class_pattern["within_type_pct"] = (
    type_class_pattern["provisional_races"]
    / type_class_pattern.groupby("type")["provisional_races"].transform("sum")
    * 100
).round(4)


# Inspect the raw sex-rest vocabulary separately by race type.
type_sex_restriction = (
    relationship_frame
    .groupby(
        ["type", "sex_rest_display"],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "provisional_races"})
    .sort_values(
        ["type", "provisional_races", "sex_rest_display"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

type_sex_restriction["within_type_pct"] = (
    type_sex_restriction["provisional_races"]
    / type_sex_restriction.groupby("type")["provisional_races"].transform("sum")
    * 100
).round(4)


# Preserve every race whose age_band is blank because these 13 exceptions are
# likely to be high-value evidence for jurisdiction or extraction behaviour.
blank_age_band_races = (
    relationship_frame.loc[
        relationship_frame["age_band"].astype("string").str.strip().eq(""),
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "rating_band",
            "sex_rest",
        ],
    ]
    .sort_values(["date", "course", "off"])
    .reset_index(drop=True)
)


print("Exact raw rating-band values with provisional syntax")
display(rating_band_profile)

print("Rating-band syntax summary")
display(rating_syntax_summary)

print("Observed type, class and pattern combinations")
display(type_class_pattern)

print("Observed type and sex-rest combinations")
display(type_sex_restriction)

print("All provisional races with blank age_band")
display(blank_age_band_races)

Exact raw rating-band values with provisional syntax


,raw_rating_band,provisional_races,rating_band_state,provisional_syntax,lower_bound,upper_bound
0,<BLANK>,107640,blank,blank,<NA>,<NA>
1,0-40,7,populated,closed_integer_range,0,40
2,0-45,1,populated,closed_integer_range,0,45
3,0-50,1154,populated,closed_integer_range,0,50
4,0-52,428,populated,closed_integer_range,0,52
...,...,...,...,...,...,...
379,100-114,1,populated,closed_integer_range,100,114
380,100-115,2,populated,closed_integer_range,100,115
381,100-123,4,populated,closed_integer_range,100,123
382,(75-100),1,populated,unrecognised_text,<NA>,<NA>


Rating-band syntax summary


,rating_band_state,provisional_syntax,distinct_raw_values,provisional_races,minimum_lower_bound,maximum_lower_bound,minimum_upper_bound,maximum_upper_bound,race_pct
0,blank,blank,1,107640,NaN,NaN,NaN,NaN,56.9394
1,populated,closed_integer_range,381,81390,0.0,100.0,40.0,155.0,43.0537
2,populated,unrecognised_text,2,13,NaN,NaN,NaN,NaN,0.0069


Observed type, class and pattern combinations


,type,class_display,pattern_display,provisional_races,within_type_pct
0,Chase,Class 4,<BLANK>,5602,24.8459
1,Chase,<BLANK>,<BLANK>,5375,23.8391
2,Chase,Class 3,<BLANK>,4060,18.0068
3,Chase,Class 5,<BLANK>,3292,14.6006
4,Chase,Class 2,<BLANK>,1325,5.8766
...,...,...,...,...,...
70,NH Flat,<BLANK>,Grade 1,14,0.3015
71,NH Flat,<BLANK>,Grade 3,12,0.2585
72,NH Flat,Class 1,Grade 1,12,0.2585
73,NH Flat,Class 1,Grade 3,2,0.0431


Observed type and sex-rest combinations


,type,sex_rest_display,provisional_races,within_type_pct
0,Chase,<BLANK>,21507,95.3874
1,Chase,M,937,4.1558
2,Chase,F & M,77,0.3415
3,Chase,F,23,0.1020
4,Chase,C & G,3,0.0133
5,Flat,<BLANK>,107937,85.3993
6,Flat,F,12722,10.0656
7,Flat,F & M,3560,2.8167
8,Flat,C & G,2089,1.6528
9,Flat,M,78,0.0617


All provisional races with blank age_band


,date,course,off,race_name,type,class,pattern,rating_band,sex_rest
0,2015-06-07,Baden-Baden (GER),3:30,Badener Roulette Preis (Turf),Flat,,,,
1,2021-05-27,Limerick (IRE),8:10,Athea INH Flat Race,NH Flat,,,,
2,2021-07-16,Kilbeggan (IRE),8:10,Follow Kilbeggan On Facebook INH Flat Race,NH Flat,,,,
3,2026-02-14,Sha Tin,04:45,Tvb Lok Sin Tong Charity Corner Hcp (C4) (Hand...,Flat,,,,
4,2026-02-14,Sha Tin,05:40,Tvb Yan Oi Tong Charity Show Hcp (C4) (Handica...,Flat,,,,
5,2026-02-14,Sha Tin,06:10,Tvb Yan Chai Charity Show Hcp (C3) (Handicap) ...,Flat,,,,
6,2026-02-14,Sha Tin,06:40,Tvb Pok Oi Charity Show Hcp (C4) (Handicap) (T...,Flat,,,,
7,2026-02-14,Sha Tin,07:10,Tvb Lo And Behold Hcp (C3) (Handicap) (Dirt),Flat,,,,
8,2026-02-14,Sha Tin,07:40,Tvb The Queen Of News Hcp (C4) (Handicap) (Dirt),Flat,,,,
9,2026-02-14,Sha Tin,08:15,The Tvb Cup (C2) (Handicap) (Turf),Flat,,,,


In [4]:
# Input grain:
#   One row per provisional race from `relationship_frame`, created in Stage 3.
#
# Output grain:
#   1. One coverage-summary row per course label and race type.
#   2. One coverage-summary row per year and race type.
#   3. One detailed row for every race carrying an unrecognised rating_band.
#   4. One summary row per course label represented among those exceptions.
#
# Purpose:
#   Determine whether classification-field availability and the 13 rating-band
#   exceptions are concentrated by course, period or race type.
#
#   This is an intermediate source-pattern study. It does not yet assign a
#   formal jurisdiction or regulatory meaning to any field.
#
# Raw versus derived values:
#   Raw race fields remain unchanged.
#
#   Derived values are limited to:
#   - calendar year extracted from the raw race date;
#   - Boolean populated-state indicators;
#   - the provisional rating-band syntax already established in Stage 3;
#   - exact source course-label summaries.
#
# Assumptions deliberately not made:
#   - a course suffix is not automatically treated as a governed jurisdiction;
#   - course labels without suffixes are not assumed to be British;
#   - blank fields are not assumed to mean unrestricted or not applicable;
#   - `--` is not converted to blank or null;
#   - `(75-100)` is not silently normalised to `75-100`;
#   - the 13 exceptions are not yet declared source errors;
#   - frequency by course does not by itself prove a regulatory convention.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - Stage 3 variables are unavailable;
#   - the race population differs from 189,043;
#   - the exception population differs from the 13 races observed in Stage 3;
#   - coverage counts fail to reconstruct their parent populations.
#
#   The source database is not queried or modified by this cell.

required_stage_3_names = [
    "relationship_frame",
    "rating_band_profile",
    "EXPECTED_PROVISIONAL_RACES",
]

missing_stage_3_names = [
    name
    for name in required_stage_3_names
    if name not in globals()
]

if missing_stage_3_names:
    raise RuntimeError(
        "Run the Stage 3 relationship and rating-band cell first. "
        f"Missing variables: {missing_stage_3_names}"
    )


# Work from a fresh copy so the preceding evidence dataframes remain unchanged.
coverage_frame = relationship_frame.copy()


# Parse only enough of the raw date to derive a calendar year for coverage
# analysis. Invalid dates fail visibly rather than becoming silent nulls.
coverage_frame["parsed_date"] = pd.to_datetime(
    coverage_frame["date"],
    errors="raise",
)

coverage_frame["year"] = coverage_frame["parsed_date"].dt.year


# Create explicit populated-state indicators.
#
# These indicators do not classify the content. They record only whether the
# source supplied nonblank text for each field.
for field in [
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
]:
    coverage_frame[f"{field}_populated"] = (
        coverage_frame[field]
        .astype("string")
        .str.strip()
        .ne("")
    )


# Apply the Stage 3 syntax classifier at race grain.
rating_syntax_rows = coverage_frame["rating_band"].map(
    classify_rating_band
)

rating_syntax_frame = pd.DataFrame(
    rating_syntax_rows.tolist(),
    index=coverage_frame.index,
)

coverage_frame = pd.concat(
    [coverage_frame, rating_syntax_frame],
    axis=1,
)


# Validate the race population before producing grouped summaries.
if len(coverage_frame) != EXPECTED_PROVISIONAL_RACES:
    raise AssertionError(
        "Coverage dataframe does not match the established race population: "
        f"{len(coverage_frame):,} != {EXPECTED_PROVISIONAL_RACES:,}"
    )


def build_coverage_summary(frame, grouping_fields):
    """
    Summarise field availability for an exact grouping.

    Output counts remain provisional-race counts. Percentages describe the
    proportion of races within each group carrying nonblank source text.
    """
    summary = (
        frame
        .groupby(
            grouping_fields,
            dropna=False,
            as_index=False,
        )
        .agg(
            provisional_races=("race_name", "size"),
            class_populated_races=("class_populated", "sum"),
            pattern_populated_races=("pattern_populated", "sum"),
            rating_band_populated_races=("rating_band_populated", "sum"),
            age_band_populated_races=("age_band_populated", "sum"),
            sex_rest_populated_races=("sex_rest_populated", "sum"),
            unrecognised_rating_band_races=(
                "provisional_syntax",
                lambda values: int(
                    values.eq("unrecognised_text").sum()
                ),
            ),
        )
    )

    percentage_columns = [
        "class",
        "pattern",
        "rating_band",
        "age_band",
        "sex_rest",
    ]

    for field in percentage_columns:
        summary[f"{field}_coverage_pct"] = (
            summary[f"{field}_populated_races"]
            / summary["provisional_races"]
            * 100
        ).round(3)

    return summary


# Course/type coverage is useful because source conventions often enter through
# a particular feed, venue group or geographical subset.
course_type_coverage = build_coverage_summary(
    coverage_frame,
    ["course", "type"],
).sort_values(
    [
        "unrecognised_rating_band_races",
        "provisional_races",
        "course",
        "type",
    ],
    ascending=[False, False, True, True],
).reset_index(drop=True)


# Year/type coverage tests whether field availability changes materially over
# time or merely reflects the composition of jurisdictions and race codes.
year_type_coverage = build_coverage_summary(
    coverage_frame,
    ["year", "type"],
).sort_values(
    ["year", "type"],
).reset_index(drop=True)


# Preserve every unrecognised raw rating-band race for direct inspection.
unrecognised_rating_band_races = (
    coverage_frame.loc[
        coverage_frame["provisional_syntax"].eq("unrecognised_text"),
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "rating_band",
            "age_band",
            "sex_rest",
        ],
    ]
    .sort_values(["date", "course", "off"])
    .reset_index(drop=True)
)


# The previous stage established exactly 13 such races.
EXPECTED_UNRECOGNISED_RATING_RACES = 13

if len(unrecognised_rating_band_races) != EXPECTED_UNRECOGNISED_RATING_RACES:
    raise AssertionError(
        "Unrecognised rating-band population changed: "
        f"{len(unrecognised_rating_band_races):,} != "
        f"{EXPECTED_UNRECOGNISED_RATING_RACES:,}"
    )


# Summarise the exception residue by exact course label and raw value.
unrecognised_rating_by_course = (
    unrecognised_rating_band_races
    .groupby(
        ["course", "type", "rating_band"],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        distinct_race_names=("race_name", "nunique"),
    )
    .sort_values(
        ["provisional_races", "course", "rating_band"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)


# Reconcile the grouped course/type counts to the complete race population.
course_type_total = int(
    course_type_coverage["provisional_races"].sum()
)

if course_type_total != EXPECTED_PROVISIONAL_RACES:
    raise AssertionError(
        "Course/type coverage does not reconstruct the race population: "
        f"{course_type_total:,} != {EXPECTED_PROVISIONAL_RACES:,}"
    )


# Each race belongs to exactly one year/type group as well.
year_type_total = int(
    year_type_coverage["provisional_races"].sum()
)

if year_type_total != EXPECTED_PROVISIONAL_RACES:
    raise AssertionError(
        "Year/type coverage does not reconstruct the race population: "
        f"{year_type_total:,} != {EXPECTED_PROVISIONAL_RACES:,}"
    )


print("Course and race-type coverage")
display(course_type_coverage)

print("Year and race-type coverage")
display(year_type_coverage)

print("All races with unrecognised rating-band syntax")
display(unrecognised_rating_band_races)

print("Unrecognised rating-band races by course and raw value")
display(unrecognised_rating_by_course)

Course and race-type coverage


,course,type,provisional_races,class_populated_races,pattern_populated_races,rating_band_populated_races,age_band_populated_races,sex_rest_populated_races,unrecognised_rating_band_races,class_coverage_pct,pattern_coverage_pct,rating_band_coverage_pct,age_band_coverage_pct,sex_rest_coverage_pct
0,Les Landes (JER),Flat,349,0,0,74,349,6,4,0.0,0.0,21.203,100.0,1.719
1,Jebel Ali (UAE),Flat,606,0,27,360,606,16,3,0.0,4.455,59.406,100.0,2.64
2,Ohi (JPN),Flat,35,0,35,2,35,4,2,0.0,100.0,5.714,100.0,11.429
3,Happy Valley,Flat,251,223,1,18,250,0,1,88.845,0.398,7.171,99.602,0.0
4,Les Landes (JER),Hurdle,84,0,0,4,84,0,1,0.0,0.0,4.762,100.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830,Wagga Wagga (AUS),Flat,1,0,0,0,1,0,0,0.0,0.0,0.0,100.0,0.0
831,Wangaratta (AUS),Flat,1,0,0,0,1,1,0,0.0,0.0,0.0,100.0,100.0
832,Waregem (BEL),Chase,1,0,0,0,1,0,0,0.0,0.0,0.0,100.0,0.0
833,Werribee (AUS),Flat,1,0,0,0,1,0,0,0.0,0.0,0.0,100.0,0.0


Year and race-type coverage


,year,type,provisional_races,class_populated_races,pattern_populated_races,rating_band_populated_races,age_band_populated_races,sex_rest_populated_races,unrecognised_rating_band_races,class_coverage_pct,pattern_coverage_pct,rating_band_coverage_pct,age_band_coverage_pct,sex_rest_coverage_pct
0,2015,Chase,2080,1454,212,1134,2080,62,0,69.904,10.192,54.519,100.0,2.981
1,2015,Flat,10984,6277,2110,4554,10983,1941,0,57.147,19.21,41.46,99.991,17.671
2,2015,Hurdle,3098,2032,236,1284,3098,417,0,65.591,7.618,41.446,100.0,13.46
3,2015,NH Flat,447,295,14,0,447,88,0,65.996,3.132,0.0,100.0,19.687
4,2016,Chase,2064,1435,229,1099,2064,64,0,69.525,11.095,53.246,100.0,3.101
5,2016,Flat,10604,6392,2063,4627,10604,1766,1,60.279,19.455,43.634,100.0,16.654
6,2016,Hurdle,3131,2035,248,1293,3131,446,0,64.995,7.921,41.297,100.0,14.245
7,2016,NH Flat,436,278,16,0,436,105,0,63.761,3.67,0.0,100.0,24.083
8,2017,Chase,2112,1495,226,1120,2112,78,0,70.786,10.701,53.03,100.0,3.693
9,2017,Flat,11483,7174,2044,4725,11483,1837,2,62.475,17.8,41.148,100.0,15.998


All races with unrecognised rating-band syntax


,date,course,off,race_name,type,class,pattern,rating_band,age_band,sex_rest
0,2016-02-19,Jebel Ali (UAE),11:20,Jebel Ali Stakes () (Dirt),Flat,,Listed,--,3yo+,
1,2017-11-03,Ohi (JPN),10:07,JBC Sprint (Local (Dirt),Flat,,Grade 1,--,3yo+,
2,2017-11-03,Ohi (JPN),11:07,JBC Classic (Local (Dirt),Flat,,Grade 1,--,3yo+,
3,2018-01-26,Jebel Ali (UAE),11:30,Jebel Ali Mile Sponsored By Shadwell Farm (Dirt),Flat,,Group 3,--,3yo+,
4,2018-04-30,NAGOYA (JPN),11:07,Kakitsubata Kinen (Handicap) (4yo+) (Local (Dirt),Flat,,Grade 3,--,4yo+,
5,2018-05-13,Les Landes (JER),4:50,Bloodstock Advisory Services Handicap,Flat,,,--,3yo+,
6,2018-05-30,Urawa (JPN),11:07,Sakitama Hai (Local (Dirt),Flat,,Grade 2,--,4yo+,
7,2018-07-22,Les Landes (JER),2:30,Milbrook July Conditions Hurdle,Hurdle,,,--,3yo+,
8,2018-07-22,Les Landes (JER),3:05,Patricia K Pritchard Handicap Sprint,Flat,,,--,3yo+,
9,2018-07-22,Les Landes (JER),3:40,La Vallette 2018 Jersey Derby,Flat,,,--,3yo+,


Unrecognised rating-band races by course and raw value


,course,type,rating_band,provisional_races,first_date,last_date,distinct_race_names
0,Les Landes (JER),Flat,--,4,2018-05-13,2019-06-21,4
1,Jebel Ali (UAE),Flat,--,3,2016-02-19,2021-11-26,3
2,Ohi (JPN),Flat,--,2,2017-11-03,2017-11-03,2
3,Happy Valley,Flat,(75-100),1,2026-02-25,2026-02-25,1
4,Les Landes (JER),Hurdle,--,1,2018-07-22,2018-07-22,1
5,NAGOYA (JPN),Flat,--,1,2018-04-30,2018-04-30,1
6,Urawa (JPN),Flat,--,1,2018-05-30,2018-05-30,1


In [5]:
# Input grain:
#   One row per provisional race from `coverage_frame`, created in Stage 4.
#
# Output grain:
#   1. One coverage row per governed jurisdiction and source race type.
#   2. One raw-value row per jurisdiction, field and exact source value for
#      class, pattern, rating_band, age_band and sex_rest.
#   3. One detailed row for each unrecognised rating-band race with governed
#      jurisdiction attached.
#
# Purpose:
#   Establish whether the compact classification and eligibility vocabularies
#   are jurisdiction-specific before defining any parser or shared semantics.
#
# Raw versus derived values:
#   Raw source values remain unchanged.
#
#   Candidate course identity and jurisdiction are derived through the existing
#   governed course-location implementation. The resulting jurisdiction is used
#   only to group and inspect the raw classification fields.
#
# Assumptions deliberately not made:
#   - unsuffixed courses are not automatically treated as Great Britain;
#   - Group and Grade labels are not treated as globally equivalent;
#   - Class 1–7 is not assumed to have the same meaning worldwide;
#   - rating-band ranges are not assumed to use one global scale;
#   - `--` is not converted to blank, null or unrestricted;
#   - `(75-100)` is not normalised to `75-100`;
#   - blank sex_rest is not interpreted as unrestricted eligibility.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - Stage 4 variables are unavailable;
#   - the local `src` package or governed reference cannot be found;
#   - any source race fails to join to a governed course identity;
#   - the join changes the established 189,043-race population;
#   - the 13 unrecognised rating-band races are not preserved.
#
#   No source or reference data is written or modified.

import sys
from pathlib import Path


# Make the repository's src-layout package visible to the notebook kernel.
SOURCE_PACKAGE_ROOT = PROJECT_ROOT / "src"

if not SOURCE_PACKAGE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Source package directory not found: {SOURCE_PACKAGE_ROOT}"
    )

source_package_text = str(SOURCE_PACKAGE_ROOT)

if source_package_text not in sys.path:
    sys.path.insert(0, source_package_text)


from inside_rails.course_locations import (
    load_course_locations,
    merge_source_course_locations,
)


if "coverage_frame" not in globals():
    raise RuntimeError(
        "Run the Stage 4 course/year coverage cell before this cell."
    )


COURSE_LOCATION_REFERENCE_PATH = (
    PROJECT_ROOT
    / "data/reference/course_locations.csv"
)

if not COURSE_LOCATION_REFERENCE_PATH.is_file():
    raise FileNotFoundError(
        "Governed course-location reference not found: "
        f"{COURSE_LOCATION_REFERENCE_PATH}"
    )


# Load and validate the governed course-location reference.
course_reference = load_course_locations(
    COURSE_LOCATION_REFERENCE_PATH
)


# Retain the raw context needed by the governed identity derivation and the
# classification fields required by this stage.
source_context = coverage_frame[
    [
        "date",
        "course",
        "type",
        "race_name",
        "off",
        "class",
        "pattern",
        "rating_band",
        "age_band",
        "sex_rest",
        "class_populated",
        "pattern_populated",
        "rating_band_populated",
        "age_band_populated",
        "sex_rest_populated",
        "provisional_syntax",
    ]
].copy()


# Derive candidate course identity and jurisdiction, then attach the governed
# course reference. Strict matching prevents unresolved labels being guessed.
jurisdiction_frame = merge_source_course_locations(
    source_context,
    course_reference,
    require_all_matches=True,
)


# Confirm that the many-to-one reference join preserved race grain.
if len(jurisdiction_frame) != EXPECTED_PROVISIONAL_RACES:
    raise AssertionError(
        "Governed course-location join changed the race population: "
        f"{len(jurisdiction_frame):,} != "
        f"{EXPECTED_PROVISIONAL_RACES:,}"
    )


if "candidate_jurisdiction" not in jurisdiction_frame.columns:
    raise AssertionError(
        "Governed join did not return candidate_jurisdiction."
    )


# Use a shorter exploratory display column while preserving the governed field.
jurisdiction_frame["jurisdiction"] = (
    jurisdiction_frame["candidate_jurisdiction"]
)


# Make blank and null states visible in output tables without altering raw text.
for field in [
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
]:
    jurisdiction_frame[f"{field}_display"] = (
        jurisdiction_frame[field].map(display_raw_state)
    )


# Summarise availability by governed jurisdiction and source race type.
jurisdiction_type_coverage = (
    jurisdiction_frame
    .groupby(
        ["jurisdiction", "type"],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        class_populated_races=("class_populated", "sum"),
        pattern_populated_races=("pattern_populated", "sum"),
        rating_band_populated_races=("rating_band_populated", "sum"),
        age_band_populated_races=("age_band_populated", "sum"),
        sex_rest_populated_races=("sex_rest_populated", "sum"),
        unrecognised_rating_band_races=(
            "provisional_syntax",
            lambda values: int(
                values.eq("unrecognised_text").sum()
            ),
        ),
    )
)


for field in [
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
]:
    jurisdiction_type_coverage[f"{field}_coverage_pct"] = (
        jurisdiction_type_coverage[f"{field}_populated_races"]
        / jurisdiction_type_coverage["provisional_races"]
        * 100
    ).round(3)


jurisdiction_type_coverage = (
    jurisdiction_type_coverage
    .sort_values(
        [
            "provisional_races",
            "jurisdiction",
            "type",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)


# Build exact raw vocabularies by jurisdiction and race type.
jurisdiction_vocabulary_parts = []

for field in [
    "class",
    "pattern",
    "rating_band",
    "age_band",
    "sex_rest",
]:
    field_vocabulary = (
        jurisdiction_frame
        .groupby(
            [
                "jurisdiction",
                "type",
                f"{field}_display",
            ],
            dropna=False,
            as_index=False,
        )
        .size()
        .rename(
            columns={
                f"{field}_display": "raw_value",
                "size": "provisional_races",
            }
        )
    )

    field_vocabulary.insert(
        2,
        "source_field",
        field,
    )

    jurisdiction_vocabulary_parts.append(
        field_vocabulary
    )


jurisdiction_vocabulary = (
    pd.concat(
        jurisdiction_vocabulary_parts,
        ignore_index=True,
    )
    .sort_values(
        [
            "source_field",
            "jurisdiction",
            "type",
            "provisional_races",
            "raw_value",
        ],
        ascending=[True, True, True, False, True],
    )
    .reset_index(drop=True)
)


# Preserve all 13 unrecognised rating-band races with governed jurisdiction.
unrecognised_rating_with_jurisdiction = (
    jurisdiction_frame.loc[
        jurisdiction_frame["provisional_syntax"].eq(
            "unrecognised_text"
        ),
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "rating_band",
            "age_band",
            "sex_rest",
        ],
    ]
    .sort_values(
        ["jurisdiction", "date", "course", "off"]
    )
    .reset_index(drop=True)
)


if len(unrecognised_rating_with_jurisdiction) != 13:
    raise AssertionError(
        "Jurisdiction-enriched rating exception population changed: "
        f"{len(unrecognised_rating_with_jurisdiction):,} != 13"
    )


# Reconcile grouped coverage to the full provisional-race population.
coverage_total = int(
    jurisdiction_type_coverage["provisional_races"].sum()
)

if coverage_total != EXPECTED_PROVISIONAL_RACES:
    raise AssertionError(
        "Jurisdiction/type coverage does not reconstruct the race population: "
        f"{coverage_total:,} != {EXPECTED_PROVISIONAL_RACES:,}"
    )


print("Classification-field coverage by jurisdiction and race type")
display(jurisdiction_type_coverage)

print("Exact compact-field vocabularies by jurisdiction and race type")
display(jurisdiction_vocabulary)

print("Unrecognised rating-band races with governed jurisdiction")
display(unrecognised_rating_with_jurisdiction)

Classification-field coverage by jurisdiction and race type


,jurisdiction,type,provisional_races,class_populated_races,pattern_populated_races,rating_band_populated_races,age_band_populated_races,sex_rest_populated_races,unrecognised_rating_band_races,class_coverage_pct,pattern_coverage_pct,rating_band_coverage_pct,age_band_coverage_pct,sex_rest_coverage_pct
0,Great Britain,Flat,70218,70218,3122,48690,70218,7996,0,100.0,4.446,69.341,100.0,11.387
1,Great Britain,Hurdle,22645,22645,973,12168,22645,2715,0,100.0,4.297,53.734,100.0,11.989
2,Great Britain,Chase,15671,15671,966,11501,15671,593,0,100.0,6.164,73.39,100.0,3.784
3,France,Flat,15514,108,2614,0,15514,3297,0,0.696,16.849,0.0,100.0,21.252
4,Ireland,Flat,14763,51,1431,3920,14763,2183,0,0.345,9.693,26.553,100.0,14.787
5,Ireland,Hurdle,9667,70,917,2740,9667,1261,0,0.724,9.486,28.344,100.0,13.044
6,Hong Kong,Flat,7481,6999,323,47,7471,0,1,93.557,4.318,0.628,99.866,0.0
7,United States,Flat,6023,227,4844,0,6023,2264,0,3.769,80.425,0.0,100.0,37.589
8,Ireland,Chase,4833,75,963,869,4833,322,0,1.552,19.926,17.981,100.0,6.663
9,Australia,Flat,4059,266,3770,0,4059,1149,0,6.553,92.88,0.0,100.0,28.307


Exact compact-field vocabularies by jurisdiction and race type


,jurisdiction,type,source_field,raw_value,provisional_races
0,Argentina,Flat,age_band,3yo+,239
1,Argentina,Flat,age_band,3yo,107
2,Argentina,Flat,age_band,2yo,63
3,Argentina,Flat,age_band,4yo+,25
4,Argentina,Flat,age_band,2yo+,2
...,...,...,...,...,...
1279,United States,Hurdle,sex_rest,F & M,6
1280,Uruguay,Flat,sex_rest,<BLANK>,39
1281,Uruguay,Flat,sex_rest,F,14
1282,Uruguay,Flat,sex_rest,F & M,10


Unrecognised rating-band races with governed jurisdiction


,date,course,jurisdiction,off,race_name,type,class,pattern,rating_band,age_band,sex_rest
0,2026-02-25,Happy Valley,Hong Kong,11:40,Magazine Gap Handicap (B Course) (Turf),Flat,Class 2,,(75-100),,
1,2017-11-03,Ohi (JPN),Japan,10:07,JBC Sprint (Local (Dirt),Flat,,Grade 1,--,3yo+,
2,2017-11-03,Ohi (JPN),Japan,11:07,JBC Classic (Local (Dirt),Flat,,Grade 1,--,3yo+,
3,2018-04-30,NAGOYA (JPN),Japan,11:07,Kakitsubata Kinen (Handicap) (4yo+) (Local (Dirt),Flat,,Grade 3,--,4yo+,
4,2018-05-30,Urawa (JPN),Japan,11:07,Sakitama Hai (Local (Dirt),Flat,,Grade 2,--,4yo+,
5,2018-05-13,Les Landes (JER),Jersey,4:50,Bloodstock Advisory Services Handicap,Flat,,,--,3yo+,
6,2018-07-22,Les Landes (JER),Jersey,2:30,Milbrook July Conditions Hurdle,Hurdle,,,--,3yo+,
7,2018-07-22,Les Landes (JER),Jersey,3:05,Patricia K Pritchard Handicap Sprint,Flat,,,--,3yo+,
8,2018-07-22,Les Landes (JER),Jersey,3:40,La Vallette 2018 Jersey Derby,Flat,,,--,3yo+,
9,2019-06-21,Les Landes (JER),Jersey,8:50,Mid-Summer Glorious Les Landes Handicap,Flat,,,--,3yo+,


In [6]:
# Input grain:
#   Governed source runner rows from SQLite table `data`, restricted by
#   DATA_ROW_PREDICATE = "rowid <> 1".
#
#   Race-level classification values come from `jurisdiction_frame`, where
#   governed course identity and jurisdiction have already been attached.
#
# Output grain:
#   1. One parser-status row per exact raw age_band value.
#   2. One age-consistency summary row per raw age_band value.
#   3. One detailed row per runner whose recorded age falls outside the parsed
#      age-band limits.
#   4. One sex-rest consistency summary row per raw sex_rest value.
#   5. One detailed row per runner whose raw sex value is not admitted by the
#      provisional sex-rest token mapping.
#
# Purpose:
#   Test whether `age_band` and `sex_rest` behave like eligibility conditions
#   by comparing them with the actual recorded characteristics of runners in
#   the same race.
#
# Raw versus derived values:
#   Raw race and runner values remain unchanged.
#
#   Derived age-band components are limited to exact observed formats:
#   - `Nyo`       -> minimum age N and maximum age N;
#   - `Nyo+`      -> minimum age N and no stated maximum;
#   - `N-Myo`     -> minimum age N and maximum age M.
#
#   Derived sex-rest membership is provisional and based only on the observed
#   source abbreviations:
#   - F     -> F;
#   - M     -> M;
#   - F & M -> F or M;
#   - C & G -> C or G;
#   - C & F -> C or F.
#
# Assumptions deliberately not made:
#   - blank age_band is not parsed or inferred from race_name;
#   - runner age zero or malformed age is not silently corrected;
#   - an out-of-band runner is not automatically declared ineligible;
#   - amendments, substitutions and source errors remain possible;
#   - blank sex_rest is not interpreted as unrestricted in this cell;
#   - sex codes are not expanded beyond their literal source tokens;
#   - rare contradictions are preserved for later manual verification.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - required prior-stage variables are unavailable;
#   - the source runner population differs from 1,851,285;
#   - a populated age_band value cannot be assigned exactly once to either the
#     supported parser or an explicit unrecognised state;
#   - parsed age bands fail exact text reconstruction;
#   - the runner-to-race merge changes runner grain or creates duplicate rows.
#
#   No source or reference values are written or modified.

import re


required_prior_names = [
    "jurisdiction_frame",
    "SOURCE_DATABASE_PATH",
    "SOURCE_TABLE",
    "DATA_ROW_PREDICATE",
    "EXPECTED_RUNNER_ROWS",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding stages first. "
        f"Missing variables: {missing_prior_names}"
    )


def parse_age_band(raw_value):
    """
    Parse only complete age-band formats observed in the source.

    The result describes text structure. It does not yet establish that the
    source value is a formal regulatory eligibility rule.
    """
    if pd.isna(raw_value):
        return {
            "age_band_state": "null",
            "age_band_syntax": "null",
            "minimum_age": pd.NA,
            "maximum_age": pd.NA,
            "open_ended": pd.NA,
        }

    raw_text = str(raw_value)

    if raw_text.strip() == "":
        return {
            "age_band_state": "blank",
            "age_band_syntax": "blank",
            "minimum_age": pd.NA,
            "maximum_age": pd.NA,
            "open_ended": pd.NA,
        }

    exact_age_match = re.fullmatch(r"(\d+)yo", raw_text)

    if exact_age_match:
        age = int(exact_age_match.group(1))

        return {
            "age_band_state": "populated",
            "age_band_syntax": "exact_age",
            "minimum_age": age,
            "maximum_age": age,
            "open_ended": False,
        }

    open_ended_match = re.fullmatch(r"(\d+)yo\+", raw_text)

    if open_ended_match:
        return {
            "age_band_state": "populated",
            "age_band_syntax": "minimum_age_open_ended",
            "minimum_age": int(open_ended_match.group(1)),
            "maximum_age": pd.NA,
            "open_ended": True,
        }

    closed_range_match = re.fullmatch(r"(\d+)-(\d+)yo", raw_text)

    if closed_range_match:
        return {
            "age_band_state": "populated",
            "age_band_syntax": "closed_age_range",
            "minimum_age": int(closed_range_match.group(1)),
            "maximum_age": int(closed_range_match.group(2)),
            "open_ended": False,
        }

    return {
        "age_band_state": "populated",
        "age_band_syntax": "unrecognised_text",
        "minimum_age": pd.NA,
        "maximum_age": pd.NA,
        "open_ended": pd.NA,
    }


# Classify each exact age-band source value once.
age_band_values = (
    jurisdiction_frame["age_band"]
    .map(display_raw_state)
    .value_counts(dropna=False)
    .rename_axis("raw_age_band")
    .reset_index(name="provisional_races")
)

age_band_parser_rows = []

for raw_value in age_band_values["raw_age_band"]:
    parser_input = (
        None
        if raw_value == "<NULL>"
        else ""
        if raw_value == "<BLANK>"
        else raw_value
    )

    age_band_parser_rows.append(
        {
            "raw_age_band": raw_value,
            **parse_age_band(parser_input),
        }
    )

age_band_parser_profile = (
    age_band_values
    .merge(
        pd.DataFrame(age_band_parser_rows),
        on="raw_age_band",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "age_band_syntax",
            "minimum_age",
            "maximum_age",
            "raw_age_band",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)


if age_band_parser_profile["age_band_syntax"].isna().any():
    raise AssertionError(
        "At least one age_band value was not assigned a parser state."
    )


# Validate exact reconstruction for every recognised populated format.
for row in age_band_parser_profile.itertuples(index=False):
    if row.age_band_syntax == "exact_age":
        reconstructed = f"{int(row.minimum_age)}yo"

    elif row.age_band_syntax == "minimum_age_open_ended":
        reconstructed = f"{int(row.minimum_age)}yo+"

    elif row.age_band_syntax == "closed_age_range":
        reconstructed = (
            f"{int(row.minimum_age)}-"
            f"{int(row.maximum_age)}yo"
        )

    else:
        continue

    if reconstructed != row.raw_age_band:
        raise AssertionError(
            "Age-band reconstruction failed: "
            f"{row.raw_age_band!r} != {reconstructed!r}"
        )


# Load only the runner fields required for consistency checks.
connection_uri = f"file:{SOURCE_DATABASE_PATH}?mode=ro"

with sqlite3.connect(connection_uri, uri=True) as connection:
    runner_characteristics = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            age,
            sex

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        """,
        connection,
    )


if len(runner_characteristics) != EXPECTED_RUNNER_ROWS:
    raise AssertionError(
        "Runner-characteristic population changed: "
        f"{len(runner_characteristics):,} != "
        f"{EXPECTED_RUNNER_ROWS:,}"
    )


# Retain one classification row per provisional race for the runner-level join.
race_eligibility = jurisdiction_frame[
    [
        "date",
        "course",
        "off",
        "jurisdiction",
        "type",
        "race_name",
        "age_band",
        "sex_rest",
    ]
].copy()


if race_eligibility.duplicated(
    subset=["date", "course", "off"]
).any():
    raise AssertionError(
        "Race eligibility frame is not unique by provisional race key."
    )


runner_eligibility = runner_characteristics.merge(
    race_eligibility,
    on=["date", "course", "off"],
    how="left",
    validate="many_to_one",
    indicator=True,
)


if len(runner_eligibility) != EXPECTED_RUNNER_ROWS:
    raise AssertionError(
        "Runner-to-race eligibility join changed runner grain: "
        f"{len(runner_eligibility):,} != "
        f"{EXPECTED_RUNNER_ROWS:,}"
    )


unmatched_runners = int(
    runner_eligibility["_merge"].ne("both").sum()
)

if unmatched_runners:
    raise AssertionError(
        f"{unmatched_runners:,} runner rows failed the eligibility join."
    )

runner_eligibility = runner_eligibility.drop(columns="_merge")


# Attach parsed age-band components to each runner.
age_band_lookup = age_band_parser_profile[
    [
        "raw_age_band",
        "age_band_state",
        "age_band_syntax",
        "minimum_age",
        "maximum_age",
        "open_ended",
    ]
].copy()

age_band_lookup["age_band"] = age_band_lookup[
    "raw_age_band"
].replace(
    {
        "<BLANK>": "",
        "<NULL>": pd.NA,
    }
)

age_band_lookup = age_band_lookup.drop(
    columns="raw_age_band"
)

runner_eligibility = runner_eligibility.merge(
    age_band_lookup,
    on="age_band",
    how="left",
    validate="many_to_one",
)


if runner_eligibility["age_band_syntax"].isna().any():
    raise AssertionError(
        "At least one runner race failed the age-band parser lookup."
    )


# Convert runner ages and parsed bounds to nullable numeric dtypes.
#
# This avoids ambiguous pd.NA comparisons while preserving missing values.
runner_eligibility["runner_age_numeric"] = pd.to_numeric(
    runner_eligibility["age"],
    errors="coerce",
).astype("Float64")

runner_eligibility["minimum_age_numeric"] = pd.to_numeric(
    runner_eligibility["minimum_age"],
    errors="coerce",
).astype("Float64")

runner_eligibility["maximum_age_numeric"] = pd.to_numeric(
    runner_eligibility["maximum_age"],
    errors="coerce",
).astype("Float64")


recognised_age_mask = runner_eligibility[
    "age_band_syntax"
].isin(
    [
        "exact_age",
        "minimum_age_open_ended",
        "closed_age_range",
    ]
)

age_testable_mask = (
    recognised_age_mask
    & runner_eligibility["runner_age_numeric"].notna()
    & runner_eligibility["minimum_age_numeric"].notna()
)


# Compare only rows with recognised syntax and usable numeric values.
runner_eligibility["below_minimum_age"] = (
    age_testable_mask
    & (
        runner_eligibility["runner_age_numeric"]
        < runner_eligibility["minimum_age_numeric"]
    ).fillna(False)
)

runner_eligibility["above_maximum_age"] = (
    age_testable_mask
    & runner_eligibility["maximum_age_numeric"].notna()
    & (
        runner_eligibility["runner_age_numeric"]
        > runner_eligibility["maximum_age_numeric"]
    ).fillna(False)
)

runner_eligibility["outside_age_band"] = (
    runner_eligibility["below_minimum_age"]
    | runner_eligibility["above_maximum_age"]
).fillna(False)


age_band_consistency = (
    runner_eligibility
    .groupby(
        [
            "age_band",
            "age_band_syntax",
            "minimum_age_numeric",
            "maximum_age_numeric",
            "open_ended",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        numeric_age_rows=("runner_age_numeric", "count"),
        below_minimum_rows=("below_minimum_age", "sum"),
        above_maximum_rows=("above_maximum_age", "sum"),
        outside_band_rows=("outside_age_band", "sum"),
        minimum_observed_runner_age=("runner_age_numeric", "min"),
        maximum_observed_runner_age=("runner_age_numeric", "max"),
    )
    .sort_values(
        [
            "outside_band_rows",
            "age_band",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


age_band_contradictions = (
    runner_eligibility.loc[
        runner_eligibility["outside_age_band"],
        [
            "source_rowid",
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "horse",
            "age",
            "age_band",
            "minimum_age_numeric",
            "maximum_age_numeric",
            "below_minimum_age",
            "above_maximum_age",
        ],
    ]
    .sort_values(
        [
            "date",
            "course",
            "off",
            "source_rowid",
        ]
    )
    .reset_index(drop=True)
)


# Provisional literal membership rules for populated sex-rest values.
SEX_REST_ALLOWED_CODES = {
    "F": {"F"},
    "M": {"M"},
    "F & M": {"F", "M"},
    "C & G": {"C", "G"},
    "C & F": {"C", "F"},
}


runner_eligibility["sex_rest_testable"] = (
    runner_eligibility["sex_rest"].isin(
        SEX_REST_ALLOWED_CODES
    )
    & runner_eligibility["sex"].notna()
    & runner_eligibility["sex"].astype("string").str.strip().ne("")
)


def runner_sex_matches_restriction(row):
    """
    Test literal source-code membership only.

    Blank restrictions and unknown sex codes remain untested rather than being
    classified as consistent or contradictory.
    """
    if not row["sex_rest_testable"]:
        return pd.NA

    allowed_codes = SEX_REST_ALLOWED_CODES[
        row["sex_rest"]
    ]

    return str(row["sex"]) in allowed_codes


runner_eligibility["sex_rest_match"] = (
    runner_eligibility.apply(
        runner_sex_matches_restriction,
        axis=1,
    )
    .astype("boolean")
)


sex_rest_consistency = (
    runner_eligibility
    .groupby(
        "sex_rest",
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        testable_runner_rows=("sex_rest_testable", "sum"),
        matching_runner_rows=(
            "sex_rest_match",
            lambda values: int(values.eq(True).sum()),
        ),
        contradictory_runner_rows=(
            "sex_rest_match",
            lambda values: int(values.eq(False).sum()),
        ),
        distinct_runner_sex_values=("sex", "nunique"),
    )
    .sort_values(
        [
            "contradictory_runner_rows",
            "runner_rows",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


sex_rest_contradictions = (
    runner_eligibility.loc[
        runner_eligibility["sex_rest_match"].eq(False),
        [
            "source_rowid",
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "horse",
            "sex",
            "sex_rest",
        ],
    ]
    .sort_values(
        [
            "date",
            "course",
            "off",
            "source_rowid",
        ]
    )
    .reset_index(drop=True)
)


print("Exact age-band values and parser states")
display(age_band_parser_profile)

print("Runner-age consistency by raw age band")
display(age_band_consistency)

print("Runner rows outside the parsed age band")
display(age_band_contradictions)

print("Runner-sex consistency by raw sex-rest value")
display(sex_rest_consistency)

print("Runner rows contradicting the provisional sex-rest mapping")
display(sex_rest_contradictions)

Exact age-band values and parser states


,raw_age_band,provisional_races,age_band_state,age_band_syntax,minimum_age,maximum_age,open_ended
0,<BLANK>,13,blank,blank,<NA>,<NA>,<NA>
1,2-3yo,105,populated,closed_age_range,2,3,False
2,3-4yo,567,populated,closed_age_range,3,4,False
3,3-5yo,1078,populated,closed_age_range,3,5,False
4,3-6yo,23,populated,closed_age_range,3,6,False
5,3-7yo,3,populated,closed_age_range,3,7,False
6,4-5yo,827,populated,closed_age_range,4,5,False
7,4-6yo,2634,populated,closed_age_range,4,6,False
8,4-7yo,1021,populated,closed_age_range,4,7,False
9,4-8yo,15,populated,closed_age_range,4,8,False


Runner-age consistency by raw age band


,age_band,age_band_syntax,minimum_age_numeric,maximum_age_numeric,open_ended,runner_rows,numeric_age_rows,below_minimum_rows,above_maximum_rows,outside_band_rows,minimum_observed_runner_age,maximum_observed_runner_age
0,3yo,exact_age,3.0,3.0,False,237914,237914,60,428,488,2.0,31.0
1,4yo,exact_age,4.0,4.0,False,44999,44999,9,188,197,3.0,12.0
2,2yo,exact_age,2.0,2.0,False,179439,179439,5,124,129,1.0,6.0
3,5yo,exact_age,5.0,5.0,False,3171,3171,2,99,101,4.0,11.0
4,4yo+,minimum_age_open_ended,4.0,<NA>,True,606576,606576,27,0,27,3.0,18.0
5,3yo+,minimum_age_open_ended,3.0,<NA>,True,556237,556237,14,0,14,2.0,16.0
6,3-4yo,closed_age_range,3.0,4.0,False,4646,4646,0,1,1,3.0,5.0
7,4-5yo,closed_age_range,4.0,5.0,False,8057,8057,1,0,1,3.0,5.0
8,,blank,<NA>,<NA>,NaN,159,159,0,0,0,3.0,8.0
9,10yo+,minimum_age_open_ended,10.0,<NA>,True,2061,2061,0,0,0,10.0,15.0


Runner rows outside the parsed age band


,source_rowid,date,course,jurisdiction,off,race_name,type,horse,age,age_band,minimum_age_numeric,maximum_age_numeric,below_minimum_age,above_maximum_age
0,5191,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,All Stormy (USA),6,4yo,4.0,4.0,False,True
1,5193,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,Villandry (USA),6,4yo,4.0,4.0,False,True
2,5194,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,Golden Soul (USA),5,4yo,4.0,4.0,False,True
3,5195,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,Infinite Magic (USA),5,4yo,4.0,4.0,False,True
4,5197,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,Gentlemans Kitten (USA),5,4yo,4.0,4.0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953,1842056,2026-05-09,Sha Tin,Hong Kong,05:30,Butterfly Bay Griffin Plate (Course C) (Turf),Flat,Ever Wealth (AUS),2,3yo+,3.0,<NA>,True,False
954,1842057,2026-05-09,Sha Tin,Hong Kong,05:30,Butterfly Bay Griffin Plate (Course C) (Turf),Flat,Gaudium Magnum (AUS),2,3yo+,3.0,<NA>,True,False
955,1842060,2026-05-09,Sha Tin,Hong Kong,05:30,Butterfly Bay Griffin Plate (Course C) (Turf),Flat,Talents Champion (AUS),2,3yo+,3.0,<NA>,True,False
956,1842062,2026-05-09,Sha Tin,Hong Kong,05:30,Butterfly Bay Griffin Plate (Course C) (Turf),Flat,Glorious Hero (AUS),2,3yo+,3.0,<NA>,True,False


Runner-sex consistency by raw sex-rest value


,sex_rest,runner_rows,testable_runner_rows,matching_runner_rows,contradictory_runner_rows,distinct_runner_sex_values
0,F,123667,123667,115436,8231,7
1,M,52478,52478,44543,7935,4
2,F & M,37354,37354,37318,36,5
3,C & G,25292,25292,25257,35,5
4,,1612425,0,0,0,7
5,C & F,69,69,69,0,1


Runner rows contradicting the provisional sex-rest mapping


,source_rowid,date,course,jurisdiction,off,race_name,type,horse,sex,sex_rest
0,996,2015-01-03,Newcastle,Great Britain,3:25,North Sea Logistics/EBF Stallions Sponsor Mare...,NH Flat,Still Acting (GB),F,M
1,1034,2015-01-03,Newcastle,Great Britain,3:25,North Sea Logistics/EBF Stallions Sponsor Mare...,NH Flat,Nowreyna (GB),F,M
2,1825,2015-01-05,Wolverhampton (AW),Great Britain,1:50,32Red Fillies Handicap (Tapeta),Flat,Oasis Spirit (GB),M,F
3,1827,2015-01-05,Wolverhampton (AW),Great Britain,1:50,32Red Fillies Handicap (Tapeta),Flat,Mayfield Girl (IRE),M,F
4,1832,2015-01-05,Wolverhampton (AW),Great Britain,1:50,32Red Fillies Handicap (Tapeta),Flat,Medam (GB),M,F
...,...,...,...,...,...,...,...,...,...,...
16232,1851196,2026-05-27,Newton Abbot,Great Britain,14:53,Stock Exe Building Supplies Mares National Hun...,Hurdle,Minnie Belle (GB),F,M
16233,1851197,2026-05-27,Newton Abbot,Great Britain,14:53,Stock Exe Building Supplies Mares National Hun...,Hurdle,Whodunit (IRE),F,M
16234,1851198,2026-05-27,Newton Abbot,Great Britain,14:53,Stock Exe Building Supplies Mares National Hun...,Hurdle,Frenati (GB),F,M
16235,1851199,2026-05-27,Newton Abbot,Great Britain,14:53,Stock Exe Building Supplies Mares National Hun...,Hurdle,Amhranai (IRE),F,M


In [7]:
# Input grain:
#   Runner-level `runner_eligibility` rows produced by the preceding cell.
#
# Output grain:
#   1. One row per jurisdiction and raw age_band summarising age contradictions.
#   2. One row per provisional race containing at least one age contradiction.
#   3. One row per raw runner-sex code, with counts by raw sex_rest value.
#   4. One row per jurisdiction, runner-sex code and raw sex_rest value.
#
# Purpose:
#   Locate the 958 apparent age-band contradictions and establish the actual
#   relationship between race-level sex restrictions and runner-level sex codes.
#
#   This stage deliberately replaces the invalid literal sex-code comparison
#   with descriptive cross-tabulation. It does not yet define a semantic parser
#   for sex restrictions.
#
# Raw versus derived values:
#   Raw `age`, `age_band`, `sex` and `sex_rest` values remain unchanged.
#
#   Derived values are limited to:
#   - counts of runners outside parsed age limits;
#   - counts and percentages by jurisdiction and race;
#   - exact cross-tabulations of raw runner-sex and sex-rest values;
#   - race-name indicators used only to expose words such as Fillies, Mares,
#     Colts and Geldings for later interpretation.
#
# Assumptions deliberately not made:
#   - runner `sex = F` is not assumed to mean filly rather than mare;
#   - runner `sex = M` is not assumed to mean mare or male without context;
#   - sex_rest abbreviations are not compared by literal equality;
#   - race-name words are not treated as authoritative structured fields;
#   - age contradictions are not automatically corrected or discarded;
#   - international age conventions are not treated as globally equivalent.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - `runner_eligibility` is unavailable;
#   - the age-contradiction count differs from the established 958 rows;
#   - grouped contradiction counts do not reconcile to 958;
#   - raw sex-rest cross-tabulations do not reconstruct the runner population.
#
#   No source data is written or modified.

if "runner_eligibility" not in globals():
    raise RuntimeError(
        "Run the preceding runner eligibility cell first."
    )


EXPECTED_AGE_CONTRADICTION_ROWS = 958


# Preserve only the runner rows that failed the literal parsed age bounds.
age_contradiction_frame = runner_eligibility.loc[
    runner_eligibility["outside_age_band"]
].copy()


if len(age_contradiction_frame) != EXPECTED_AGE_CONTRADICTION_ROWS:
    raise AssertionError(
        "Age contradiction population changed: "
        f"{len(age_contradiction_frame):,} != "
        f"{EXPECTED_AGE_CONTRADICTION_ROWS:,}"
    )


# Summarise contradiction concentration by jurisdiction and exact raw age band.
age_contradictions_by_jurisdiction = (
    runner_eligibility
    .groupby(
        ["jurisdiction", "type", "age_band"],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        outside_band_rows=("outside_age_band", "sum"),
        distinct_races=(
            "race_name",
            lambda values: values.nunique(),
        ),
        minimum_observed_age=("runner_age_numeric", "min"),
        maximum_observed_age=("runner_age_numeric", "max"),
    )
)

age_contradictions_by_jurisdiction = (
    age_contradictions_by_jurisdiction.loc[
        age_contradictions_by_jurisdiction[
            "outside_band_rows"
        ].gt(0)
    ]
    .copy()
)

age_contradictions_by_jurisdiction[
    "outside_band_runner_pct"
] = (
    age_contradictions_by_jurisdiction["outside_band_rows"]
    / age_contradictions_by_jurisdiction["runner_rows"]
    * 100
).round(4)

age_contradictions_by_jurisdiction = (
    age_contradictions_by_jurisdiction
    .sort_values(
        [
            "outside_band_rows",
            "jurisdiction",
            "type",
            "age_band",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)


if int(
    age_contradictions_by_jurisdiction[
        "outside_band_rows"
    ].sum()
) != EXPECTED_AGE_CONTRADICTION_ROWS:
    raise AssertionError(
        "Jurisdiction age summaries do not reconcile to 958 rows."
    )


# Summarise at provisional-race grain so clusters caused by one problematic race
# or feed can be distinguished from isolated runner-level anomalies.
age_contradictions_by_race = (
    runner_eligibility
    .groupby(
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "age_band",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        outside_band_rows=("outside_age_band", "sum"),
        minimum_observed_age=("runner_age_numeric", "min"),
        maximum_observed_age=("runner_age_numeric", "max"),
    )
)

age_contradictions_by_race = (
    age_contradictions_by_race.loc[
        age_contradictions_by_race[
            "outside_band_rows"
        ].gt(0)
    ]
    .copy()
)

age_contradictions_by_race[
    "outside_band_runner_pct"
] = (
    age_contradictions_by_race["outside_band_rows"]
    / age_contradictions_by_race["runner_rows"]
    * 100
).round(3)


# Add descriptive race-name indicators only to help inspect likely semantics.
race_name_upper = age_contradictions_by_race[
    "race_name"
].astype("string").str.upper()

for word in [
    "FILLIES",
    "MARES",
    "COLTS",
    "GELDINGS",
    "JUVENILE",
    "GRIFFIN",
]:
    age_contradictions_by_race[
        f"race_name_mentions_{word.lower()}"
    ] = race_name_upper.str.contains(
        rf"\b{word}\b",
        regex=True,
        na=False,
    )


age_contradictions_by_race = (
    age_contradictions_by_race
    .sort_values(
        [
            "outside_band_rows",
            "date",
            "course",
            "off",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)


# Replace the invalid literal membership test with an exact source cross-tab.
sex_rest_runner_sex = (
    runner_eligibility
    .assign(
        sex_rest_display=runner_eligibility[
            "sex_rest"
        ].map(display_raw_state),
        runner_sex_display=runner_eligibility[
            "sex"
        ].map(display_raw_state),
    )
    .groupby(
        [
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.nunique(),
        ),
    )
    .sort_values(
        [
            "sex_rest_display",
            "runner_rows",
            "runner_sex_display",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


if int(sex_rest_runner_sex["runner_rows"].sum()) != EXPECTED_RUNNER_ROWS:
    raise AssertionError(
        "Sex-rest cross-tab does not reconstruct the runner population."
    )


# Add jurisdiction to determine whether the runner sex coding itself varies
# between source feeds.
sex_rest_runner_sex_by_jurisdiction = (
    runner_eligibility
    .assign(
        sex_rest_display=runner_eligibility[
            "sex_rest"
        ].map(display_raw_state),
        runner_sex_display=runner_eligibility[
            "sex"
        ].map(display_raw_state),
    )
    .groupby(
        [
            "jurisdiction",
            "type",
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.nunique(),
        ),
    )
    .sort_values(
        [
            "jurisdiction",
            "type",
            "sex_rest_display",
            "runner_rows",
            "runner_sex_display",
        ],
        ascending=[True, True, True, False, True],
    )
    .reset_index(drop=True)
)


print("Age contradictions by jurisdiction, race type and raw age band")
display(age_contradictions_by_jurisdiction)

print("Provisional races containing apparent age-band contradictions")
display(age_contradictions_by_race)

print("Exact runner-sex values by raw sex-rest value")
display(sex_rest_runner_sex)

print("Runner-sex values by jurisdiction, race type and sex-rest value")
display(sex_rest_runner_sex_by_jurisdiction)

Age contradictions by jurisdiction, race type and raw age band


,jurisdiction,type,age_band,runner_rows,outside_band_rows,distinct_races,minimum_observed_age,maximum_observed_age,outside_band_runner_pct
0,Hong Kong,Flat,3yo,437,138,34,3.0,8.0,31.5789
1,France,Flat,4yo,13638,121,680,4.0,10.0,0.8872
2,South Korea,Flat,3yo,75,69,4,3.0,9.0,92.0
3,Peru,Flat,2yo,64,64,3,3.0,6.0,100.0
4,France,Chase,5yo,1400,50,64,5.0,11.0,3.5714
5,Peru,Flat,3yo,552,46,23,2.0,8.0,8.3333
6,United States,Flat,3yo,14837,41,774,3.0,8.0,0.2763
7,France,Hurdle,5yo,1586,40,47,5.0,11.0,2.5221
8,Argentina,Flat,3yo,1083,34,39,2.0,5.0,3.1394
9,South Africa,Flat,2yo,798,29,27,2.0,3.0,3.6341


Provisional races containing apparent age-band contradictions


,date,course,jurisdiction,off,race_name,type,age_band,runner_rows,outside_band_rows,minimum_observed_age,maximum_observed_age,outside_band_runner_pct,race_name_mentions_fillies,race_name_mentions_mares,race_name_mentions_colts,race_name_mentions_geldings,race_name_mentions_juvenile,race_name_mentions_griffin
0,2015-08-01,Greyville (SAF),South Africa,2:30,Premiers Champion Stakes (2yo) (Turf),Flat,2yo,16,16,3.0,3.0,100.0,False,False,False,False,False,False
1,2024-06-23,Monterrico (PER),Peru,10:45,Premio Pamplona (2yo Fillies & Mares) (Turf),Flat,2yo,16,16,3.0,5.0,100.0,True,True,False,False,False,False
2,2015-10-25,Saint-Cloud (FR),France,4:15,Prix des Bords de Seine (Handicap) (4yo ) (Turf),Flat,4yo,17,15,4.0,9.0,88.235,False,False,False,False,False,False
3,2015-07-04,Hipodromo Chile (CHI),Chile,9:52,Premio Tanteo de Potrillos (2yo Colts) (Dirt),Flat,2yo,14,14,3.0,3.0,100.0,False,False,True,False,False,False
4,2015-10-25,Saint-Cloud (FR),France,3:10,Finale du Galop Tour Inter-Regional - GTI (Han...,Flat,4yo,15,14,4.0,8.0,93.333,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,2025-06-21,Ascot,Great Britain,3:40,Queen Elizabeth II Jubilee Stakes,Flat,4yo+,14,1,3.0,7.0,7.143,False,False,False,False,False,False
179,2025-06-27,Newmarket (July),Great Britain,6:10,Boodles Fillies Novice Stakes (GBB Race),Flat,3yo+,6,1,2.0,3.0,16.667,True,False,False,False,False,False
180,2025-12-21,Monterrico,Peru,22:50,Gran Premio Nacional Augusta B Leguia (3yo) (T...,Flat,3yo,10,1,2.0,3.0,10.0,False,False,False,False,False,False
181,2026-02-07,Caulfield,Australia,05:15,Evergreen Turf Peter Le Grand Stakes (Fillies)...,Flat,3yo,12,1,3.0,4.0,8.333,True,False,False,False,False,False


Exact runner-sex values by raw sex-rest value


,sex_rest_display,runner_sex_display,runner_rows,provisional_races
0,<BLANK>,G,1067054,89702
1,<BLANK>,F,224783,45536
2,<BLANK>,C,162181,32277
3,<BLANK>,M,126844,40536
4,<BLANK>,H,30695,9303
5,<BLANK>,R,867,694
6,<BLANK>,BB,1,1
7,C & F,C,69,5
8,C & G,C,14130,1314
9,C & G,G,11127,1275


Runner-sex values by jurisdiction, race type and sex-rest value


,jurisdiction,type,sex_rest_display,runner_sex_display,runner_rows,provisional_races
0,Argentina,Flat,<BLANK>,C,2045,126
1,Argentina,Flat,<BLANK>,H,870,99
2,Argentina,Flat,<BLANK>,F,293,62
3,Argentina,Flat,<BLANK>,M,104,38
4,Argentina,Flat,<BLANK>,G,21,14
...,...,...,...,...,...,...
464,Uruguay,Flat,<BLANK>,G,1,1
465,Uruguay,Flat,C & G,C,86,5
466,Uruguay,Flat,F,F,166,4
467,Uruguay,Flat,F & M,F,103,5


In [8]:
# Input grain:
#   Runner-level `runner_eligibility` rows produced by the age and sex-rest
#   consistency stage.
#
# Output grain:
#   1. One row per age-band contradiction direction and signed age difference.
#   2. One row per jurisdiction, calendar month and signed age difference.
#   3. One row per contradictory provisional race with its dominant difference.
#   4. One detailed row for contradictions whose difference is greater than
#      one year in either direction.
#
# Purpose:
#   Distinguish likely calendar or jurisdictional age-convention differences
#   from isolated source defects.
#
#   A systematic difference of exactly one year across every runner in a race
#   may indicate that the race eligibility age and stored runner age use
#   different birthday conventions. Larger or internally mixed differences
#   require separate review.
#
# Raw versus derived values:
#   Raw runner age and raw age_band remain unchanged.
#
#   Derived values are limited to:
#   - the signed distance from the relevant permitted age boundary;
#   - calendar month extracted from the raw race date;
#   - per-race counts and dominant difference values;
#   - flags distinguishing uniform and mixed contradiction patterns.
#
# Assumptions deliberately not made:
#   - a one-year difference is not automatically corrected;
#   - Southern Hemisphere or jurisdictional birthday rules are not assigned
#     without external verification;
#   - the race date is not used to recompute horse age;
#   - large differences are not automatically treated as horse-age errors;
#   - an internally uniform race is not automatically treated as valid.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - `runner_eligibility` is unavailable;
#   - the contradiction population differs from 958 rows;
#   - every contradiction is not assigned exactly one signed difference;
#   - grouped outputs fail to reconcile to 958 rows.
#
#   No source values are written or modified.

if "runner_eligibility" not in globals():
    raise RuntimeError(
        "Run the preceding age and sex-rest consistency stages first."
    )


EXPECTED_AGE_CONTRADICTIONS = 958


# Work only with rows already proven to lie outside the literal parsed bounds.
age_difference_frame = runner_eligibility.loc[
    runner_eligibility["outside_age_band"]
].copy()


if len(age_difference_frame) != EXPECTED_AGE_CONTRADICTIONS:
    raise AssertionError(
        "Age contradiction population changed: "
        f"{len(age_difference_frame):,} != "
        f"{EXPECTED_AGE_CONTRADICTIONS:,}"
    )


# Calculate signed distance from the violated eligibility boundary.
#
# Negative values mean the runner is younger than the stated minimum.
# Positive values mean the runner is older than the stated maximum.
age_difference_frame["age_difference_from_band"] = pd.NA

below_mask = age_difference_frame["below_minimum_age"]

age_difference_frame.loc[
    below_mask,
    "age_difference_from_band",
] = (
    age_difference_frame.loc[
        below_mask,
        "runner_age_numeric",
    ]
    - age_difference_frame.loc[
        below_mask,
        "minimum_age_numeric",
    ]
)


above_mask = age_difference_frame["above_maximum_age"]

age_difference_frame.loc[
    above_mask,
    "age_difference_from_band",
] = (
    age_difference_frame.loc[
        above_mask,
        "runner_age_numeric",
    ]
    - age_difference_frame.loc[
        above_mask,
        "maximum_age_numeric",
    ]
)


age_difference_frame["age_difference_from_band"] = pd.to_numeric(
    age_difference_frame["age_difference_from_band"],
    errors="raise",
).astype("Int64")


# Every contradiction must violate exactly one side of the band.
invalid_direction_rows = age_difference_frame.loc[
    (
        age_difference_frame["below_minimum_age"]
        == age_difference_frame["above_maximum_age"]
    )
]

if not invalid_direction_rows.empty:
    raise AssertionError(
        "At least one age contradiction does not violate exactly one boundary."
    )


if age_difference_frame[
    "age_difference_from_band"
].isna().any():
    raise AssertionError(
        "At least one age contradiction has no signed age difference."
    )


# Derive date components for temporal concentration analysis.
age_difference_frame["parsed_race_date"] = pd.to_datetime(
    age_difference_frame["date"],
    errors="raise",
)

age_difference_frame["race_year"] = (
    age_difference_frame["parsed_race_date"].dt.year
)

age_difference_frame["race_month"] = (
    age_difference_frame["parsed_race_date"].dt.month
)


# Global distribution of signed differences.
age_difference_distribution = (
    age_difference_frame
    .assign(
        contradiction_direction=age_difference_frame[
            "below_minimum_age"
        ].map(
            {
                True: "below_minimum",
                False: "above_maximum",
            }
        )
    )
    .groupby(
        [
            "contradiction_direction",
            "age_difference_from_band",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.nunique(),
        ),
        jurisdictions=("jurisdiction", "nunique"),
    )
    .sort_values(
        [
            "runner_rows",
            "contradiction_direction",
            "age_difference_from_band",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)


if int(
    age_difference_distribution["runner_rows"].sum()
) != EXPECTED_AGE_CONTRADICTIONS:
    raise AssertionError(
        "Age-difference distribution does not reconcile to 958 rows."
    )


# Test month concentration by jurisdiction and exact signed difference.
age_difference_by_month = (
    age_difference_frame
    .groupby(
        [
            "jurisdiction",
            "type",
            "race_month",
            "age_difference_from_band",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.nunique(),
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "runner_rows",
            "jurisdiction",
            "race_month",
            "age_difference_from_band",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)


if int(
    age_difference_by_month["runner_rows"].sum()
) != EXPECTED_AGE_CONTRADICTIONS:
    raise AssertionError(
        "Monthly age-difference summary does not reconcile to 958 rows."
    )


# Summarise whether each contradictory race has one uniform difference or a
# mixture of differences.
age_difference_by_race = (
    age_difference_frame
    .groupby(
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "age_band",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_age_differences=(
            "age_difference_from_band",
            "nunique",
        ),
        minimum_age_difference=(
            "age_difference_from_band",
            "min",
        ),
        maximum_age_difference=(
            "age_difference_from_band",
            "max",
        ),
        most_common_age_difference=(
            "age_difference_from_band",
            lambda values: values.value_counts().index[0],
        ),
        most_common_difference_rows=(
            "age_difference_from_band",
            lambda values: int(values.value_counts().iloc[0]),
        ),
        minimum_observed_runner_age=(
            "runner_age_numeric",
            "min",
        ),
        maximum_observed_runner_age=(
            "runner_age_numeric",
            "max",
        ),
    )
)


age_difference_by_race["uniform_difference"] = (
    age_difference_by_race[
        "distinct_age_differences"
    ].eq(1)
)

age_difference_by_race["dominant_difference_pct"] = (
    age_difference_by_race["most_common_difference_rows"]
    / age_difference_by_race["runner_rows"]
    * 100
).round(3)


age_difference_by_race = (
    age_difference_by_race
    .sort_values(
        [
            "runner_rows",
            "distinct_age_differences",
            "date",
            "course",
            "off",
        ],
        ascending=[False, False, True, True, True],
    )
    .reset_index(drop=True)
)


if int(age_difference_by_race["runner_rows"].sum()) != (
    EXPECTED_AGE_CONTRADICTIONS
):
    raise AssertionError(
        "Race-level age-difference summary does not reconcile to 958 rows."
    )


# Preserve the larger differences as a bounded high-priority review residue.
large_age_difference_rows = (
    age_difference_frame.loc[
        age_difference_frame[
            "age_difference_from_band"
        ].abs().gt(1),
        [
            "source_rowid",
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "horse",
            "age",
            "age_band",
            "minimum_age_numeric",
            "maximum_age_numeric",
            "age_difference_from_band",
        ],
    ]
    .sort_values(
        [
            "age_difference_from_band",
            "date",
            "course",
            "off",
            "source_rowid",
        ]
    )
    .reset_index(drop=True)
)


print("Signed distance from the parsed age band")
display(age_difference_distribution)

print("Age differences by jurisdiction and calendar month")
display(age_difference_by_month)

print("Contradictory races and their within-race difference patterns")
display(age_difference_by_race)

print("Runner contradictions more than one year outside the band")
display(large_age_difference_rows)

Signed distance from the parsed age band


,contradiction_direction,age_difference_from_band,runner_rows,provisional_races,jurisdictions
0,above_maximum,1,347,94,23
1,above_maximum,2,225,70,17
2,above_maximum,3,156,67,16
3,below_minimum,-1,118,67,12
4,above_maximum,4,68,45,13
5,above_maximum,5,31,25,11
6,above_maximum,6,8,7,3
7,above_maximum,7,3,3,3
8,above_maximum,8,1,1,1
9,above_maximum,28,1,1,1


Age differences by jurisdiction and calendar month


,jurisdiction,type,race_month,age_difference_from_band,runner_rows,provisional_races,first_date,last_date
0,Peru,Flat,6,1,49,5,2021-06-26,2025-06-22
1,France,Flat,10,1,28,6,2015-10-25,2016-10-18
2,South Africa,Flat,8,1,28,2,2015-08-01,2015-08-01
3,Peru,Flat,6,2,22,5,2021-06-26,2025-06-22
4,Argentina,Flat,6,-1,21,2,2018-06-30,2018-06-30
...,...,...,...,...,...,...,...,...
248,United States,Flat,8,-1,1,1,2015-08-08,2015-08-08
249,United States,Flat,8,4,1,1,2016-08-13,2016-08-13
250,United States,Hurdle,10,1,1,1,2024-10-27,2024-10-27
251,United States,Flat,11,1,1,1,2022-11-05,2022-11-05


Contradictory races and their within-race difference patterns


,date,course,jurisdiction,off,race_name,type,age_band,runner_rows,distinct_age_differences,minimum_age_difference,maximum_age_difference,most_common_age_difference,most_common_difference_rows,minimum_observed_runner_age,maximum_observed_runner_age,uniform_difference,dominant_difference_pct
0,2024-06-23,Monterrico (PER),Peru,10:45,Premio Pamplona (2yo Fillies & Mares) (Turf),Flat,2yo,16,3,1,3,1,11,3.0,5.0,False,68.75
1,2015-08-01,Greyville (SAF),South Africa,2:30,Premiers Champion Stakes (2yo) (Turf),Flat,2yo,16,1,1,1,1,16,3.0,3.0,True,100.0
2,2015-10-25,Saint-Cloud (FR),France,4:15,Prix des Bords de Seine (Handicap) (4yo ) (Turf),Flat,4yo,15,3,1,5,2,7,5.0,9.0,False,46.667
3,2024-09-08,Seoul (KOR),South Korea,7:20,Korea Sprint (3yo ) (Dirt),Flat,3yo,14,6,1,6,1,4,4.0,9.0,False,28.571
4,2017-05-31,Sha Tin (HK),Hong Kong,12:45,Silvermine Bay Handicap (3yo ) (All Weather T...,Flat,3yo,14,5,1,5,1,5,4.0,8.0,False,35.714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,2025-06-21,Ascot,Great Britain,3:40,Queen Elizabeth II Jubilee Stakes,Flat,4yo+,1,1,-1,-1,-1,1,3.0,3.0,True,100.0
179,2025-06-27,Newmarket (July),Great Britain,6:10,Boodles Fillies Novice Stakes (GBB Race),Flat,3yo+,1,1,-1,-1,-1,1,2.0,2.0,True,100.0
180,2025-12-21,Monterrico,Peru,22:50,Gran Premio Nacional Augusta B Leguia (3yo) (T...,Flat,3yo,1,1,-1,-1,-1,1,2.0,2.0,True,100.0
181,2026-02-07,Caulfield,Australia,05:15,Evergreen Turf Peter Le Grand Stakes (Fillies)...,Flat,3yo,1,1,1,1,1,1,4.0,4.0,True,100.0


Runner contradictions more than one year outside the band


,source_rowid,date,course,jurisdiction,off,race_name,type,horse,age,age_band,minimum_age_numeric,maximum_age_numeric,age_difference_from_band
0,5191,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,All Stormy (USA),6,4yo,4.0,4.0,2
1,5193,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,Villandry (USA),6,4yo,4.0,4.0,2
2,60016,2015-05-25,Prairie Meadows (USA),United States,10:10,Jim Rasmussen Memorial Stakes (Dirt),Flat,Dens Legacy (USA),5,3yo,3.0,3.0,2
3,60020,2015-05-25,Prairie Meadows (USA),United States,10:10,Jim Rasmussen Memorial Stakes (Dirt),Flat,Cougar Ridge (USA),5,3yo,3.0,3.0,2
4,60023,2015-05-25,Prairie Meadows (USA),United States,10:10,Jim Rasmussen Memorial Stakes (Dirt),Flat,Lahshad (USA),5,3yo,3.0,3.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,207820,2016-05-14,Percy Warner Park (USA),United States,10:30,Calvin Houghland Iroquois (Hurdle) (4yo ) (Turf),Hurdle,Italian Wedding (USA),11,4yo,4.0,4.0,7
489,1461352,2024-02-04,St Moritz (SWI),Switzerland,1:45,Preis DC Aviation (Conditions) (4yo) (Snow),Flat,Fleur DIpanema (FR),11,4yo,4.0,4.0,7
490,1780730,2025-12-18,Bahrain,Bahrain,15:30,Dallah Cup (Handicap) (3yo ) (Outer Track) (Turf),Flat,Nine Below Zero (GB),10,3yo,3.0,3.0,7
491,207819,2016-05-14,Percy Warner Park (USA),United States,10:30,Calvin Houghland Iroquois (Hurdle) (4yo ) (Turf),Hurdle,Pierrot Lunaire (USA),12,4yo,4.0,4.0,8


In [9]:
# Input grain:
#   Runner-level `runner_eligibility` rows produced by the age and sex-rest
#   consistency stage.
#
# Output grain:
#   1. One row per age-band contradiction direction and signed age difference.
#   2. One row per jurisdiction, calendar month and signed age difference.
#   3. One row per contradictory provisional race with its dominant difference.
#   4. One detailed row for contradictions whose difference is greater than
#      one year in either direction.
#
# Purpose:
#   Distinguish likely calendar or jurisdictional age-convention differences
#   from isolated source defects.
#
#   A systematic difference of exactly one year across every runner in a race
#   may indicate that the race eligibility age and stored runner age use
#   different birthday conventions. Larger or internally mixed differences
#   require separate review.
#
# Raw versus derived values:
#   Raw runner age and raw age_band remain unchanged.
#
#   Derived values are limited to:
#   - the signed distance from the relevant permitted age boundary;
#   - calendar month extracted from the raw race date;
#   - per-race counts and dominant difference values;
#   - flags distinguishing uniform and mixed contradiction patterns.
#
# Assumptions deliberately not made:
#   - a one-year difference is not automatically corrected;
#   - Southern Hemisphere or jurisdictional birthday rules are not assigned
#     without external verification;
#   - the race date is not used to recompute horse age;
#   - large differences are not automatically treated as horse-age errors;
#   - an internally uniform race is not automatically treated as valid.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - `runner_eligibility` is unavailable;
#   - the contradiction population differs from 958 rows;
#   - every contradiction is not assigned exactly one signed difference;
#   - grouped outputs fail to reconcile to 958 rows.
#
#   No source values are written or modified.

if "runner_eligibility" not in globals():
    raise RuntimeError(
        "Run the preceding age and sex-rest consistency stages first."
    )


EXPECTED_AGE_CONTRADICTIONS = 958


# Work only with rows already proven to lie outside the literal parsed bounds.
age_difference_frame = runner_eligibility.loc[
    runner_eligibility["outside_age_band"]
].copy()


if len(age_difference_frame) != EXPECTED_AGE_CONTRADICTIONS:
    raise AssertionError(
        "Age contradiction population changed: "
        f"{len(age_difference_frame):,} != "
        f"{EXPECTED_AGE_CONTRADICTIONS:,}"
    )


# Calculate signed distance from the violated eligibility boundary.
#
# Negative values mean the runner is younger than the stated minimum.
# Positive values mean the runner is older than the stated maximum.
age_difference_frame["age_difference_from_band"] = pd.NA

below_mask = age_difference_frame["below_minimum_age"]

age_difference_frame.loc[
    below_mask,
    "age_difference_from_band",
] = (
    age_difference_frame.loc[
        below_mask,
        "runner_age_numeric",
    ]
    - age_difference_frame.loc[
        below_mask,
        "minimum_age_numeric",
    ]
)


above_mask = age_difference_frame["above_maximum_age"]

age_difference_frame.loc[
    above_mask,
    "age_difference_from_band",
] = (
    age_difference_frame.loc[
        above_mask,
        "runner_age_numeric",
    ]
    - age_difference_frame.loc[
        above_mask,
        "maximum_age_numeric",
    ]
)


age_difference_frame["age_difference_from_band"] = pd.to_numeric(
    age_difference_frame["age_difference_from_band"],
    errors="raise",
).astype("Int64")


# Every contradiction must violate exactly one side of the band.
invalid_direction_rows = age_difference_frame.loc[
    (
        age_difference_frame["below_minimum_age"]
        == age_difference_frame["above_maximum_age"]
    )
]

if not invalid_direction_rows.empty:
    raise AssertionError(
        "At least one age contradiction does not violate exactly one boundary."
    )


if age_difference_frame[
    "age_difference_from_band"
].isna().any():
    raise AssertionError(
        "At least one age contradiction has no signed age difference."
    )


# Derive date components for temporal concentration analysis.
age_difference_frame["parsed_race_date"] = pd.to_datetime(
    age_difference_frame["date"],
    errors="raise",
)

age_difference_frame["race_year"] = (
    age_difference_frame["parsed_race_date"].dt.year
)

age_difference_frame["race_month"] = (
    age_difference_frame["parsed_race_date"].dt.month
)


# Global distribution of signed differences.
age_difference_distribution = (
    age_difference_frame
    .assign(
        contradiction_direction=age_difference_frame[
            "below_minimum_age"
        ].map(
            {
                True: "below_minimum",
                False: "above_maximum",
            }
        )
    )
    .groupby(
        [
            "contradiction_direction",
            "age_difference_from_band",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.nunique(),
        ),
        jurisdictions=("jurisdiction", "nunique"),
    )
    .sort_values(
        [
            "runner_rows",
            "contradiction_direction",
            "age_difference_from_band",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)


if int(
    age_difference_distribution["runner_rows"].sum()
) != EXPECTED_AGE_CONTRADICTIONS:
    raise AssertionError(
        "Age-difference distribution does not reconcile to 958 rows."
    )


# Test month concentration by jurisdiction and exact signed difference.
age_difference_by_month = (
    age_difference_frame
    .groupby(
        [
            "jurisdiction",
            "type",
            "race_month",
            "age_difference_from_band",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.nunique(),
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "runner_rows",
            "jurisdiction",
            "race_month",
            "age_difference_from_band",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)


if int(
    age_difference_by_month["runner_rows"].sum()
) != EXPECTED_AGE_CONTRADICTIONS:
    raise AssertionError(
        "Monthly age-difference summary does not reconcile to 958 rows."
    )


# Summarise whether each contradictory race has one uniform difference or a
# mixture of differences.
age_difference_by_race = (
    age_difference_frame
    .groupby(
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "age_band",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_age_differences=(
            "age_difference_from_band",
            "nunique",
        ),
        minimum_age_difference=(
            "age_difference_from_band",
            "min",
        ),
        maximum_age_difference=(
            "age_difference_from_band",
            "max",
        ),
        most_common_age_difference=(
            "age_difference_from_band",
            lambda values: values.value_counts().index[0],
        ),
        most_common_difference_rows=(
            "age_difference_from_band",
            lambda values: int(values.value_counts().iloc[0]),
        ),
        minimum_observed_runner_age=(
            "runner_age_numeric",
            "min",
        ),
        maximum_observed_runner_age=(
            "runner_age_numeric",
            "max",
        ),
    )
)


age_difference_by_race["uniform_difference"] = (
    age_difference_by_race[
        "distinct_age_differences"
    ].eq(1)
)

age_difference_by_race["dominant_difference_pct"] = (
    age_difference_by_race["most_common_difference_rows"]
    / age_difference_by_race["runner_rows"]
    * 100
).round(3)


age_difference_by_race = (
    age_difference_by_race
    .sort_values(
        [
            "runner_rows",
            "distinct_age_differences",
            "date",
            "course",
            "off",
        ],
        ascending=[False, False, True, True, True],
    )
    .reset_index(drop=True)
)


if int(age_difference_by_race["runner_rows"].sum()) != (
    EXPECTED_AGE_CONTRADICTIONS
):
    raise AssertionError(
        "Race-level age-difference summary does not reconcile to 958 rows."
    )


# Preserve the larger differences as a bounded high-priority review residue.
large_age_difference_rows = (
    age_difference_frame.loc[
        age_difference_frame[
            "age_difference_from_band"
        ].abs().gt(1),
        [
            "source_rowid",
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "horse",
            "age",
            "age_band",
            "minimum_age_numeric",
            "maximum_age_numeric",
            "age_difference_from_band",
        ],
    ]
    .sort_values(
        [
            "age_difference_from_band",
            "date",
            "course",
            "off",
            "source_rowid",
        ]
    )
    .reset_index(drop=True)
)


print("Signed distance from the parsed age band")
display(age_difference_distribution)

print("Age differences by jurisdiction and calendar month")
display(age_difference_by_month)

print("Contradictory races and their within-race difference patterns")
display(age_difference_by_race)

print("Runner contradictions more than one year outside the band")
display(large_age_difference_rows)

Signed distance from the parsed age band


,contradiction_direction,age_difference_from_band,runner_rows,provisional_races,jurisdictions
0,above_maximum,1,347,94,23
1,above_maximum,2,225,70,17
2,above_maximum,3,156,67,16
3,below_minimum,-1,118,67,12
4,above_maximum,4,68,45,13
5,above_maximum,5,31,25,11
6,above_maximum,6,8,7,3
7,above_maximum,7,3,3,3
8,above_maximum,8,1,1,1
9,above_maximum,28,1,1,1


Age differences by jurisdiction and calendar month


,jurisdiction,type,race_month,age_difference_from_band,runner_rows,provisional_races,first_date,last_date
0,Peru,Flat,6,1,49,5,2021-06-26,2025-06-22
1,France,Flat,10,1,28,6,2015-10-25,2016-10-18
2,South Africa,Flat,8,1,28,2,2015-08-01,2015-08-01
3,Peru,Flat,6,2,22,5,2021-06-26,2025-06-22
4,Argentina,Flat,6,-1,21,2,2018-06-30,2018-06-30
...,...,...,...,...,...,...,...,...
248,United States,Flat,8,-1,1,1,2015-08-08,2015-08-08
249,United States,Flat,8,4,1,1,2016-08-13,2016-08-13
250,United States,Hurdle,10,1,1,1,2024-10-27,2024-10-27
251,United States,Flat,11,1,1,1,2022-11-05,2022-11-05


Contradictory races and their within-race difference patterns


,date,course,jurisdiction,off,race_name,type,age_band,runner_rows,distinct_age_differences,minimum_age_difference,maximum_age_difference,most_common_age_difference,most_common_difference_rows,minimum_observed_runner_age,maximum_observed_runner_age,uniform_difference,dominant_difference_pct
0,2024-06-23,Monterrico (PER),Peru,10:45,Premio Pamplona (2yo Fillies & Mares) (Turf),Flat,2yo,16,3,1,3,1,11,3.0,5.0,False,68.75
1,2015-08-01,Greyville (SAF),South Africa,2:30,Premiers Champion Stakes (2yo) (Turf),Flat,2yo,16,1,1,1,1,16,3.0,3.0,True,100.0
2,2015-10-25,Saint-Cloud (FR),France,4:15,Prix des Bords de Seine (Handicap) (4yo ) (Turf),Flat,4yo,15,3,1,5,2,7,5.0,9.0,False,46.667
3,2024-09-08,Seoul (KOR),South Korea,7:20,Korea Sprint (3yo ) (Dirt),Flat,3yo,14,6,1,6,1,4,4.0,9.0,False,28.571
4,2017-05-31,Sha Tin (HK),Hong Kong,12:45,Silvermine Bay Handicap (3yo ) (All Weather T...,Flat,3yo,14,5,1,5,1,5,4.0,8.0,False,35.714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,2025-06-21,Ascot,Great Britain,3:40,Queen Elizabeth II Jubilee Stakes,Flat,4yo+,1,1,-1,-1,-1,1,3.0,3.0,True,100.0
179,2025-06-27,Newmarket (July),Great Britain,6:10,Boodles Fillies Novice Stakes (GBB Race),Flat,3yo+,1,1,-1,-1,-1,1,2.0,2.0,True,100.0
180,2025-12-21,Monterrico,Peru,22:50,Gran Premio Nacional Augusta B Leguia (3yo) (T...,Flat,3yo,1,1,-1,-1,-1,1,2.0,2.0,True,100.0
181,2026-02-07,Caulfield,Australia,05:15,Evergreen Turf Peter Le Grand Stakes (Fillies)...,Flat,3yo,1,1,1,1,1,1,4.0,4.0,True,100.0


Runner contradictions more than one year outside the band


,source_rowid,date,course,jurisdiction,off,race_name,type,horse,age,age_band,minimum_age_numeric,maximum_age_numeric,age_difference_from_band
0,5191,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,All Stormy (USA),6,4yo,4.0,4.0,2
1,5193,2015-01-17,Fair Grounds (USA),United States,8:55,Colonel E.R Bradley Handicap (Turf),Flat,Villandry (USA),6,4yo,4.0,4.0,2
2,60016,2015-05-25,Prairie Meadows (USA),United States,10:10,Jim Rasmussen Memorial Stakes (Dirt),Flat,Dens Legacy (USA),5,3yo,3.0,3.0,2
3,60020,2015-05-25,Prairie Meadows (USA),United States,10:10,Jim Rasmussen Memorial Stakes (Dirt),Flat,Cougar Ridge (USA),5,3yo,3.0,3.0,2
4,60023,2015-05-25,Prairie Meadows (USA),United States,10:10,Jim Rasmussen Memorial Stakes (Dirt),Flat,Lahshad (USA),5,3yo,3.0,3.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,207820,2016-05-14,Percy Warner Park (USA),United States,10:30,Calvin Houghland Iroquois (Hurdle) (4yo ) (Turf),Hurdle,Italian Wedding (USA),11,4yo,4.0,4.0,7
489,1461352,2024-02-04,St Moritz (SWI),Switzerland,1:45,Preis DC Aviation (Conditions) (4yo) (Snow),Flat,Fleur DIpanema (FR),11,4yo,4.0,4.0,7
490,1780730,2025-12-18,Bahrain,Bahrain,15:30,Dallah Cup (Handicap) (3yo ) (Outer Track) (Turf),Flat,Nine Below Zero (GB),10,3yo,3.0,3.0,7
491,207819,2016-05-14,Percy Warner Park (USA),United States,10:30,Calvin Houghland Iroquois (Hurdle) (4yo ) (Turf),Hurdle,Pierrot Lunaire (USA),12,4yo,4.0,4.0,8


In [10]:
# Input grain:
#   One row per provisional race from `jurisdiction_frame`, plus runner-level
#   contradiction evidence from `runner_eligibility`.
#
# Output grain:
#   1. One row per exact raw age_band showing how often the race name contains
#      a matching age expression.
#   2. One row per contradictory race with age expressions extracted from its
#      race name.
#   3. One summary row per relationship between structured age_band and
#      race-name age text.
#   4. One detailed row for races where the structured field and race-name text
#      visibly disagree.
#
# Purpose:
#   Test whether apparent runner-age contradictions are caused by the meaning of
#   `age_band`, by a malformed structured field, or by source extraction from
#   race-name text.
#
# Raw versus derived values:
#   Raw race_name and age_band values remain unchanged.
#
#   Derived race-name evidence is limited to complete age expressions matching
#   the same source-like forms already observed:
#   - Nyo
#   - Nyo+
#   - N-Myo
#
#   Parentheses, spaces and surrounding words are retained in the raw race name.
#   Extracted age expressions are evidence only and do not overwrite age_band.
#
# Assumptions deliberately not made:
#   - race-name text is not treated as more authoritative than age_band;
#   - the first age expression is not automatically the race eligibility rule;
#   - ages appearing in sponsor names or unrelated text remain possible;
#   - missing `+` is not silently repaired;
#   - races without age text are not assumed to have no age restriction;
#   - runner-age contradictions remain unresolved until their source pattern is
#     understood.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - required prior-stage dataframes are unavailable;
#   - the race population differs from 189,043;
#   - the contradictory-race population differs from 183;
#   - the relationship partition does not reconstruct all contradictory races.
#
#   No source values are written or modified.

import re


required_prior_names = [
    "jurisdiction_frame",
    "runner_eligibility",
    "age_difference_by_race",
    "EXPECTED_PROVISIONAL_RACES",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding age-band stages first. "
        f"Missing variables: {missing_prior_names}"
    )


EXPECTED_CONTRADICTORY_RACES = 183


# Match only age expressions shaped like the structured source vocabulary.
#
# The negative lookbehind prevents matching the trailing part of a larger
# number. Matching is case-insensitive because source race names vary in style.
RACE_NAME_AGE_PATTERN = re.compile(
    r"(?<!\d)(\d{1,2}(?:-\d{1,2})?yo\+?)",
    flags=re.IGNORECASE,
)


def extract_race_name_age_expressions(race_name):
    """
    Return exact age-like expressions found in a raw race name.

    Duplicates are removed while preserving their first-seen order.
    """
    if pd.isna(race_name):
        return tuple()

    matches = RACE_NAME_AGE_PATTERN.findall(str(race_name))

    ordered_unique = []
    seen = set()

    for match in matches:
        normalised_case = match.lower()

        if normalised_case not in seen:
            seen.add(normalised_case)
            ordered_unique.append(match)

    return tuple(ordered_unique)


def normalise_age_expression_for_comparison(raw_value):
    """
    Normalise case only for comparing two already-extracted source strings.

    Punctuation, plus signs and numeric ranges are otherwise preserved.
    """
    if pd.isna(raw_value):
        return None

    return str(raw_value).strip().lower()


# Build one clean race-level evidence frame.
race_name_age_frame = jurisdiction_frame[
    [
        "date",
        "course",
        "jurisdiction",
        "off",
        "race_name",
        "type",
        "age_band",
    ]
].copy()


if len(race_name_age_frame) != EXPECTED_PROVISIONAL_RACES:
    raise AssertionError(
        "Race-name age frame does not match the established race population: "
        f"{len(race_name_age_frame):,} != "
        f"{EXPECTED_PROVISIONAL_RACES:,}"
    )


race_name_age_frame["race_name_age_expressions"] = (
    race_name_age_frame["race_name"].map(
        extract_race_name_age_expressions
    )
)

race_name_age_frame["race_name_age_expression_count"] = (
    race_name_age_frame["race_name_age_expressions"].map(len)
)

race_name_age_frame["normalised_age_band"] = (
    race_name_age_frame["age_band"].map(
        normalise_age_expression_for_comparison
    )
)


def age_band_appears_in_race_name(row):
    """
    Return whether the exact structured age-band text appears among the
    extracted age expressions in the same race name.
    """
    age_band = row["normalised_age_band"]

    if age_band in {None, ""}:
        return False

    extracted = {
        normalise_age_expression_for_comparison(value)
        for value in row["race_name_age_expressions"]
    }

    return age_band in extracted


race_name_age_frame["exact_age_band_in_race_name"] = (
    race_name_age_frame.apply(
        age_band_appears_in_race_name,
        axis=1,
    )
)


# Profile race-name age evidence for every structured age-band value.
age_band_race_name_summary = (
    race_name_age_frame
    .groupby(
        "age_band",
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        races_with_any_age_text=(
            "race_name_age_expression_count",
            lambda values: int(values.gt(0).sum()),
        ),
        races_with_exact_age_band_text=(
            "exact_age_band_in_race_name",
            "sum",
        ),
        distinct_race_name_age_expression_sets=(
            "race_name_age_expressions",
            "nunique",
        ),
    )
)


age_band_race_name_summary["any_age_text_pct"] = (
    age_band_race_name_summary["races_with_any_age_text"]
    / age_band_race_name_summary["provisional_races"]
    * 100
).round(3)

age_band_race_name_summary["exact_age_band_text_pct"] = (
    age_band_race_name_summary["races_with_exact_age_band_text"]
    / age_band_race_name_summary["provisional_races"]
    * 100
).round(3)

age_band_race_name_summary = (
    age_band_race_name_summary
    .sort_values(
        [
            "provisional_races",
            "age_band",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# Attach race-name age evidence to the 183 contradictory races.
contradictory_race_name_age = (
    age_difference_by_race
    .merge(
        race_name_age_frame[
            [
                "date",
                "course",
                "off",
                "race_name_age_expressions",
                "race_name_age_expression_count",
                "exact_age_band_in_race_name",
            ]
        ],
        on=["date", "course", "off"],
        how="left",
        validate="one_to_one",
    )
)


if len(contradictory_race_name_age) != EXPECTED_CONTRADICTORY_RACES:
    raise AssertionError(
        "Contradictory-race population changed: "
        f"{len(contradictory_race_name_age):,} != "
        f"{EXPECTED_CONTRADICTORY_RACES:,}"
    )


def classify_age_text_relationship(row):
    """
    Describe the relationship between structured age_band and race-name text.

    Categories are observations only, not correction decisions.
    """
    if row["race_name_age_expression_count"] == 0:
        return "no_age_expression_in_race_name"

    if row["exact_age_band_in_race_name"]:
        return "exact_structured_value_repeated_in_race_name"

    return "different_age_expression_in_race_name"


contradictory_race_name_age["age_text_relationship"] = (
    contradictory_race_name_age.apply(
        classify_age_text_relationship,
        axis=1,
    )
)


age_text_relationship_summary = (
    contradictory_race_name_age
    .groupby(
        [
            "age_text_relationship",
            "jurisdiction",
            "type",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        contradictory_races=("race_name", "size"),
        contradictory_runner_rows=("runner_rows", "sum"),
        minimum_age_difference=("minimum_age_difference", "min"),
        maximum_age_difference=("maximum_age_difference", "max"),
    )
    .sort_values(
        [
            "contradictory_races",
            "age_text_relationship",
            "jurisdiction",
            "type",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)


if int(
    age_text_relationship_summary["contradictory_races"].sum()
) != EXPECTED_CONTRADICTORY_RACES:
    raise AssertionError(
        "Age-text relationship summary does not reconstruct 183 races."
    )


# Preserve races where another age expression is visible in the race name.
different_age_text_races = (
    contradictory_race_name_age.loc[
        contradictory_race_name_age[
            "age_text_relationship"
        ].eq("different_age_expression_in_race_name"),
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "age_band",
            "race_name_age_expressions",
            "runner_rows",
            "minimum_observed_runner_age",
            "maximum_observed_runner_age",
            "minimum_age_difference",
            "maximum_age_difference",
        ],
    ]
    .sort_values(
        [
            "jurisdiction",
            "date",
            "course",
            "off",
        ]
    )
    .reset_index(drop=True)
)


print("Race-name age evidence by structured age-band value")
display(age_band_race_name_summary)

print("Contradictory races with extracted race-name age expressions")
display(contradictory_race_name_age)

print("Age-text relationships among contradictory races")
display(age_text_relationship_summary)

print("Contradictory races carrying a different age expression in race_name")
display(different_age_text_races)

Race-name age evidence by structured age-band value


,age_band,provisional_races,races_with_any_age_text,races_with_exact_age_band_text,distinct_race_name_age_expression_sets,any_age_text_pct,exact_age_band_text_pct
0,4yo+,62474,6666,6660,6,10.670,10.660
1,3yo+,53295,14133,14125,4,26.518,26.503
2,3yo,25175,10142,10141,4,40.286,40.282
3,2yo,19300,4261,4261,4,22.078,22.078
4,5yo+,16223,2481,2480,3,15.293,15.287
5,4yo,4029,2459,2459,3,61.033,61.033
6,4-6yo,2634,0,0,1,0.000,0.000
7,3-5yo,1078,2,2,2,0.186,0.186
8,4-7yo,1021,0,0,1,0.000,0.000
9,2yo+,884,761,760,4,86.086,85.973


Contradictory races with extracted race-name age expressions


,date,course,jurisdiction,off,race_name,type,age_band,runner_rows,distinct_age_differences,minimum_age_difference,...,most_common_age_difference,most_common_difference_rows,minimum_observed_runner_age,maximum_observed_runner_age,uniform_difference,dominant_difference_pct,race_name_age_expressions,race_name_age_expression_count,exact_age_band_in_race_name,age_text_relationship
0,2024-06-23,Monterrico (PER),Peru,10:45,Premio Pamplona (2yo Fillies & Mares) (Turf),Flat,2yo,16,3,1,...,1,11,3.0,5.0,False,68.75,"(2yo,)",1,True,exact_structured_value_repeated_in_race_name
1,2015-08-01,Greyville (SAF),South Africa,2:30,Premiers Champion Stakes (2yo) (Turf),Flat,2yo,16,1,1,...,1,16,3.0,3.0,True,100.0,"(2yo,)",1,True,exact_structured_value_repeated_in_race_name
2,2015-10-25,Saint-Cloud (FR),France,4:15,Prix des Bords de Seine (Handicap) (4yo ) (Turf),Flat,4yo,15,3,1,...,2,7,5.0,9.0,False,46.667,"(4yo,)",1,True,exact_structured_value_repeated_in_race_name
3,2024-09-08,Seoul (KOR),South Korea,7:20,Korea Sprint (3yo ) (Dirt),Flat,3yo,14,6,1,...,1,4,4.0,9.0,False,28.571,"(3yo,)",1,True,exact_structured_value_repeated_in_race_name
4,2017-05-31,Sha Tin (HK),Hong Kong,12:45,Silvermine Bay Handicap (3yo ) (All Weather T...,Flat,3yo,14,5,1,...,1,5,4.0,8.0,False,35.714,"(3yo,)",1,True,exact_structured_value_repeated_in_race_name
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,2025-06-21,Ascot,Great Britain,3:40,Queen Elizabeth II Jubilee Stakes,Flat,4yo+,1,1,-1,...,-1,1,3.0,3.0,True,100.0,(),0,False,no_age_expression_in_race_name
179,2025-06-27,Newmarket (July),Great Britain,6:10,Boodles Fillies Novice Stakes (GBB Race),Flat,3yo+,1,1,-1,...,-1,1,2.0,2.0,True,100.0,(),0,False,no_age_expression_in_race_name
180,2025-12-21,Monterrico,Peru,22:50,Gran Premio Nacional Augusta B Leguia (3yo) (T...,Flat,3yo,1,1,-1,...,-1,1,2.0,2.0,True,100.0,"(3yo,)",1,True,exact_structured_value_repeated_in_race_name
181,2026-02-07,Caulfield,Australia,05:15,Evergreen Turf Peter Le Grand Stakes (Fillies)...,Flat,3yo,1,1,1,...,1,1,4.0,4.0,True,100.0,(),0,False,no_age_expression_in_race_name


Age-text relationships among contradictory races


,age_text_relationship,jurisdiction,type,contradictory_races,contradictory_runner_rows,minimum_age_difference,maximum_age_difference
0,no_age_expression_in_race_name,Great Britain,Flat,24,26,-1,1
1,exact_structured_value_repeated_in_race_name,France,Flat,23,155,-1,6
2,exact_structured_value_repeated_in_race_name,Peru,Flat,21,110,-1,5
3,exact_structured_value_repeated_in_race_name,Hong Kong,Flat,13,139,-1,5
4,exact_structured_value_repeated_in_race_name,United States,Flat,13,47,-1,4
5,exact_structured_value_repeated_in_race_name,Australia,Flat,11,45,-1,5
6,no_age_expression_in_race_name,Ireland,Flat,8,8,-1,1
7,exact_structured_value_repeated_in_race_name,Argentina,Flat,6,36,-1,2
8,exact_structured_value_repeated_in_race_name,Canada,Flat,6,23,1,28
9,exact_structured_value_repeated_in_race_name,South Korea,Flat,6,69,1,6


Contradictory races carrying a different age expression in race_name


,date,course,jurisdiction,off,race_name,type,age_band,race_name_age_expressions,runner_rows,minimum_observed_runner_age,maximum_observed_runner_age,minimum_age_difference,maximum_age_difference
0,2017-05-16,Compiegne (FR),France,1:35,Prix du Morbihan (Hurdle) (Claimer) (5yo+) (Turf),Hurdle,5yo,"(5yo+,)",11,6.0,11.0,1,6


## Age-band semantics and runner-age consistency

The source `age_band` field is structurally consistent and fully parseable across the current extract.

Observed forms are limited to:

- exact ages such as `2yo`, `3yo`, `4yo`;
- open-ended minimum ages such as `3yo+`, `4yo+`, `5yo+`;
- closed ranges such as `3-5yo`, `4-6yo`, `5-7yo`;
- 13 blank race values.

All 27 populated raw forms matched one of these syntactic families without requiring correction.

The field can therefore be parsed safely into:

- raw source value;
- syntax type;
- minimum stated age;
- maximum stated age where explicitly present;
- whether the source explicitly includes a `+`.

However, a literal comparison with the runner-level source `age` field produced 958 apparent contradictions across 183 provisional races.

These contradictions are not uniform:

- some runners are one year outside the stated band;
- some races contain several older ages despite an exact-age label such as `3yo` or `4yo`;
- some contradictions are concentrated by jurisdiction and race;
- one runner has an obviously implausible recorded age of `31`;
- only one contradictory race showed a visible disagreement between structured `age_band` and age text in `race_name`:
  - Compiegne, 16 May 2017, 13:35;
  - structured `age_band`: `5yo`;
  - race name: `Prix du Morbihan (Hurdle) (Claimer) (5yo+) (Turf)`.

The race-name comparison otherwise showed that the structured field usually repeats the source-presented age expression exactly. The apparent contradictions therefore cannot be explained simply as widespread extraction errors from `race_name`.

### Analytical conclusion

`age_band` is safe to treat as a structured representation of the source-presented race age condition.

It is not yet safe to treat every parsed value as a universally enforceable eligibility rule against the stored runner `age`.

In particular, an exact form such as `4yo` must not automatically be interpreted as proving both:

- minimum eligible age = 4; and
- maximum eligible age = 4

across every jurisdiction and source feed.

The reusable implementation should therefore:

1. preserve the raw `age_band`;
2. parse its observed syntax deterministically;
3. expose stated minimum, stated maximum and explicit open-ended status;
4. preserve blank values as null rather than inferring a condition;
5. flag runner-age inconsistencies for review;
6. avoid automatically correcting either `age_band` or runner `age`;
7. avoid using runner-age disagreement as a parser failure.

### Limitations

This study did not independently reconstruct horse ages from foaling dates.

The runner-level `age` values used in the comparison are the ages recorded directly in the source.

Jurisdiction-specific ageing rules, birthday conventions, source-feed transformations and individual source errors may all contribute to the observed contradictions. External verification is required before defining jurisdiction-specific correction rules.

In [11]:
# Input grain:
#   One row per provisional race from `jurisdiction_frame`.
#
# Output grain:
#   1. One row per Great Britain race type, class and rating-band combination.
#   2. One summary row per British class showing rating-band coverage and
#      observed lower and upper bounds.
#   3. One row per jurisdiction, race type and pattern value.
#   4. One row per race where both class and pattern are populated.
#   5. One row per race carrying the specific Compiegne age-band anomaly.
#
# Purpose:
#   Establish how `class`, `pattern` and `rating_band` coexist and whether any
#   deterministic relationship can be safely inferred.
#
# Raw versus derived values:
#   Raw class, pattern, rating_band and race-name values remain unchanged.
#
#   Numeric rating bounds are reused only from rating bands already proven to
#   match the complete `N-N` syntax. Unrecognised values such as `--` and
#   `(75-100)` remain unparsed.
#
# Assumptions deliberately not made:
#   - class is not derived from rating-band boundaries;
#   - equal rating bands do not imply equal race classes;
#   - pattern status is not treated as mutually exclusive with class;
#   - Group and Grade values are not merged;
#   - Listed, Group and Grade labels are not assigned one global hierarchy;
#   - blank class or pattern is not interpreted as absence of status;
#   - the Compiegne anomaly is not silently corrected.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - required prior-stage frames are unavailable;
#   - the British subset is empty;
#   - parsed rating bounds fail to attach to their exact raw values;
#   - grouped British counts do not reconstruct the British race population;
#   - the Compiegne anomaly cannot be reproduced exactly once.
#
#   No source or reference data is written or modified.

required_prior_names = [
    "jurisdiction_frame",
    "rating_band_profile",
    "EXPECTED_PROVISIONAL_RACES",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding classification and eligibility stages first. "
        f"Missing variables: {missing_prior_names}"
    )


# Build a lookup only for exact closed integer rating ranges.
rating_bound_lookup = (
    rating_band_profile.loc[
        rating_band_profile[
            "provisional_syntax"
        ].eq("closed_integer_range"),
        [
            "raw_rating_band",
            "lower_bound",
            "upper_bound",
        ],
    ]
    .rename(
        columns={
            "raw_rating_band": "rating_band",
            "lower_bound": "rating_lower_bound",
            "upper_bound": "rating_upper_bound",
        }
    )
    .copy()
)


if rating_bound_lookup["rating_band"].duplicated().any():
    raise AssertionError(
        "Rating-bound lookup contains duplicate raw values."
    )


classification_relationships = jurisdiction_frame[
    [
        "date",
        "course",
        "jurisdiction",
        "off",
        "race_name",
        "type",
        "class",
        "pattern",
        "rating_band",
        "age_band",
    ]
].copy()


classification_relationships = (
    classification_relationships
    .merge(
        rating_bound_lookup,
        on="rating_band",
        how="left",
        validate="many_to_one",
    )
)


# Make blank values visible for grouped output without altering raw fields.
for field in ["class", "pattern", "rating_band"]:
    classification_relationships[
        f"{field}_display"
    ] = classification_relationships[field].map(
        display_raw_state
    )


# Restrict the class/rating study to Great Britain because class is complete
# there and sparse elsewhere.
british_races = classification_relationships.loc[
    classification_relationships[
        "jurisdiction"
    ].eq("Great Britain")
].copy()


if british_races.empty:
    raise AssertionError(
        "No Great Britain races were available for class/rating analysis."
    )


# Count every exact British type/class/rating combination.
british_class_rating_combinations = (
    british_races
    .groupby(
        [
            "type",
            "class_display",
            "rating_band_display",
            "rating_lower_bound",
            "rating_upper_bound",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "type",
            "class_display",
            "provisional_races",
            "rating_band_display",
        ],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)


if int(
    british_class_rating_combinations[
        "provisional_races"
    ].sum()
) != len(british_races):
    raise AssertionError(
        "British class/rating combinations do not reconstruct "
        "the British race population."
    )


# Summarise the range of rating bands observed inside each British class.
british_class_rating_summary = (
    british_races
    .groupby(
        [
            "type",
            "class_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        populated_rating_band_races=(
            "rating_band",
            lambda values: int(
                values.astype("string").str.strip().ne("").sum()
            ),
        ),
        distinct_populated_rating_bands=(
            "rating_band",
            lambda values: values.loc[
                values.astype("string").str.strip().ne("")
            ].nunique(),
        ),
        minimum_observed_lower_bound=(
            "rating_lower_bound",
            "min",
        ),
        maximum_observed_lower_bound=(
            "rating_lower_bound",
            "max",
        ),
        minimum_observed_upper_bound=(
            "rating_upper_bound",
            "min",
        ),
        maximum_observed_upper_bound=(
            "rating_upper_bound",
            "max",
        ),
    )
)


british_class_rating_summary[
    "rating_band_coverage_pct"
] = (
    british_class_rating_summary[
        "populated_rating_band_races"
    ]
    / british_class_rating_summary["provisional_races"]
    * 100
).round(3)


british_class_rating_summary = (
    british_class_rating_summary
    .sort_values(
        ["type", "class_display"]
    )
    .reset_index(drop=True)
)


# Profile pattern vocabulary independently by jurisdiction and race type.
pattern_by_jurisdiction = (
    classification_relationships
    .groupby(
        [
            "jurisdiction",
            "type",
            "pattern_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "jurisdiction",
            "type",
            "provisional_races",
            "pattern_display",
        ],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)


# Preserve every race where class and pattern coexist.
class_and_pattern_races = (
    classification_relationships.loc[
        classification_relationships[
            "class"
        ].astype("string").str.strip().ne("")
        & classification_relationships[
            "pattern"
        ].astype("string").str.strip().ne(""),
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "rating_band",
        ],
    ]
    .sort_values(
        [
            "jurisdiction",
            "date",
            "course",
            "off",
        ]
    )
    .reset_index(drop=True)
)


# Reproduce the sole observed structured age-band versus race-name discrepancy.
compiegne_age_band_anomaly = (
    classification_relationships.loc[
        classification_relationships["date"].eq(
            "2017-05-16"
        )
        & classification_relationships["course"].eq(
            "Compiegne (FR)"
        )
        & classification_relationships["off"].eq(
            "1:35"
        ),
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "class",
            "pattern",
            "rating_band",
            "age_band",
        ],
    ]
    .reset_index(drop=True)
)


if len(compiegne_age_band_anomaly) != 1:
    raise AssertionError(
        "Expected exactly one Compiegne age-band anomaly, found "
        f"{len(compiegne_age_band_anomaly):,}."
    )


print("Great Britain type, class and exact rating-band combinations")
display(british_class_rating_combinations)

print("Great Britain rating-band coverage and bounds by class")
display(british_class_rating_summary)

print("Pattern vocabulary by jurisdiction and source race type")
display(pattern_by_jurisdiction)

print("Races where both class and pattern are populated")
display(class_and_pattern_races)

print("Sole structured age-band versus race-name discrepancy")
display(compiegne_age_band_anomaly)

Great Britain type, class and exact rating-band combinations


,type,class_display,rating_band_display,rating_lower_bound,rating_upper_bound,provisional_races,distinct_race_names,first_date,last_date
0,Chase,Class 1,<BLANK>,NaN,NaN,1078,658,2015-01-01,2026-04-25
1,Chase,Class 1,0-155,0.0,155.0,13,7,2015-01-24,2021-11-06
2,Chase,Class 1,0-150,0.0,150.0,9,5,2015-10-31,2021-10-30
3,Chase,Class 1,0-145,0.0,145.0,4,3,2018-03-13,2021-03-13
4,Chase,Class 1,0-140,0.0,140.0,3,2,2015-03-10,2017-03-14
...,...,...,...,...,...,...,...,...,...
139,NH Flat,Class 2,<BLANK>,NaN,NaN,61,51,2015-03-21,2026-04-18
140,NH Flat,Class 3,<BLANK>,NaN,NaN,53,44,2015-04-18,2026-05-09
141,NH Flat,Class 4,<BLANK>,NaN,NaN,475,413,2015-02-14,2026-05-24
142,NH Flat,Class 5,<BLANK>,NaN,NaN,1771,1535,2015-01-07,2026-05-26


Great Britain rating-band coverage and bounds by class


,type,class_display,provisional_races,populated_rating_band_races,distinct_populated_rating_bands,minimum_observed_lower_bound,maximum_observed_lower_bound,minimum_observed_upper_bound,maximum_observed_upper_bound,rating_band_coverage_pct
0,Chase,Class 1,1107,29,4,0.0,0.0,140.0,155.0,2.620
1,Chase,Class 2,1325,497,3,0.0,0.0,145.0,155.0,37.509
2,Chase,Class 3,4060,3044,4,0.0,0.0,125.0,140.0,74.975
3,Chase,Class 4,5602,4933,6,0.0,0.0,105.0,135.0,88.058
4,Chase,Class 5,3292,2997,3,0.0,0.0,70.0,105.0,91.039
5,Chase,Class 6,285,1,1,0.0,0.0,65.0,65.0,0.351
6,Flat,Class 1,3122,14,2,0.0,0.0,105.0,110.0,0.448
7,Flat,Class 2,4836,2661,8,0.0,86.0,90.0,110.0,55.025
8,Flat,Class 3,5482,4587,11,0.0,81.0,85.0,130.0,83.674
9,Flat,Class 4,14563,10165,14,0.0,70.0,75.0,95.0,69.800


Pattern vocabulary by jurisdiction and source race type


,jurisdiction,type,pattern_display,provisional_races,distinct_race_names,first_date,last_date
0,Argentina,Flat,Group 1,414,189,2015-02-07,2026-05-25
1,Argentina,Flat,Grade 1,10,10,2019-10-05,2026-02-07
2,Argentina,Flat,Group 3,8,8,2015-01-03,2015-04-10
3,Argentina,Flat,Group 2,3,3,2015-01-27,2015-02-21
4,Argentina,Flat,<BLANK>,2,2,2015-01-24,2024-06-28
...,...,...,...,...,...,...,...
208,United States,Hurdle,<BLANK>,25,21,2017-10-21,2026-05-09
209,United States,Hurdle,Grade 3,1,1,2025-11-01,2025-11-01
210,Uruguay,Flat,Group 1,67,32,2015-01-06,2026-01-06
211,Uruguay,Flat,Grade 1,2,2,2020-01-06,2020-01-06


Races where both class and pattern are populated


,date,course,jurisdiction,off,race_name,type,class,pattern,rating_band
0,2025-11-01,San Isidro,Argentina,21:30,Gran Premio Enrique Acebal (3yo Fillies) (Roun...,Flat,Class 1,Group 1,
1,2025-11-01,San Isidro,Argentina,22:40,Gran Premio Copa de Oro - Alfredo Lalor (4yo+)...,Flat,Class 1,Group 1,
2,2025-11-08,Palermo,Argentina,19:30,Gran Premio Palermo - Copa Haras Firmamento (3...,Flat,Class 1,Group 1,
3,2025-11-08,Palermo,Argentina,21:15,Gran Premio Nacional (3yo) (Dirt),Flat,Class 1,Group 1,
4,2025-11-08,Palermo,Argentina,22:15,Gran Premio Maipu (3yo+) (Dirt),Flat,Class 1,Group 1,
...,...,...,...,...,...,...,...,...,...
6382,2026-05-25,Santa Anita,United States,23:30,Gamely Stakes (Fillies & Mares) (Turf),Flat,Class 1,Grade 1,
6383,2026-05-26,Lone Star Park,United States,01:07,Steve Sexton Mile Stakes (Dirt),Flat,Class 1,Grade 3,
6384,2026-05-26,Santa Anita,United States,00:30,Shoemaker Mile Stakes (Turf),Flat,Class 1,Grade 1,
6385,2026-01-06,Maronas,Uruguay,20:50,Gran Premio Ciudad de Montevideo - Presidente ...,Flat,Class 1,Group 1,


Sole structured age-band versus race-name discrepancy


,date,course,jurisdiction,off,race_name,type,class,pattern,rating_band,age_band
0,2017-05-16,Compiegne (FR),France,1:35,Prix du Morbihan (Hurdle) (Claimer) (5yo+) (Turf),Hurdle,,,,5yo


## Age-band semantics and runner-age consistency

The source `age_band` field is structurally consistent and fully parseable across the current extract.

Observed forms are limited to:

- exact ages such as `2yo`, `3yo`, `4yo`;
- open-ended minimum ages such as `3yo+`, `4yo+`, `5yo+`;
- closed ranges such as `3-5yo`, `4-6yo`, `5-7yo`;
- 13 blank race values.

All 27 populated raw forms matched one of these syntactic families without requiring correction.

The field can therefore be parsed safely into:

- raw source value;
- syntax type;
- minimum stated age;
- maximum stated age where explicitly present;
- whether the source explicitly includes a `+`.

However, a literal comparison with the runner-level source `age` field produced 958 apparent contradictions across 183 provisional races.

These contradictions are not uniform:

- some runners are one year outside the stated band;
- some races contain several older ages despite an exact-age label such as `3yo` or `4yo`;
- some contradictions are concentrated by jurisdiction and race;
- some runner ages are individually implausible;
- one race showed a visible disagreement between structured `age_band` and the age expression embedded in `race_name`.

The race-name comparison otherwise showed that the structured field usually repeats the source-presented age expression exactly. The apparent contradictions therefore cannot be explained as one general extraction failure.

## Manual and external verification decision

Manual-verification status for this notebook: `captured`.

Four bounded checks have been preserved in `data/reference/manual_verifications.csv`.

### `NB16-AGE-0001` — Compiegne age-band defect

Race:

- date: 16 May 2017;
- course: Compiegne (FR);
- off: 13:35;
- race: Prix du Morbihan.

Observed source values:

- structured `age_band`: `5yo`;
- race-name age expression: `5yo+`.

External result evidence supports `5yo+`.

Governed decision:

- verification status: `contradicted`;
- confidence: high;
- database action: `source_correction_candidate`.

The immutable raw `age_band = 5yo` must remain preserved. A downstream governed reconciliation layer may expose the externally verified `5yo+` condition with `NB16-AGE-0001` retained as provenance.

### `NB16-AGE-0002` — implausible runner age

The source records Ecstasy as age `31` in a race externally identified as a three-year-old race.

External evidence identifies the horse as age `3`.

Governed decision:

- verification status: `contradicted`;
- confidence: high;
- database action: `source_correction_candidate`.

The immutable source age `31` must remain preserved. A corrected age of `3` may be applied only through a governed downstream reconciliation layer carrying `NB16-AGE-0002` as provenance.

This case confirms that the runner-level `age` field can contain isolated source errors and requires anomaly validation before use.

### `NB16-AGE-0003` — exact-looking label with older runners

A Fair Grounds race was externally presented with an age expression of `(4yo)` while the published result included runners older than four.

Governed decision:

- verification status: `confirmed`;
- confidence: high;
- database action: `evidence_only`.

This evidence does not establish a replacement source value. It establishes that an isolated exact-looking age expression cannot always be interpreted globally as a closed eligibility rule.

The case therefore supports semantic caution rather than a source correction.

### `NB16-AGE-0004` — Greyville age discrepancy

A Greyville race was externally identified as a two-year-old race, while the source records all runners as age `3`.

Governed decision:

- verification status: `partially_confirmed`;
- confidence: medium;
- database action: `preserve_raw_unresolved`.

The external evidence confirms the published race condition but does not establish why the source runner ages differ.

Possible explanations include:

- jurisdictional age conventions;
- source-feed age conversion;
- date or season-boundary treatment;
- systematic runner-age error.

No correction rule is justified from this case alone.

## Analytical conclusion

`age_band` is safe to treat as a structured representation of the source-presented race age condition.

It is not safe to treat every parsed value as a universally enforceable eligibility rule against the stored runner `age`.

In particular, an exact form such as `4yo` must not automatically be interpreted as proving both:

- minimum eligible age = 4; and
- maximum eligible age = 4

across every jurisdiction and source feed.

The runner-level `age` field remains usable for analysis, but it requires:

- numeric parsing;
- plausible-range validation;
- explicit anomaly flags;
- preservation of the raw source value;
- governed correction provenance where an external correction is confirmed.

The reusable implementation should therefore:

1. preserve raw `age_band`;
2. parse its observed syntax deterministically;
3. expose stated minimum, stated maximum and explicit open-ended status;
4. preserve blank values as null rather than inferring a condition;
5. preserve raw runner `age`;
6. create a numeric runner-age representation only where parsing succeeds;
7. flag implausible runner ages separately from race-condition contradictions;
8. treat runner-age versus age-band disagreements as review evidence rather than automatic parser failures;
9. apply externally verified corrections only through the governed reconciliation layer;
10. retain the relevant `verification_id`, method, confidence and database action with every applied correction.

## Limitations

This study did not independently reconstruct every horse’s age from foaling dates.

The runner-level `age` values used in the source-wide comparison are the ages recorded directly in the source.

The four manual checks establish several distinct failure modes but do not explain all 958 apparent contradictions.

Jurisdiction-specific ageing rules, birthday conventions, source-feed transformations, shorthand race conditions and individual source errors may all contribute.

No universal jurisdiction-level correction rule should be created until the relevant convention has been established from broader evidence.

## Class, pattern and rating-band relationships

The fields `class`, `pattern` and `rating_band` describe related but distinct aspects of race classification.

### British class and rating bands

The `class` field is populated for all Great Britain races in the current extract.

British values range from `Class 1` to `Class 7`, with different coverage by race type:

- Flat races use Classes 1–7;
- Chase races use Classes 1–6;
- Hurdle races use Classes 1–5;
- NH Flat races use Classes 1–6.

The `rating_band` field is not complete within each class and does not map one-to-one to class.

Examples include:

- Chase Class 3 races with upper rating limits from 125 to 140;
- Flat Class 4 races with upper limits from 75 to 95;
- Flat Class 5 races with upper limits from 55 to 80;
- Flat Class 6 races with upper limits from 50 to 85;
- Hurdle Class 4 races with upper limits from 85 to 125.

Rating ranges overlap across adjacent classes.

Therefore:

> British `class` cannot be derived safely from `rating_band`, and `rating_band` cannot be inferred safely from `class`.

The two fields must remain separate source attributes.

Rating-band coverage also varies substantially:

- many Class 1 races have no populated rating band;
- NH Flat races have class values but no populated rating bands;
- lower-class handicaps generally have stronger rating-band coverage;
- rating bands appear to describe eligibility restrictions where applicable rather than universal race quality.

### Pattern status

The populated `pattern` vocabulary contains:

- `Listed`;
- `Group 1`, `Group 2`, `Group 3`;
- `Grade 1`, `Grade 2`, `Grade 3`;
- `Grade A`, `Grade B`, `Grade C`.

Pattern labels occur across multiple jurisdictions and race types.

They must not be reduced to one universal hierarchy without jurisdiction and racing-code context.

In particular:

- `Group` and `Grade` must remain distinct raw families;
- identical-looking Grade labels must not automatically be treated as equivalent across jurisdictions or racing codes;
- historical Grade A, B and C values must remain preserved rather than translated automatically.

### Coexistence of class and pattern

There are 6,387 provisional races where both `class` and `pattern` are populated.

This proves that the fields are not mutually exclusive.

A race can simultaneously carry:

- a broad class value such as `Class 1`; and
- a pattern status such as `Group 1`, `Grade 1` or `Listed`.

The database must therefore store them independently.

The presence of `Class 1` on international races also shows that the field is not exclusively British. International `Class 1` values must not automatically be assigned the regulatory meaning of Great Britain Class 1.

### Analytical conclusion

The safe database treatment is:

1. preserve raw `class`;
2. parse the numeric component only from canonical `Class N` values;
3. preserve raw `pattern`;
4. preserve Listed, Group and Grade families separately;
5. preserve raw `rating_band`;
6. parse only recognised closed integer ranges;
7. retain `--` and `(75-100)` as explicit unresolved source forms;
8. do not derive class from rating band;
9. do not derive rating band from class;
10. do not collapse class and pattern into one field;
11. require jurisdiction and race type before assigning deeper regulatory meaning.

### Limitations

This analysis establishes source structure, coexistence and coverage. It does not establish the full regulatory meaning of every class, Group or Grade label across every jurisdiction and historical period.

Jurisdiction-specific classification hierarchies should be investigated separately where required for a defined analytical question.

In [12]:
# Input grain:
#   Runner-level rows from `runner_eligibility`, with one row per retained
#   source runner.
#
# Output grain:
#   1. One row per raw `sex_rest` and raw runner `sex` combination.
#   2. One row per `sex_rest`, jurisdiction, race type and runner-sex value.
#   3. One row per `sex_rest`, runner age and runner-sex value.
#   4. Sample races for rare or apparently contradictory combinations.
#
# Purpose:
#   Investigate what the race-level `sex_rest` field appears to represent,
#   without assuming that its letters can be compared literally with the
#   runner-level `sex` field.
#
# Important semantic caution:
#   The two source fields use different vocabularies:
#
#   - runner `sex` contains horse sex codes such as C, F, G, H, M and R;
#   - race `sex_rest` contains labels such as F, M, F & M, C & F and C & G.
#
#   In racing terminology, labels such as F, M, C and G may represent words
#   such as filly, mare, colt and gelding. Their relationship to a runner's
#   stored code can also depend on age, jurisdiction and source convention.
#
#   This cell therefore profiles combinations descriptively. It does not
#   classify a runner as eligible or ineligible.
#
# Raw versus derived values:
#   Raw `sex_rest`, runner `sex`, runner `age`, jurisdiction and race type are
#   preserved unchanged.
#
#   Display columns are derived only to make blank values visible.
#
# Assumptions deliberately not made:
#   - `sex_rest = F` is not treated as requiring runner `sex = F`;
#   - `sex_rest = M` is not treated as requiring runner `sex = M`;
#   - fillies and mares are not distinguished solely from the stored sex code;
#   - colts and horses are not distinguished solely from age;
#   - rare combinations are not automatically classified as source errors;
#   - blank `sex_rest` is not interpreted as an unrestricted race;
#   - jurisdiction vocabularies are not assumed to be equivalent.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - the required runner-level dataframe is unavailable;
#   - required columns are absent;
#   - the runner population is empty;
#   - grouped counts fail to reconstruct the runner population;
#   - race-level jurisdiction cannot be attached uniquely.
#
# No source or reference data is written or modified.

required_prior_names = [
    "runner_eligibility",
    "jurisdiction_frame",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding race-classification, jurisdiction and "
        "runner-eligibility stages first. "
        f"Missing variables: {missing_prior_names}"
    )


required_runner_columns = {
    "date",
    "course",
    "off",
    "race_name",
    "type",
    "sex_rest",
    "horse",
    "sex",
    "age",
}

missing_runner_columns = sorted(
    required_runner_columns
    - set(runner_eligibility.columns)
)

if missing_runner_columns:
    raise RuntimeError(
        "`runner_eligibility` is missing required columns: "
        f"{missing_runner_columns}"
    )


required_jurisdiction_columns = {
    "date",
    "course",
    "off",
    "jurisdiction",
}

missing_jurisdiction_columns = sorted(
    required_jurisdiction_columns
    - set(jurisdiction_frame.columns)
)

if missing_jurisdiction_columns:
    raise RuntimeError(
        "`jurisdiction_frame` is missing required columns: "
        f"{missing_jurisdiction_columns}"
    )


if runner_eligibility.empty:
    raise AssertionError(
        "The runner-level eligibility dataframe is empty."
    )


# Build one unique jurisdiction row per provisional race.
race_jurisdiction_lookup = (
    jurisdiction_frame[
        [
            "date",
            "course",
            "off",
            "jurisdiction",
        ]
    ]
    .drop_duplicates()
    .copy()
)


if race_jurisdiction_lookup.duplicated(
    subset=["date", "course", "off"]
).any():
    duplicate_race_keys = (
        race_jurisdiction_lookup.loc[
            race_jurisdiction_lookup.duplicated(
                subset=["date", "course", "off"],
                keep=False,
            )
        ]
        .sort_values(["date", "course", "off"])
    )

    raise AssertionError(
        "Jurisdiction is not unique for some provisional race keys.\n"
        f"{duplicate_race_keys.head(20)}"
    )


sex_restriction_runner_frame = (
    runner_eligibility[
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "horse",
            "sex",
            "age",
        ]
    ]
    .merge(
        race_jurisdiction_lookup,
        on=["date", "course", "off"],
        how="left",
        validate="many_to_one",
    )
)


if sex_restriction_runner_frame["jurisdiction"].isna().any():
    unmatched_races = (
        sex_restriction_runner_frame.loc[
            sex_restriction_runner_frame[
                "jurisdiction"
            ].isna(),
            [
                "date",
                "course",
                "off",
                "race_name",
            ],
        ]
        .drop_duplicates()
        .head(20)
    )

    raise AssertionError(
        "Some runner rows did not receive a jurisdiction.\n"
        f"{unmatched_races}"
    )


def display_source_state(value):
    """
    Return a visible label for blank or null source values while preserving
    every populated raw value unchanged.
    """

    if value is None or pd.isna(value):
        return "<NULL>"

    text = str(value)

    if text.strip() == "":
        return "<BLANK>"

    return text


sex_restriction_runner_frame[
    "sex_rest_display"
] = sex_restriction_runner_frame["sex_rest"].map(
    display_source_state
)

sex_restriction_runner_frame[
    "runner_sex_display"
] = sex_restriction_runner_frame["sex"].map(
    display_source_state
)


# Parse age only for descriptive grouping.
#
# This is not an independent reconstruction of horse age. It is merely the
# numeric representation of the source runner `age` field.
sex_restriction_runner_frame[
    "runner_age_numeric"
] = pd.to_numeric(
    sex_restriction_runner_frame["age"],
    errors="coerce",
)


# Overall cross-tabulation of race restriction labels and runner sex codes.
sex_rest_runner_sex_profile = (
    sex_restriction_runner_frame
    .groupby(
        [
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.index.nunique(),
        ),
        distinct_horses=("horse", "nunique"),
        minimum_runner_age=("runner_age_numeric", "min"),
        maximum_runner_age=("runner_age_numeric", "max"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)


# The race count above cannot be derived safely from dataframe indexes after
# grouping, so replace it with an explicit unique race-key count.
overall_race_counts = (
    sex_restriction_runner_frame[
        [
            "sex_rest_display",
            "runner_sex_display",
            "date",
            "course",
            "off",
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "provisional_races",
        }
    )
)


sex_rest_runner_sex_profile = (
    sex_rest_runner_sex_profile
    .drop(columns=["provisional_races"])
    .merge(
        overall_race_counts,
        on=[
            "sex_rest_display",
            "runner_sex_display",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "sex_rest_display",
            "runner_rows",
            "runner_sex_display",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_runner_sex_profile["runner_rows"].sum()
) != len(sex_restriction_runner_frame):
    raise AssertionError(
        "Overall sex-restriction profile does not reconstruct "
        "the runner population."
    )


# Profile the same combinations by jurisdiction and source race type.
sex_rest_by_jurisdiction_and_type = (
    sex_restriction_runner_frame
    .groupby(
        [
            "jurisdiction",
            "type",
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        distinct_horses=("horse", "nunique"),
        minimum_runner_age=("runner_age_numeric", "min"),
        maximum_runner_age=("runner_age_numeric", "max"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "jurisdiction",
            "type",
            "sex_rest_display",
            "runner_rows",
        ],
        ascending=[True, True, True, False],
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_by_jurisdiction_and_type[
        "runner_rows"
    ].sum()
) != len(sex_restriction_runner_frame):
    raise AssertionError(
        "Jurisdiction/type profile does not reconstruct "
        "the runner population."
    )


# Profile restriction labels by the source runner age and runner sex.
#
# This may reveal age-related transitions such as F versus M or C versus H,
# but it does not itself establish the regulatory definition.
sex_rest_by_runner_age = (
    sex_restriction_runner_frame
    .groupby(
        [
            "sex_rest_display",
            "runner_age_numeric",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        provisional_races=(
            "race_name",
            "nunique",
        ),
        distinct_horses=("horse", "nunique"),
    )
    .sort_values(
        [
            "sex_rest_display",
            "runner_age_numeric",
            "runner_rows",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_by_runner_age["runner_rows"].sum()
) != len(sex_restriction_runner_frame):
    raise AssertionError(
        "Age profile does not reconstruct the runner population."
    )


# Select rare combinations for manual inspection.
#
# We deliberately avoid calling these contradictions because the governing
# semantics have not yet been established.
rare_sex_rest_combinations = (
    sex_rest_runner_sex_profile.loc[
        sex_rest_runner_sex_profile[
            "sex_rest_display"
        ].ne("<BLANK>")
        & sex_rest_runner_sex_profile[
            "sex_rest_display"
        ].ne("<NULL>")
        & (
            sex_rest_runner_sex_profile[
                "runner_rows"
            ] <= 25
        )
    ]
    .copy()
)


rare_combination_samples = (
    sex_restriction_runner_frame
    .merge(
        rare_sex_rest_combinations[
            [
                "sex_rest_display",
                "runner_sex_display",
            ]
        ],
        on=[
            "sex_rest_display",
            "runner_sex_display",
        ],
        how="inner",
        validate="many_to_one",
    )
    [
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "horse",
            "sex",
            "age",
        ]
    ]
    .sort_values(
        [
            "sex_rest",
            "sex",
            "jurisdiction",
            "date",
            "course",
            "off",
            "horse",
        ]
    )
    .reset_index(drop=True)
)


print("Overall race sex-restriction label versus runner sex code")
display(sex_rest_runner_sex_profile)

print(
    "Sex-restriction and runner-sex combinations "
    "by jurisdiction and race type"
)
display(sex_rest_by_jurisdiction_and_type)

print("Sex-restriction combinations by recorded runner age")
display(sex_rest_by_runner_age)

print(
    "Runner samples from rare populated sex-restriction combinations"
)
display(rare_combination_samples)

Overall race sex-restriction label versus runner sex code


,sex_rest_display,runner_sex_display,runner_rows,distinct_horses,minimum_runner_age,maximum_runner_age,first_date,last_date,provisional_races
0,<BLANK>,G,1067054,92829,2,18,2015-01-01,2026-05-27,155335
1,<BLANK>,F,224783,51150,2,4,2015-01-01,2026-05-27,78506
2,<BLANK>,C,162181,43301,1,4,2015-01-01,2026-05-27,53679
3,<BLANK>,M,126844,22550,5,14,2015-01-01,2026-05-27,62457
4,<BLANK>,H,30695,6824,5,13,2015-01-01,2026-05-26,14307
5,<BLANK>,R,867,218,2,9,2015-01-03,2026-05-23,815
6,<BLANK>,BB,1,1,3,3,2017-10-15,2017-10-15,1
7,C & F,C,69,66,2,3,2025-10-26,2026-04-19,5
8,C & G,C,14130,9954,2,4,2015-01-12,2026-05-23,2329
9,C & G,G,11127,8113,2,12,2015-01-12,2026-05-26,2187


Sex-restriction and runner-sex combinations by jurisdiction and race type


,jurisdiction,type,sex_rest_display,runner_sex_display,runner_rows,distinct_horses,minimum_runner_age,maximum_runner_age,first_date,last_date
0,Argentina,Flat,<BLANK>,C,2045,1084,2,4,2015-01-16,2026-05-25
1,Argentina,Flat,<BLANK>,H,870,472,5,9,2015-01-16,2026-05-25
2,Argentina,Flat,<BLANK>,F,293,219,2,4,2015-01-24,2026-05-25
3,Argentina,Flat,<BLANK>,M,104,79,5,7,2015-02-22,2025-12-13
4,Argentina,Flat,<BLANK>,G,21,9,3,9,2015-02-07,2018-12-15
...,...,...,...,...,...,...,...,...,...,...
464,Uruguay,Flat,<BLANK>,G,1,1,3,3,2018-03-11,2018-03-11
465,Uruguay,Flat,C & G,C,86,74,3,3,2015-09-06,2022-09-05
466,Uruguay,Flat,F,F,166,119,3,3,2015-09-06,2022-10-02
467,Uruguay,Flat,F & M,F,103,96,3,4,2015-01-06,2026-01-06


Sex-restriction combinations by recorded runner age


,sex_rest_display,runner_age_numeric,runner_sex_display,runner_rows,provisional_races,distinct_horses
0,<BLANK>,1,C,5,5,1
1,<BLANK>,2,C,62261,8197,24297
2,<BLANK>,2,F,43709,7376,18787
3,<BLANK>,2,G,22988,6527,8601
4,<BLANK>,2,R,99,74,63
...,...,...,...,...,...,...
136,M,10,M,660,540,355
137,M,10,G,2,1,2
138,M,11,M,171,157,103
139,M,12,M,32,31,26


Runner samples from rare populated sex-restriction combinations


,date,course,jurisdiction,off,race_name,type,sex_rest,horse,sex,age
0,2021-02-07,Gavea (BRZ),Brazil,9:00,Grande Premio Estado do Rio de Janeiro (3yo C...,Flat,C & G,Inevitable (BRZ),F,3
1,2015-10-01,Auteuil (FR),France,11:15,Prix Pride of Kildare (Hurdle) (Conditions) (3...,Hurdle,C & G,Yosille (FR),F,3
2,2018-05-08,Saint-Cloud (FR),France,2:45,Prix Pas de Deux (Maiden) (Unraced 3yo Colts &...,Flat,C & G,Ironor (FR),F,3
3,2016-08-11,Salisbury,Great Britain,4:10,totepool Sovereign Stakes (Colts & Geldings),Flat,C & G,Belgian Bill (GB),H,8
4,2016-08-11,Salisbury,Great Britain,4:10,totepool Sovereign Stakes (Colts & Geldings),Flat,C & G,Master Carpenter (IRE),H,5
...,...,...,...,...,...,...,...,...,...,...
88,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Parmenide (FR),G,6
89,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Top Glory (FR),G,8
90,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Tytan (FR),G,8
91,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Zygfryd (FR),G,8


In [13]:
# Input grain:
#   Runner-level rows from `runner_eligibility`, with one row per retained
#   source runner.
#
# Output grain:
#   1. One row per raw `sex_rest` and raw runner `sex` combination.
#   2. One row per `sex_rest`, jurisdiction, race type and runner-sex value.
#   3. One row per `sex_rest`, runner age and runner-sex value.
#   4. Sample races for rare or apparently contradictory combinations.
#
# Purpose:
#   Investigate what the race-level `sex_rest` field appears to represent,
#   without assuming that its letters can be compared literally with the
#   runner-level `sex` field.
#
# Important semantic caution:
#   The two source fields use different vocabularies:
#
#   - runner `sex` contains horse sex codes such as C, F, G, H, M and R;
#   - race `sex_rest` contains labels such as F, M, F & M, C & F and C & G.
#
#   In racing terminology, labels such as F, M, C and G may represent words
#   such as filly, mare, colt and gelding. Their relationship to a runner's
#   stored code can also depend on age, jurisdiction and source convention.
#
#   This cell therefore profiles combinations descriptively. It does not
#   classify a runner as eligible or ineligible.
#
# Raw versus derived values:
#   Raw `sex_rest`, runner `sex`, runner `age`, jurisdiction and race type are
#   preserved unchanged.
#
#   Display columns are derived only to make blank values visible.
#
# Assumptions deliberately not made:
#   - `sex_rest = F` is not treated as requiring runner `sex = F`;
#   - `sex_rest = M` is not treated as requiring runner `sex = M`;
#   - fillies and mares are not distinguished solely from the stored sex code;
#   - colts and horses are not distinguished solely from age;
#   - rare combinations are not automatically classified as source errors;
#   - blank `sex_rest` is not interpreted as an unrestricted race;
#   - jurisdiction vocabularies are not assumed to be equivalent.
#
# Validation and failure behaviour:
#   The cell fails if:
#   - the required runner-level dataframe is unavailable;
#   - required columns are absent;
#   - the runner population is empty;
#   - grouped counts fail to reconstruct the runner population;
#   - race-level jurisdiction cannot be attached uniquely.
#
# No source or reference data is written or modified.

required_prior_names = [
    "runner_eligibility",
    "jurisdiction_frame",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding race-classification, jurisdiction and "
        "runner-eligibility stages first. "
        f"Missing variables: {missing_prior_names}"
    )


required_runner_columns = {
    "date",
    "course",
    "off",
    "race_name",
    "type",
    "sex_rest",
    "horse",
    "sex",
    "age",
}

missing_runner_columns = sorted(
    required_runner_columns
    - set(runner_eligibility.columns)
)

if missing_runner_columns:
    raise RuntimeError(
        "`runner_eligibility` is missing required columns: "
        f"{missing_runner_columns}"
    )


required_jurisdiction_columns = {
    "date",
    "course",
    "off",
    "jurisdiction",
}

missing_jurisdiction_columns = sorted(
    required_jurisdiction_columns
    - set(jurisdiction_frame.columns)
)

if missing_jurisdiction_columns:
    raise RuntimeError(
        "`jurisdiction_frame` is missing required columns: "
        f"{missing_jurisdiction_columns}"
    )


if runner_eligibility.empty:
    raise AssertionError(
        "The runner-level eligibility dataframe is empty."
    )


# Build one unique jurisdiction row per provisional race.
race_jurisdiction_lookup = (
    jurisdiction_frame[
        [
            "date",
            "course",
            "off",
            "jurisdiction",
        ]
    ]
    .drop_duplicates()
    .copy()
)


if race_jurisdiction_lookup.duplicated(
    subset=["date", "course", "off"]
).any():
    duplicate_race_keys = (
        race_jurisdiction_lookup.loc[
            race_jurisdiction_lookup.duplicated(
                subset=["date", "course", "off"],
                keep=False,
            )
        ]
        .sort_values(["date", "course", "off"])
    )

    raise AssertionError(
        "Jurisdiction is not unique for some provisional race keys.\n"
        f"{duplicate_race_keys.head(20)}"
    )


sex_restriction_runner_frame = (
    runner_eligibility[
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "horse",
            "sex",
            "age",
        ]
    ]
    .merge(
        race_jurisdiction_lookup,
        on=["date", "course", "off"],
        how="left",
        validate="many_to_one",
    )
)


if sex_restriction_runner_frame["jurisdiction"].isna().any():
    unmatched_races = (
        sex_restriction_runner_frame.loc[
            sex_restriction_runner_frame[
                "jurisdiction"
            ].isna(),
            [
                "date",
                "course",
                "off",
                "race_name",
            ],
        ]
        .drop_duplicates()
        .head(20)
    )

    raise AssertionError(
        "Some runner rows did not receive a jurisdiction.\n"
        f"{unmatched_races}"
    )


def display_source_state(value):
    """
    Return a visible label for blank or null source values while preserving
    every populated raw value unchanged.
    """

    if value is None or pd.isna(value):
        return "<NULL>"

    text = str(value)

    if text.strip() == "":
        return "<BLANK>"

    return text


sex_restriction_runner_frame[
    "sex_rest_display"
] = sex_restriction_runner_frame["sex_rest"].map(
    display_source_state
)

sex_restriction_runner_frame[
    "runner_sex_display"
] = sex_restriction_runner_frame["sex"].map(
    display_source_state
)


# Parse age only for descriptive grouping.
#
# This is not an independent reconstruction of horse age. It is merely the
# numeric representation of the source runner `age` field.
sex_restriction_runner_frame[
    "runner_age_numeric"
] = pd.to_numeric(
    sex_restriction_runner_frame["age"],
    errors="coerce",
)


# Overall cross-tabulation of race restriction labels and runner sex codes.
sex_rest_runner_sex_profile = (
    sex_restriction_runner_frame
    .groupby(
        [
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        provisional_races=(
            "race_name",
            lambda values: values.index.nunique(),
        ),
        distinct_horses=("horse", "nunique"),
        minimum_runner_age=("runner_age_numeric", "min"),
        maximum_runner_age=("runner_age_numeric", "max"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)


# The race count above cannot be derived safely from dataframe indexes after
# grouping, so replace it with an explicit unique race-key count.
overall_race_counts = (
    sex_restriction_runner_frame[
        [
            "sex_rest_display",
            "runner_sex_display",
            "date",
            "course",
            "off",
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "provisional_races",
        }
    )
)


sex_rest_runner_sex_profile = (
    sex_rest_runner_sex_profile
    .drop(columns=["provisional_races"])
    .merge(
        overall_race_counts,
        on=[
            "sex_rest_display",
            "runner_sex_display",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "sex_rest_display",
            "runner_rows",
            "runner_sex_display",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_runner_sex_profile["runner_rows"].sum()
) != len(sex_restriction_runner_frame):
    raise AssertionError(
        "Overall sex-restriction profile does not reconstruct "
        "the runner population."
    )


# Profile the same combinations by jurisdiction and source race type.
sex_rest_by_jurisdiction_and_type = (
    sex_restriction_runner_frame
    .groupby(
        [
            "jurisdiction",
            "type",
            "sex_rest_display",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        distinct_horses=("horse", "nunique"),
        minimum_runner_age=("runner_age_numeric", "min"),
        maximum_runner_age=("runner_age_numeric", "max"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "jurisdiction",
            "type",
            "sex_rest_display",
            "runner_rows",
        ],
        ascending=[True, True, True, False],
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_by_jurisdiction_and_type[
        "runner_rows"
    ].sum()
) != len(sex_restriction_runner_frame):
    raise AssertionError(
        "Jurisdiction/type profile does not reconstruct "
        "the runner population."
    )


# Profile restriction labels by the source runner age and runner sex.
#
# This may reveal age-related transitions such as F versus M or C versus H,
# but it does not itself establish the regulatory definition.
sex_rest_by_runner_age = (
    sex_restriction_runner_frame
    .groupby(
        [
            "sex_rest_display",
            "runner_age_numeric",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        provisional_races=(
            "race_name",
            "nunique",
        ),
        distinct_horses=("horse", "nunique"),
    )
    .sort_values(
        [
            "sex_rest_display",
            "runner_age_numeric",
            "runner_rows",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_by_runner_age["runner_rows"].sum()
) != len(sex_restriction_runner_frame):
    raise AssertionError(
        "Age profile does not reconstruct the runner population."
    )


# Select rare combinations for manual inspection.
#
# We deliberately avoid calling these contradictions because the governing
# semantics have not yet been established.
rare_sex_rest_combinations = (
    sex_rest_runner_sex_profile.loc[
        sex_rest_runner_sex_profile[
            "sex_rest_display"
        ].ne("<BLANK>")
        & sex_rest_runner_sex_profile[
            "sex_rest_display"
        ].ne("<NULL>")
        & (
            sex_rest_runner_sex_profile[
                "runner_rows"
            ] <= 25
        )
    ]
    .copy()
)


rare_combination_samples = (
    sex_restriction_runner_frame
    .merge(
        rare_sex_rest_combinations[
            [
                "sex_rest_display",
                "runner_sex_display",
            ]
        ],
        on=[
            "sex_rest_display",
            "runner_sex_display",
        ],
        how="inner",
        validate="many_to_one",
    )
    [
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "horse",
            "sex",
            "age",
        ]
    ]
    .sort_values(
        [
            "sex_rest",
            "sex",
            "jurisdiction",
            "date",
            "course",
            "off",
            "horse",
        ]
    )
    .reset_index(drop=True)
)


print("Overall race sex-restriction label versus runner sex code")
display(sex_rest_runner_sex_profile)

print(
    "Sex-restriction and runner-sex combinations "
    "by jurisdiction and race type"
)
display(sex_rest_by_jurisdiction_and_type)

print("Sex-restriction combinations by recorded runner age")
display(sex_rest_by_runner_age)

print(
    "Runner samples from rare populated sex-restriction combinations"
)
display(rare_combination_samples)

Overall race sex-restriction label versus runner sex code


,sex_rest_display,runner_sex_display,runner_rows,distinct_horses,minimum_runner_age,maximum_runner_age,first_date,last_date,provisional_races
0,<BLANK>,G,1067054,92829,2,18,2015-01-01,2026-05-27,155335
1,<BLANK>,F,224783,51150,2,4,2015-01-01,2026-05-27,78506
2,<BLANK>,C,162181,43301,1,4,2015-01-01,2026-05-27,53679
3,<BLANK>,M,126844,22550,5,14,2015-01-01,2026-05-27,62457
4,<BLANK>,H,30695,6824,5,13,2015-01-01,2026-05-26,14307
5,<BLANK>,R,867,218,2,9,2015-01-03,2026-05-23,815
6,<BLANK>,BB,1,1,3,3,2017-10-15,2017-10-15,1
7,C & F,C,69,66,2,3,2025-10-26,2026-04-19,5
8,C & G,C,14130,9954,2,4,2015-01-12,2026-05-23,2329
9,C & G,G,11127,8113,2,12,2015-01-12,2026-05-26,2187


Sex-restriction and runner-sex combinations by jurisdiction and race type


,jurisdiction,type,sex_rest_display,runner_sex_display,runner_rows,distinct_horses,minimum_runner_age,maximum_runner_age,first_date,last_date
0,Argentina,Flat,<BLANK>,C,2045,1084,2,4,2015-01-16,2026-05-25
1,Argentina,Flat,<BLANK>,H,870,472,5,9,2015-01-16,2026-05-25
2,Argentina,Flat,<BLANK>,F,293,219,2,4,2015-01-24,2026-05-25
3,Argentina,Flat,<BLANK>,M,104,79,5,7,2015-02-22,2025-12-13
4,Argentina,Flat,<BLANK>,G,21,9,3,9,2015-02-07,2018-12-15
...,...,...,...,...,...,...,...,...,...,...
464,Uruguay,Flat,<BLANK>,G,1,1,3,3,2018-03-11,2018-03-11
465,Uruguay,Flat,C & G,C,86,74,3,3,2015-09-06,2022-09-05
466,Uruguay,Flat,F,F,166,119,3,3,2015-09-06,2022-10-02
467,Uruguay,Flat,F & M,F,103,96,3,4,2015-01-06,2026-01-06


Sex-restriction combinations by recorded runner age


,sex_rest_display,runner_age_numeric,runner_sex_display,runner_rows,provisional_races,distinct_horses
0,<BLANK>,1,C,5,5,1
1,<BLANK>,2,C,62261,8197,24297
2,<BLANK>,2,F,43709,7376,18787
3,<BLANK>,2,G,22988,6527,8601
4,<BLANK>,2,R,99,74,63
...,...,...,...,...,...,...
136,M,10,M,660,540,355
137,M,10,G,2,1,2
138,M,11,M,171,157,103
139,M,12,M,32,31,26


Runner samples from rare populated sex-restriction combinations


,date,course,jurisdiction,off,race_name,type,sex_rest,horse,sex,age
0,2021-02-07,Gavea (BRZ),Brazil,9:00,Grande Premio Estado do Rio de Janeiro (3yo C...,Flat,C & G,Inevitable (BRZ),F,3
1,2015-10-01,Auteuil (FR),France,11:15,Prix Pride of Kildare (Hurdle) (Conditions) (3...,Hurdle,C & G,Yosille (FR),F,3
2,2018-05-08,Saint-Cloud (FR),France,2:45,Prix Pas de Deux (Maiden) (Unraced 3yo Colts &...,Flat,C & G,Ironor (FR),F,3
3,2016-08-11,Salisbury,Great Britain,4:10,totepool Sovereign Stakes (Colts & Geldings),Flat,C & G,Belgian Bill (GB),H,8
4,2016-08-11,Salisbury,Great Britain,4:10,totepool Sovereign Stakes (Colts & Geldings),Flat,C & G,Master Carpenter (IRE),H,5
...,...,...,...,...,...,...,...,...,...,...
88,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Parmenide (FR),G,6
89,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Top Glory (FR),G,8
90,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Tytan (FR),G,8
91,2026-02-13,Chantilly,France,12:43,Prix du Carrefour des Deux Mares (Handicap) (A...,Flat,M,Zygfryd (FR),G,8


In [14]:
# Input grain:
#   One row per provisional race from `jurisdiction_frame`.
#
# Output grain:
#   1. One row per race with a populated `sex_rest`.
#   2. One row per structured `sex_rest` and detected race-name sex phrase.
#   3. One row per race where the structured field and race-name wording
#      appear inconsistent.
#
# Purpose:
#   Test whether the structured `sex_rest` field reproduces the sex-restriction
#   wording embedded in `race_name`, and identify bounded source anomalies for
#   later external verification.
#
# Interpretation boundary:
#   This comparison concerns two race-level source representations.
#   It does not determine individual runner eligibility and does not compare
#   `sex_rest` literally with the runner-level `sex` code.
#
# No source or reference data is written or modified.

required_prior_names = [
    "jurisdiction_frame",
    "sex_restriction_runner_frame",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding jurisdiction and sex-restriction stages first. "
        f"Missing variables: {missing_prior_names}"
    )


required_race_columns = {
    "date",
    "course",
    "off",
    "race_name",
    "type",
    "sex_rest",
    "jurisdiction",
}

missing_race_columns = sorted(
    required_race_columns
    - set(jurisdiction_frame.columns)
)

if missing_race_columns:
    raise RuntimeError(
        "`jurisdiction_frame` is missing required columns: "
        f"{missing_race_columns}"
    )


sex_restricted_races = (
    jurisdiction_frame.loc[
        jurisdiction_frame[
            "sex_rest"
        ].astype("string").str.strip().ne(""),
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "jurisdiction",
            "sex_rest",
        ],
    ]
    .copy()
)


if sex_restricted_races.empty:
    raise AssertionError(
        "No races with populated `sex_rest` were available."
    )


def detect_race_name_sex_phrase(race_name):
    """
    Detect an explicit sex-restriction phrase from the source race name.

    The function deliberately recognises only narrow, visible phrases.
    It does not infer eligibility from generic race titles or horse names.
    """

    if race_name is None or pd.isna(race_name):
        return None

    text = str(race_name).lower()

    phrase_patterns = [
        (
            "C & G",
            r"\bcolts?\s*(?:&|and)\s*geldings?\b",
        ),
        (
            "C & F",
            r"\bcolts?\s*(?:&|and)\s*fillies\b",
        ),
        (
            "F & M",
            r"\bfillies\s*(?:&|and)\s*mares\b",
        ),
        (
            "F",
            r"\bfillies\b",
        ),
        (
            "M",
            r"\bmares\b",
        ),
    ]

    for canonical_value, pattern in phrase_patterns:
        if re.search(pattern, text):
            return canonical_value

    return None


sex_restricted_races[
    "race_name_sex_phrase"
] = sex_restricted_races["race_name"].map(
    detect_race_name_sex_phrase
)


sex_restricted_races[
    "race_name_phrase_display"
] = sex_restricted_races[
    "race_name_sex_phrase"
].fillna("<NO EXPLICIT PHRASE>")


sex_rest_race_name_profile = (
    sex_restricted_races
    .groupby(
        [
            "sex_rest",
            "race_name_phrase_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "sex_rest",
            "provisional_races",
            "race_name_phrase_display",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_race_name_profile[
        "provisional_races"
    ].sum()
) != len(sex_restricted_races):
    raise AssertionError(
        "Sex-restriction race-name profile does not reconstruct "
        "the populated race population."
    )


# Flag only explicit disagreements.
#
# A race with no recognised phrase remains unverified rather than being
# classified as contradictory.
sex_rest_race_name_disagreements = (
    sex_restricted_races.loc[
        sex_restricted_races[
            "race_name_sex_phrase"
        ].notna()
        & sex_restricted_races[
            "sex_rest"
        ].ne(
            sex_restricted_races[
                "race_name_sex_phrase"
            ]
        ),
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "race_name_sex_phrase",
        ],
    ]
    .sort_values(
        [
            "jurisdiction",
            "date",
            "course",
            "off",
        ]
    )
    .reset_index(drop=True)
)


# Attach runner-sex composition to each explicit disagreement so that the
# anomaly can be reviewed as a complete race rather than as isolated runners.
disagreement_runner_composition = (
    sex_restriction_runner_frame
    .merge(
        sex_rest_race_name_disagreements[
            [
                "date",
                "course",
                "off",
                "sex_rest",
                "race_name_sex_phrase",
            ]
        ],
        on=[
            "date",
            "course",
            "off",
            "sex_rest",
        ],
        how="inner",
        validate="many_to_one",
    )
    .groupby(
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "race_name_sex_phrase",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        minimum_runner_age=("runner_age_numeric", "min"),
        maximum_runner_age=("runner_age_numeric", "max"),
    )
    .sort_values(
        [
            "date",
            "course",
            "off",
            "runner_rows",
        ],
        ascending=[True, True, True, False],
    )
    .reset_index(drop=True)
)


print("Structured sex restriction versus explicit race-name wording")
display(sex_rest_race_name_profile)

print("Explicit structured-field versus race-name disagreements")
display(sex_rest_race_name_disagreements)

print("Runner composition of explicit disagreement races")
display(disagreement_runner_composition)

Structured sex restriction versus explicit race-name wording


,sex_rest,race_name_phrase_display,provisional_races,distinct_race_names,first_date,last_date
0,C & F,C & F,5,5,2025-10-26,2026-04-19
1,C & G,C & G,2163,1201,2015-01-12,2026-05-26
2,C & G,<NO EXPLICIT PHRASE>,480,278,2015-03-08,2026-05-23
3,F,F,13154,7617,2015-01-01,2026-05-27
4,F,C & F,221,107,2015-03-22,2026-03-29
5,F,F & M,16,9,2015-04-03,2026-04-03
6,F & M,F & M,3797,1890,2015-01-01,2026-05-26
7,M,M,5748,4178,2015-01-01,2026-05-27
8,M,F & M,1,1,2021-09-18,2021-09-18


Explicit structured-field versus race-name disagreements


,date,course,jurisdiction,off,race_name,type,sex_rest,race_name_sex_phrase
0,2018-10-25,La Plata (ARG),Argentina,11:10,Gran Premio Provincia de Buenos Aires (3yo Co...,Flat,F,C & F
1,2022-10-20,La Plata (ARG),Argentina,9:30,Gran Premio Provincia de Buenos Aires (3yo Co...,Flat,F,C & F
2,2015-10-03,Hipodromo Chile (CHI),Chile,10:37,Gran Criterium Mauricio Serrano Palma (3yo Co...,Flat,F,C & F
3,2018-09-08,Hipodromo Chile (CHI),Chile,9:56,Dos Mil Guineas (3yo Colts & Fillies) (Dirt),Flat,F,C & F
4,2018-11-02,Club Hipico de Santiago (CHI),Chile,10:20,Premio El Ensayo Mega (3yo Colts & Fillies) (...,Flat,F,C & F
...,...,...,...,...,...,...,...,...
233,2024-12-28,Nakayama (JPN),Japan,6:40,Hopeful Stakes (2yo Colts & Fillies) (Turf),Flat,F,C & F
234,2025-03-16,Nakayama (JPN),Japan,6:45,Spring Stakes (3yo Colts & Fillies) (Turf),Flat,F,C & F
235,2025-04-20,Nakayama (JPN),Japan,7:40,Satsuki Sho (Japanese 2000 Guineas) (3yo Colt...,Flat,F,C & F
236,2025-06-01,Tokyo (JPN),Japan,7:40,Tokyo Yushun (Japanese Derby) (3yo Colts & Fi...,Flat,F,C & F


Runner composition of explicit disagreement races


,date,course,jurisdiction,off,race_name,type,sex_rest,race_name_sex_phrase,runner_sex_display,runner_rows,minimum_runner_age,maximum_runner_age
0,2015-03-22,Nakayama (JPN),Japan,6:45,Fuji TV Sho Spring Stakes (Japanese 2 000 Guin...,Flat,F,C & F,C,12,3,3
1,2015-04-03,Lingfield (AW),Great Britain,1:40,32Red.com All-Weather Fillies And Mares Champi...,Flat,F,F & M,F,6,4,4
2,2015-04-03,Lingfield (AW),Great Britain,1:40,32Red.com All-Weather Fillies And Mares Champi...,Flat,F,F & M,M,4,5,6
3,2015-04-19,Nakayama (JPN),Japan,7:40,Satsuki Sho (Japanese 2000 Guineas) (3yo Colt...,Flat,F,C & F,C,14,3,3
4,2015-04-19,Nakayama (JPN),Japan,7:40,Satsuki Sho (Japanese 2000 Guineas) (3yo Colt...,Flat,F,C & F,G,1,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...
414,2026-02-27,Lingfield (AW),Great Britain,14:42,BetMGM AWC Fillies And Mares Trial Handicap,Flat,F,F & M,F,5,4,4
415,2026-03-29,Ascot,Great Britain,17:25,Colts And Fillies Club Handicap Hurdle,Hurdle,F,C & F,G,7,5,12
416,2026-03-29,Ascot,Great Britain,17:25,Colts And Fillies Club Handicap Hurdle,Hurdle,F,C & F,M,2,6,8
417,2026-04-03,Newcastle (AW),Great Britain,14:25,BetMGM Fillies And Mares Championships Handicap,Flat,F,F & M,F,6,4,4


In [15]:
# Input grain:
#   One row per provisional race with populated `sex_rest`, plus the
#   runner-level sex composition already prepared.
#
# Output grain:
#   1. One row per jurisdiction and explicit race-name phrase for races where
#      structured `sex_rest = F`.
#   2. One row per such race showing its complete runner-sex composition.
#   3. A bounded set of likely race-title false positives.
#
# Purpose:
#   Determine whether `F` consistently means a female-only restriction or is
#   used more broadly in some jurisdictions and source conventions.
#
# Important boundary:
#   This is descriptive source-semantic analysis. It does not infer legal
#   eligibility and does not correct any source value.
#
# No source or reference data is written or modified.

required_prior_names = [
    "sex_restricted_races",
    "sex_restriction_runner_frame",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding sex-restriction stages first. "
        f"Missing variables: {missing_prior_names}"
    )


# Restrict attention to the ambiguous structured F value.
structured_f_races = (
    sex_restricted_races.loc[
        sex_restricted_races["sex_rest"].eq("F"),
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "race_name_sex_phrase",
            "race_name_phrase_display",
        ],
    ]
    .copy()
)


if structured_f_races.empty:
    raise AssertionError(
        "No races with structured `sex_rest = F` were found."
    )


# Summarise the race-name wording attached to structured F by jurisdiction.
structured_f_phrase_by_jurisdiction = (
    structured_f_races
    .groupby(
        [
            "jurisdiction",
            "type",
            "race_name_phrase_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "provisional_races",
            "jurisdiction",
            "type",
            "race_name_phrase_display",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)


if int(
    structured_f_phrase_by_jurisdiction[
        "provisional_races"
    ].sum()
) != len(structured_f_races):
    raise AssertionError(
        "Structured-F jurisdiction summary does not reconstruct "
        "the structured-F race population."
    )


# Count each runner-sex code within every structured-F race.
structured_f_runner_composition_long = (
    sex_restriction_runner_frame.loc[
        sex_restriction_runner_frame["sex_rest"].eq("F")
    ]
    .groupby(
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "runner_sex_display",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_rows=("horse", "size"),
        minimum_runner_age=("runner_age_numeric", "min"),
        maximum_runner_age=("runner_age_numeric", "max"),
    )
)


# Create one race-level composition string for easier inspection.
structured_f_runner_composition = (
    structured_f_runner_composition_long
    .assign(
        composition_component=lambda frame: (
            frame["runner_sex_display"]
            + "="
            + frame["runner_rows"].astype(str)
        )
    )
    .sort_values(
        [
            "date",
            "course",
            "off",
            "runner_sex_display",
        ]
    )
    .groupby(
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_sex_composition=(
            "composition_component",
            lambda values: "; ".join(values),
        ),
        minimum_runner_age=("minimum_runner_age", "min"),
        maximum_runner_age=("maximum_runner_age", "max"),
        runner_rows=("runner_rows", "sum"),
    )
    .merge(
        structured_f_races[
            [
                "date",
                "course",
                "off",
                "race_name_sex_phrase",
                "race_name_phrase_display",
            ]
        ],
        on=["date", "course", "off"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "jurisdiction",
            "date",
            "course",
            "off",
        ]
    )
    .reset_index(drop=True)
)


if int(
    structured_f_runner_composition["runner_rows"].sum()
) != int(
    sex_restriction_runner_frame.loc[
        sex_restriction_runner_frame["sex_rest"].eq("F")
    ].shape[0]
):
    raise AssertionError(
        "Structured-F race compositions do not reconstruct "
        "the structured-F runner population."
    )


# Separate explicit mixed-sex wording from ordinary female wording.
structured_f_explicit_mixed_sex = (
    structured_f_runner_composition.loc[
        structured_f_runner_composition[
            "race_name_sex_phrase"
        ].eq("C & F")
    ]
    .copy()
)


structured_f_explicit_female_combined = (
    structured_f_runner_composition.loc[
        structured_f_runner_composition[
            "race_name_sex_phrase"
        ].eq("F & M")
    ]
    .copy()
)


# Identify likely phrase-detector false positives where the phrase may belong
# to a title, club name or sponsorship rather than a parenthesised condition.
#
# These remain candidates for inspection; they are not automatically removed.
likely_title_phrase_false_positives = (
    structured_f_runner_composition.loc[
        structured_f_runner_composition[
            "race_name_sex_phrase"
        ].notna()
        & ~structured_f_runner_composition[
            "race_name"
        ].str.contains(
            r"\([^)]*(?:colts?|fillies|mares|geldings?)[^)]*\)",
            case=False,
            regex=True,
            na=False,
        )
    ]
    .copy()
)


print(
    "Structured F: explicit race-name wording by jurisdiction and race type"
)
display(structured_f_phrase_by_jurisdiction)

print(
    "Structured F races whose names explicitly say Colts & Fillies"
)
display(structured_f_explicit_mixed_sex)

print(
    "Structured F races whose names explicitly say Fillies & Mares"
)
display(structured_f_explicit_female_combined)

print(
    "Potential race-title or sponsorship phrase false positives"
)
display(likely_title_phrase_false_positives)

Structured F: explicit race-name wording by jurisdiction and race type


,jurisdiction,type,race_name_phrase_display,provisional_races,distinct_race_names,first_date,last_date
0,Great Britain,Flat,F,7219,4883,2015-01-02,2026-05-27
1,France,Flat,F,1587,823,2015-01-10,2026-05-22
2,Ireland,Flat,F,1398,473,2015-01-30,2026-05-24
3,United States,Flat,F,1040,426,2015-01-03,2026-05-15
4,Australia,Flat,F,504,307,2015-01-26,2026-05-23
5,France,Hurdle,F,436,161,2015-02-24,2026-05-26
6,Japan,Flat,F,141,44,2015-01-12,2026-05-24
7,Great Britain,Hurdle,F,100,86,2015-01-08,2026-04-25
8,Germany,Flat,F,96,77,2015-04-19,2026-05-24
9,Brazil,Flat,F,79,23,2015-01-20,2026-04-12


Structured F races whose names explicitly say Colts & Fillies


,date,course,jurisdiction,off,race_name,type,runner_sex_composition,minimum_runner_age,maximum_runner_age,runner_rows,race_name_sex_phrase,race_name_phrase_display
33,2018-10-25,La Plata (ARG),Argentina,11:10,Gran Premio Provincia de Buenos Aires (3yo Co...,Flat,C=10,3,3,10,C & F,C & F
56,2022-10-20,La Plata (ARG),Argentina,9:30,Gran Premio Provincia de Buenos Aires (3yo Co...,Flat,C=8,3,3,8,C & F,C & F
715,2015-10-03,Hipodromo Chile (CHI),Chile,10:37,Gran Criterium Mauricio Serrano Palma (3yo Co...,Flat,C=11,3,3,11,C & F,C & F
734,2018-09-08,Hipodromo Chile (CHI),Chile,9:56,Dos Mil Guineas (3yo Colts & Fillies) (Dirt),Flat,C=14,3,3,14,C & F,C & F
735,2018-11-02,Club Hipico de Santiago (CHI),Chile,10:20,Premio El Ensayo Mega (3yo Colts & Fillies) (...,Flat,C=12,3,3,12,C & F,C & F
...,...,...,...,...,...,...,...,...,...,...,...,...
12147,2024-12-28,Nakayama (JPN),Japan,6:40,Hopeful Stakes (2yo Colts & Fillies) (Turf),Flat,C=17; F=1,2,2,18,C & F,C & F
12152,2025-03-16,Nakayama (JPN),Japan,6:45,Spring Stakes (3yo Colts & Fillies) (Turf),Flat,C=12,3,3,12,C & F,C & F
12155,2025-04-20,Nakayama (JPN),Japan,7:40,Satsuki Sho (Japanese 2000 Guineas) (3yo Colt...,Flat,C=18,3,3,18,C & F,C & F
12158,2025-06-01,Tokyo (JPN),Japan,7:40,Tokyo Yushun (Japanese Derby) (3yo Colts & Fi...,Flat,C=18,3,3,18,C & F,C & F


Structured F races whose names explicitly say Fillies & Mares


,date,course,jurisdiction,off,race_name,type,runner_sex_composition,minimum_runner_age,maximum_runner_age,runner_rows,race_name_sex_phrase,race_name_phrase_display
3078,2015-04-03,Lingfield (AW),Great Britain,1:40,32Red.com All-Weather Fillies And Mares Champi...,Flat,F=6; M=4,4,6,10,F & M,F & M
3755,2016-03-25,Lingfield (AW),Great Britain,1:40,32Red All-Weather Fillies And Mares Championsh...,Flat,F=4; M=6,4,7,10,F & M,F & M
3959,2016-06-11,Musselburgh,Great Britain,4:20,William Hill Fillies And Mares (A Handicap),Flat,F=8; M=2,3,9,10,F & M,F & M
4511,2017-04-14,Lingfield (AW),Great Britain,2:40,32Red All-Weather Fillies And Mares Championsh...,Flat,F=5; M=6,4,6,11,F & M,F & M
5232,2018-03-30,Lingfield (AW),Great Britain,2:30,32Red All-Weather Fillies And Mares Championsh...,Flat,F=4; M=9,4,7,13,F & M,F & M
6016,2019-04-19,Lingfield (AW),Great Britain,2:30,Ladbrokes All-Weather Fillies And Mares Champi...,Flat,F=5; M=7,4,5,12,F & M,F & M
6675,2021-04-02,Lingfield (AW),Great Britain,2:35,Ladbrokes All-Weather Fillies And Mares Champi...,Flat,F=4; M=2,4,5,6,F & M,F & M
7389,2022-04-15,Newcastle (AW),Great Britain,3:45,Coral All-Weather Fillies And Mares Championsh...,Flat,F=4; M=3,4,5,7,F & M,F & M
8100,2023-04-07,Newcastle (AW),Great Britain,4:10,talkSPORT All-Weather Fillies And Mares Champi...,Flat,F=4; M=3,4,6,7,F & M,F & M
8754,2024-03-01,Lingfield (AW),Great Britain,2:02,BetMGM AWC Fillies And Mares Trial Handicap,Flat,F=5; M=4,4,7,9,F & M,F & M


Potential race-title or sponsorship phrase false positives


,date,course,jurisdiction,off,race_name,type,runner_sex_composition,minimum_runner_age,maximum_runner_age,runner_rows,race_name_sex_phrase,race_name_phrase_display
108,2015-04-25,Moe (AUS),Australia,4:58,Strathayr 2yo Fillies Maiden Plate (Turf),Flat,F=10,2,2,10,F,F
160,2016-05-07,Gold Coast (AUS),Australia,4:40,All-New Jaguar XE Gold Coast Bracelet QTIS 3-Y...,Flat,F=14,3,3,14,F,F
162,2016-05-21,Doomben (AUS),Australia,6:21,Magic Millions The Roses QTIS Three-Years-Old ...,Flat,F=16,3,3,16,F,F
263,2018-05-07,Wangaratta (AUS),Australia,3:30,Winsec Savings & Loans 2yo Fillies Maiden Plat...,Flat,F=9,2,2,9,F,F
265,2018-05-19,Morphettville (AUS),Australia,4:46,UBET SA Fillies Classic (3yo) (Turf),Flat,F=14,3,3,14,F,F
...,...,...,...,...,...,...,...,...,...,...,...,...
12244,2018-12-15,Kenilworth (SAF),South Africa,1:55,World Sports Betting Fillies Guineas (3yo) (T...,Flat,F=13,3,3,13,F,F
12246,2019-03-02,Turffontein Standside (SAF),South Africa,1:30,Wilgerbosdrift S A Fillies Classic (3yo) (Turf),Flat,F=12,3,3,12,F,F
12267,2024-03-02,Turffontein Standside (SAF),South Africa,12:55,Wilgerbosdrift SA Fillies Classic (3yo) (Turf),Flat,F=9,3,3,9,F,F
12269,2024-06-01,Scottsville (SAF),South Africa,2:10,South African Fillies Sprint brought to you by...,Flat,F=15; M=1,3,5,16,F,F


In [16]:
# Input grain:
#   One row per provisional race with populated `sex_rest`.
#
# Output grain:
#   1. Annual counts of every raw `sex_rest` value.
#   2. Annual counts restricted to race names explicitly saying
#      `Colts & Fillies`.
#   3. The complete five-race population carrying raw `C & F`.
#
# Purpose:
#   Determine whether `C & F` represents a recent source-encoding change and
#   whether historically equivalent races were previously stored as `F`.
#
# Interpretation boundary:
#   This cell identifies representation patterns only. It does not establish
#   an official regulatory definition or silently recode historical values.
#
# No source or reference data is written or modified.

required_prior_names = [
    "sex_restricted_races",
    "sex_restriction_runner_frame",
]

missing_prior_names = [
    name
    for name in required_prior_names
    if name not in globals()
]

if missing_prior_names:
    raise RuntimeError(
        "Run the preceding sex-restriction stages first. "
        f"Missing variables: {missing_prior_names}"
    )


sex_rest_temporal_frame = sex_restricted_races.copy()

sex_rest_temporal_frame["year"] = pd.to_datetime(
    sex_rest_temporal_frame["date"],
    errors="raise",
).dt.year


# Profile every raw structured value by year.
sex_rest_by_year = (
    sex_rest_temporal_frame
    .groupby(
        [
            "year",
            "sex_rest",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "year",
            "sex_rest",
        ]
    )
    .reset_index(drop=True)
)


if int(
    sex_rest_by_year["provisional_races"].sum()
) != len(sex_rest_temporal_frame):
    raise AssertionError(
        "Annual sex-rest profile does not reconstruct the populated "
        "sex-rest race population."
    )


# Restrict to names that explicitly contain a Colts & Fillies condition.
explicit_colts_fillies_races = (
    sex_rest_temporal_frame.loc[
        sex_rest_temporal_frame[
            "race_name_sex_phrase"
        ].eq("C & F")
    ]
    .copy()
)


explicit_colts_fillies_by_year_and_raw_value = (
    explicit_colts_fillies_races
    .groupby(
        [
            "year",
            "sex_rest",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        provisional_races=("race_name", "size"),
        jurisdictions=("jurisdiction", "nunique"),
        distinct_race_names=("race_name", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "year",
            "sex_rest",
        ]
    )
    .reset_index(drop=True)
)


if int(
    explicit_colts_fillies_by_year_and_raw_value[
        "provisional_races"
    ].sum()
) != len(explicit_colts_fillies_races):
    raise AssertionError(
        "Annual Colts & Fillies profile does not reconstruct its "
        "race population."
    )


# Inspect every occurrence of the newly observed literal C & F value.
literal_c_and_f_races = (
    sex_rest_temporal_frame.loc[
        sex_rest_temporal_frame["sex_rest"].eq("C & F"),
        [
            "date",
            "course",
            "jurisdiction",
            "off",
            "race_name",
            "type",
            "sex_rest",
            "race_name_sex_phrase",
        ],
    ]
    .merge(
        sex_restriction_runner_frame.loc[
            sex_restriction_runner_frame[
                "sex_rest"
            ].eq("C & F")
        ]
        .groupby(
            [
                "date",
                "course",
                "off",
                "runner_sex_display",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            runner_rows=("horse", "size"),
            minimum_runner_age=("runner_age_numeric", "min"),
            maximum_runner_age=("runner_age_numeric", "max"),
        )
        .assign(
            composition_component=lambda frame: (
                frame["runner_sex_display"]
                + "="
                + frame["runner_rows"].astype(str)
            )
        )
        .sort_values(
            [
                "date",
                "course",
                "off",
                "runner_sex_display",
            ]
        )
        .groupby(
            [
                "date",
                "course",
                "off",
            ],
            as_index=False,
        )
        .agg(
            runner_sex_composition=(
                "composition_component",
                lambda values: "; ".join(values),
            ),
            runner_rows=("runner_rows", "sum"),
            minimum_runner_age=("minimum_runner_age", "min"),
            maximum_runner_age=("maximum_runner_age", "max"),
        ),
        on=[
            "date",
            "course",
            "off",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "date",
            "course",
            "off",
        ]
    )
    .reset_index(drop=True)
)


if len(literal_c_and_f_races) != 5:
    raise AssertionError(
        "Expected five races with literal `C & F`, found "
        f"{len(literal_c_and_f_races):,}."
    )


print("All populated sex-rest values by year")
display(sex_rest_by_year)

print(
    "Explicit Colts & Fillies race names by year and raw sex-rest value"
)
display(explicit_colts_fillies_by_year_and_raw_value)

print("Complete population carrying literal C & F")
display(literal_c_and_f_races)

All populated sex-rest values by year


,year,sex_rest,provisional_races,distinct_race_names,first_date,last_date
0,2015,C & G,306,298,2015-01-12,2015-12-22
1,2015,F,1366,1176,2015-01-01,2015-12-29
2,2015,F & M,411,389,2015-01-01,2015-12-27
3,2015,M,425,394,2015-01-01,2015-12-31
4,2016,C & G,267,259,2016-01-03,2016-12-28
5,2016,F,1311,1160,2016-01-02,2016-12-31
6,2016,F & M,340,339,2016-01-02,2016-12-31
7,2016,M,463,427,2016-01-01,2016-12-31
8,2017,C & G,258,253,2017-01-05,2017-12-26
9,2017,F,1338,1159,2017-01-02,2017-12-30


Explicit Colts & Fillies race names by year and raw sex-rest value


,year,sex_rest,provisional_races,jurisdictions,distinct_race_names,first_date,last_date
0,2015,F,34,7,34,2015-03-22,2015-12-27
1,2016,F,33,6,33,2016-03-20,2016-12-25
2,2017,F,30,6,30,2017-03-19,2017-12-28
3,2018,F,27,8,27,2018-03-25,2018-12-28
4,2019,F,21,6,21,2019-02-03,2019-12-15
5,2021,F,22,7,22,2021-02-07,2021-12-28
6,2022,F,14,4,14,2022-04-09,2022-12-18
7,2023,F,12,5,12,2023-04-02,2023-12-28
8,2024,F,13,3,13,2024-04-06,2024-12-28
9,2025,C & F,4,2,4,2025-10-26,2025-12-27


Complete population carrying literal C & F


,date,course,jurisdiction,off,race_name,type,sex_rest,race_name_sex_phrase,runner_sex_composition,runner_rows,minimum_runner_age,maximum_runner_age
0,2025-10-26,Kyoto,Japan,06:40,Kikuka Sho (Japanese St Leger) (3yo Colts & Fi...,Flat,C & F,C & F,C=18,18,3,3
1,2025-10-26,Saint-Cloud,France,12:30,Criterium International (2yo Colts & Fillies) ...,Flat,C & F,C & F,C=7,7,2,2
2,2025-10-26,Saint-Cloud,France,13:34,Criterium de Saint-Cloud (2yo Colts & Fillies)...,Flat,C & F,C & F,C=10,10,2,2
3,2025-12-27,Nakayama,Japan,06:45,Hopeful Stakes (2yo Colts & Fillies) (Turf),Flat,C & F,C & F,C=16,16,2,2
4,2026-04-19,Nakayama,Japan,07:40,Satsuki Sho (Japanese 2000 Guineas) (Colts & F...,Flat,C & F,C & F,C=18,18,3,3


## Sex-restriction semantics

The source fields `sex_rest` and runner `sex` use related racing terminology but operate at different grains.

Runner `sex` records a runner-level category, including:

- `C` — colt;
- `F` — filly;
- `G` — gelding;
- `H` — horse or older entire male;
- `M` — mare;
- `R` — rig.

Rare values such as `B` and `BB` remain unresolved source codes and must not be interpreted without separate evidence.

The race-level `sex_rest` field contains:

- `F`;
- `M`;
- `F & M`;
- `C & G`;
- `C & F`;
- blank.

### Runner composition

The dominant runner compositions support several straightforward source meanings:

- `C & G` predominantly contains colts and geldings;
- `F & M` predominantly contains fillies and mares;
- `C & F` contains runners coded as colts in the five observed races;
- `F` predominantly contains fillies, with mares and some male-coded runners;
- `M` predominantly contains mares, with some younger runners coded as fillies.

Runner `sex` must not be compared literally with `sex_rest`.

A filly can later be recorded as a mare, and a colt can later be recorded as a horse. The race-condition wording and runner-level category therefore need not use the same age-specific term.

Rare incompatible-looking combinations remain review candidates rather than automatic eligibility failures.

### Structured field versus race-name wording

Race-name comparison showed that the explicit combined values generally reproduce the visible race wording:

- all five `C & F` races explicitly say Colts & Fillies;
- 2,163 `C & G` races explicitly say Colts & Geldings;
- 3,797 `F & M` races explicitly say Fillies & Mares.

However, the raw value `F` is overloaded.

Among races stored as `F`:

- 221 race names explicitly say Colts & Fillies;
- 16 race names explicitly say Fillies & Mares;
- the majority explicitly use Fillies wording.

The Colts & Fillies cases are systematic across several jurisdictions and years and commonly contain predominantly or exclusively colts. They cannot be interpreted as isolated runner-entry errors.

Therefore:

> Raw `sex_rest = F` does not universally mean that a race was restricted exclusively to fillies.

It is a source-presented category whose precise scope sometimes requires supporting race-condition text.

### Temporal representation

The literal value `C & F` first appears on 26 October 2025.

The complete observed population is:

- four races in 2025;
- one race in 2026;
- races in France and Japan;
- all carrying explicit Colts & Fillies wording.

Before this date, equivalent Colts & Fillies races were generally stored as `F`.

However, `F` continued to be used for at least one explicit Colts & Fillies race after `C & F` first appeared.

This is therefore a partial or selective source-representation change rather than a complete schema transition.

No date-based automatic recoding rule is justified.

### Analytical conclusion

The safe database treatment is:

1. preserve raw runner `sex`;
2. preserve raw race-level `sex_rest`;
3. document the standard runner-code meanings separately from race-condition meanings;
4. retain `C & G`, `C & F` and `F & M` as explicit combined source values;
5. treat `F` as an overloaded source category;
6. avoid converting `F` universally to `fillies_only`;
7. avoid determining runner eligibility through literal code equality;
8. preserve genuine race-condition wording extracted from `race_name` separately;
9. distinguish condition text from title, sponsorship and club-name text;
10. do not silently recode historical `F` values to `C & F` or `F & M`;
11. expose disagreements and ambiguous cases as review statuses;
12. require external or jurisdiction-specific evidence before applying any correction.

### Limitations

The analysis establishes observed source behaviour but does not define the official regulatory eligibility rules for every jurisdiction.

Race names are useful supporting evidence, but wording detection can produce false positives where terms such as Colts, Fillies or Mares occur in a title, sponsorship phrase or club name rather than in the race conditions.

The five literal `C & F` values demonstrate a recent source representation, but their small and selective population does not establish a complete feed-wide format change.

## Sex-based analytical use

The source supports two different kinds of analysis:

- runner-level analysis using the raw runner `sex` field;
- descriptive analysis using the raw race-level `sex_rest` categories.

It does not support a universal reconstruction of official sex eligibility from `sex_rest` alone.

Safe uses include:

- profiling results by recorded runner sex;
- comparing raw `sex_rest` categories within a defined jurisdiction and period;
- identifying changes in source coding;
- selecting bounded subsets where the official condition has been independently established.

Unsafe uses include:

- treating every `F` race as fillies-only;
- interpreting blank `sex_rest` as unrestricted;
- deciding runner eligibility through literal equality between `sex` and `sex_rest`;
- globally comparing fillies-only, mixed-sex and unrestricted races from this field alone;
- silently recoding historical `F` values into more specific categories.

Authoritative sex-condition reconstruction is therefore deferred to a future jurisdiction-specific study using official race-condition text and governed provenance.

For the current database build:

> Preserve raw `sex` and `sex_rest`, expose explicit source categories, retain ambiguity, and do not derive universal permitted-sex flags.

## Overall conclusion and governed field decisions

Notebook 16 investigated the source fields used to describe race classification and eligibility:

- `race_name`;
- `type`;
- `class`;
- `pattern`;
- `rating_band`;
- `age_band`;
- `sex_rest`.

The bounded question was:

> What do these fields represent, how do they relate to one another, and which values can be interpreted or derived safely?

The source supports reliable structural parsing for several fields, but it does not support one universal classification or eligibility model across every jurisdiction.

## Final field decisions

| Field | Safe source meaning | Safe derived treatment | Unsafe treatment | Status |
|---|---|---|---|---|
| `race_name` | Source-presented race title and embedded descriptive wording | Preserve unchanged; use narrowly for supporting phrase extraction and anomaly review | Treat every detected word or phrase as an official condition | Confirmed raw source field |
| `type` | Source race-type category | Preserve canonical observed values: `Flat`, `Hurdle`, `Chase`, `NH Flat` | Infer deeper code or regulatory equivalence without jurisdiction context | Confirmed source category |
| `class` | Source-presented broad race class | Preserve raw value; parse the integer from canonical `Class N` forms | Derive from `rating_band`; assume international Class 1 equals British Class 1 | Confirmed structure; contextual meaning |
| `pattern` | Source-presented Listed, Group or Grade status | Preserve raw value and separate Listed, Group and Grade families | Collapse all Group and Grade labels into one universal hierarchy | Confirmed structure; jurisdiction-dependent meaning |
| `rating_band` | Source-presented rating restriction where populated | Parse exact `N-N` forms into stated lower and upper bounds | Infer class, quality, or a universal handicap scale; parse `--` or `(75-100)` as canonical ranges | Confirmed parser with unresolved source forms |
| `age_band` | Structured representation of the source-presented age condition | Parse exact, open-ended and closed-range syntax into stated bounds and explicit `+` status | Enforce every exact-looking value as a universal closed eligibility rule against runner `age` | Confirmed syntax; contextual semantics |
| `sex_rest` | Source shorthand for a race-level sex-related condition | Preserve raw categories and explicit combined values | Treat `F` universally as fillies-only; reconstruct official eligibility globally from this field alone | Confirmed source shorthand; overloaded value |

## Core findings

### Classification fields are complementary

`class`, `pattern` and `rating_band` are related but not interchangeable.

In Great Britain:

- class values span multiple rating bands;
- rating bands overlap between adjacent classes;
- many higher-class races have no populated rating band;
- NH Flat races carry class values but no rating bands.

There are 6,387 races where both `class` and `pattern` are populated, proving that the two fields describe different properties and must remain independent.

### Rating-band parsing is narrow but reliable

There are 381 canonical closed integer ranges matching `N-N`.

These can be parsed safely into stated numeric bounds.

The source also contains:

- 12 occurrences of `--`;
- one occurrence of `(75-100)`.

These values must remain unresolved source states rather than being normalised into the canonical parser.

### Age-band syntax is reliable; eligibility interpretation is not universal

All populated `age_band` values fit one of three observed syntactic families:

- exact age;
- open-ended minimum age;
- closed age range.

The syntax is therefore safe to parse.

Literal comparison against source runner ages produced 958 apparent disagreements across 183 races. Manual verification showed several distinct causes:

- a confirmed dropped plus sign;
- a confirmed implausible runner age;
- exact-looking age wording used alongside older runners;
- an unresolved jurisdiction or feed discrepancy.

The governed verification records are:

- `NB16-AGE-0001`;
- `NB16-AGE-0002`;
- `NB16-AGE-0003`;
- `NB16-AGE-0004`.

These cases establish that disagreement must be treated as review evidence, not as one universal error type.

### Sex restriction is source shorthand rather than a complete official condition

The explicit combined values behave largely as expected:

- `C & G`;
- `C & F`;
- `F & M`.

The raw value `F` is overloaded and appears on races whose names describe:

- fillies;
- fillies and mares;
- colts and fillies.

The literal `C & F` value first appears in October 2025, but older and later equivalent races can still carry `F`. This indicates a partial or selective source-representation change rather than a clean schema boundary.

Precise official sex eligibility therefore cannot be reconstructed globally from `sex_rest` alone.

## Database consequence

The future processed database should preserve every raw source field and add separate derived fields with explicit status and provenance.

Recommended fields include:

### Class

- `source_class_raw`;
- `class_number`;
- `class_parse_status`.

### Pattern

- `source_pattern_raw`;
- `pattern_family`;
- `pattern_level_raw`;
- `pattern_parse_status`.

### Rating band

- `source_rating_band_raw`;
- `rating_lower_bound`;
- `rating_upper_bound`;
- `rating_band_parse_status`.

### Age band

- `source_age_band_raw`;
- `stated_minimum_age`;
- `stated_maximum_age`;
- `age_band_open_ended`;
- `age_band_syntax`;
- `age_band_interpretation_status`.

### Sex restriction

- `source_sex_rest_raw`;
- `sex_rest_category`;
- `sex_rest_interpretation_status`.

### Provenance and reconciliation

Where an external correction is applied, the processed database must also retain:

- immutable raw source value;
- reconciled value;
- reconciliation method;
- `verification_id`;
- confidence;
- database action;
- source-versus-corrected status.

No derived or externally corrected value may be presented as source-original.

## Analytical use

These fields support:

- descriptive source-category analysis;
- jurisdiction-specific classification studies;
- bounded studies where the relevant semantics have been established;
- anomaly detection and source-quality research.

They do not support:

- one global race-quality hierarchy;
- universal official eligibility reconstruction;
- automatic runner eligibility decisions;
- silent correction of ambiguous or contradictory values;
- betting conclusions based only on raw classification labels without controlling for jurisdiction, race type, period and field coverage.

## Confidence

Confidence is high in:

- observed vocabularies;
- race-level consistency;
- coverage counts;
- canonical parsing rules;
- coexistence relationships;
- the identified source anomalies with governed external evidence.

Confidence is moderate in:

- broader semantic interpretations of overloaded values;
- source-transition explanations;
- cross-jurisdiction equivalence.

Confidence is low or unresolved where:

- official eligibility cannot be reconstructed from shorthand;
- unusual international values lack governing-authority context;
- source and runner fields disagree without sufficient external evidence.

## Final decision

The source fields are suitable for a governed database when raw values, derived parsing, contextual interpretation, uncertainty and external corrections are stored separately.

They are not suitable for direct global classification or eligibility analysis without those safeguards.